In [1]:
# Imports et fonctions utilitaires
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd



In [2]:
# 2 Saisie interactive d'un contour d'aile
# Deux mÃƒÆ’Ã‚Â©thodes disponibles :
# 1) Coller un contour (une ligne par point "x,y") dans la Textarea puis cliquer "Charger depuis texte".
# 2) Saisie pointÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬ËœparÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Ëœpoint via l'interface widgets (x, y, Ajouter point, Terminer, Reset).

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Parsers / utilitaires ---
def parse_point(s):
    s = s.strip()
    if not s:
        return None
    if ',' in s:
        parts = s.split(',')
    else:
        parts = s.split()
    if len(parts) < 2:
        return None
    try:
        x = float(parts[0])
        y = float(parts[1])
        return (x, y)
    except Exception:
        return None


def close_polygon(pts):
    if len(pts) == 0:
        return np.array(pts)
    pts_array = np.array(pts)
    # VÃƒÆ’Ã‚Â©rifier si le premier et dernier point sont identiques (distance < 1e-9)
    dx = abs(pts_array[0][0] - pts_array[-1][0])
    dy = abs(pts_array[0][1] - pts_array[-1][1])
    if dx > 1e-9 or dy > 1e-9:
        # Ajouter le premier point ÃƒÆ’Ã‚Â  la fin pour fermer le polygone
        pts_array = np.vstack([pts_array, pts_array[0:1]])
    return pts_array


def intersect_segment_with_y(p1, p2, y):
    (x1, y1) = p1
    (x2, y2) = p2
    if (y1 - y) * (y2 - y) > 0:
        return None
    if y2 == y1:
        return None
    t = (y - y1) / (y2 - y1)
    if 0.0 <= t <= 1.0:
        x = x1 + t * (x2 - x1)
        return x
    return None


def intersections_with_line_y(polygon, y):
    xs = []
    for i in range(len(polygon) - 1):
        p1 = polygon[i]
        p2 = polygon[i+1]
        xi = intersect_segment_with_y(p1, p2, y)
        if xi is not None:
            xs.append(xi)
    return xs

# --- Fonction centrale pour dÃƒÆ’Ã‚Â©finir et tracer le contour chargÃƒÆ’Ã‚Â© ---

def _set_global_contour_and_plot(pts):
    """Stocke globalement wing_contour et wing_origin et affiche le tracÃƒÆ’Ã‚Â©."""
    global wing_contour, wing_origin
    wing_contour = np.array(pts, dtype=float)
    
    # Origine: milieu de la premiÃƒÆ’Ã‚Â¨re corde (y = y_root = y du premier point)
    y_root = wing_contour[0][1]
    xs = intersections_with_line_y(wing_contour, y_root)
    if len(xs) >= 2:
        x0 = 0.5 * (min(xs) + max(xs))
        y0 = y_root
    else:
        # Calculer le centroid manuellement
        sum_x = sum(wing_contour[i][0] for i in range(len(wing_contour)-1))
        sum_y = sum(wing_contour[i][1] for i in range(len(wing_contour)-1))
        n = len(wing_contour) - 1
        x0, y0 = sum_x / n, sum_y / n
    
    wing_origin = (x0, y0)
    
    # Affichage rapide du contour et de l'origine
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(wing_contour[:,0], wing_contour[:,1], '-k')
    ax.scatter(wing_contour[:-1,0], wing_contour[:-1,1], s=20, color='C0')
    ax.plot(x0, y0, '+r', markersize=12, markeredgewidth=2, label='origine (milieu corde racine)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_title('Contour chargÃƒÆ’Ã‚Â© et origine (milieu corde racine)')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    display(fig)
    plt.close(fig)
    print('Nombre de points (avec fermeture) :', len(wing_contour))
    print('Origine :', wing_origin)

# --- Interface 1 : Textarea pour coller un contour en texte + bouton de chargement ---
contour_textarea = widgets.Textarea(
    value = "0.0, 0.0\n0.2, 0.0\n0.463, 0.313\n0.463, 0.713\n0.09, 2.4\n-0.2, 2.4\n-0.375, 0.418\n-0.2, 0.0\n0.0, 0.0",
    placeholder = 'Une ligne par point: x,y',
    description = 'Contour (texte):',
    layout = widgets.Layout(width='100%', height='120px')
)
load_text_btn = widgets.Button(description='Charger depuis texte', button_style='info')
out_text_load = widgets.Output()

def _on_load_text(b):
    with out_text_load:
        clear_output()
        txt = contour_textarea.value.strip()
        if not txt:
            print('Aucun texte fourni.')
            return
        pts = []
        for ln in txt.splitlines():
            line = ln.strip()
            if not line:
                continue
            p = parse_point(line)
            if p is None:
                print(f'Ligne invalide ignorÃƒÆ’Ã‚Â©e: {line!r}')
                continue
            pts.append(p)
        if len(pts) < 3:
            print('Au moins 3 points valides sont nÃƒÆ’Ã‚Â©cessaires pour dÃƒÆ’Ã‚Â©finir un contour fermÃƒÆ’Ã‚Â©.')
            return
        pts = close_polygon(pts)
        _set_global_contour_and_plot(pts)
        print('Contour chargÃƒÆ’Ã‚Â© depuis le texte.')

load_text_btn.on_click(_on_load_text)

# Afficher la premiÃƒÆ’Ã‚Â¨re interface
display(widgets.VBox([contour_textarea, load_text_btn, out_text_load]))

# --- Interface 2 : saisie point-par-point via widgets (plus conviviale que input()) ---
_points_for_widget_input = []
pts_out = widgets.Output()

x_input = widgets.FloatText(value=0.0, description='x:')
y_input = widgets.FloatText(value=0.0, description='y:')
add_point_btn = widgets.Button(description='Ajouter point', button_style='primary')
finish_btn = widgets.Button(description='Terminer contour', button_style='success')
reset_btn = widgets.Button(description='Reset points', button_style='warning')

def _refresh_points_display():
    with pts_out:
        clear_output()
        if len(_points_for_widget_input) == 0:
            print('Aucun point ajoutÃƒÆ’Ã‚Â© (interface widgets).')
        else:
            for i,p in enumerate(_points_for_widget_input):
                print(f'{i+1}: {p[0]:.4f}, {p[1]:.4f}')
        print('\nEntrez au moins 3 points. Cliquez sur "Terminer contour" pour fermer et charger le contour.')

def _on_add_point(b):
    p = (float(x_input.value), float(y_input.value))
    # VÃƒÆ’Ã‚Â©rifier si le point est proche du premier point
    if len(_points_for_widget_input) > 0 and len(_points_for_widget_input) >= 3:
        p0 = _points_for_widget_input[0]
        dx = abs(p[0] - p0[0])
        dy = abs(p[1] - p0[1])
        if dx < 1e-9 and dy < 1e-9:
            with pts_out:
                print('Le point ajoutÃƒÆ’Ã‚Â© est identique au premier et il y a dÃƒÆ’Ã‚Â©jÃƒÆ’Ã‚Â  >=3 points; cliquez "Terminer contour" pour fermer.')
            return
    _points_for_widget_input.append(p)
    _refresh_points_display()

def _on_finish(b):
    if len(_points_for_widget_input) < 3:
        with pts_out:
            print('Au moins 3 points requis pour terminer le contour.')
        return
    pts_closed = close_polygon(_points_for_widget_input)
    _set_global_contour_and_plot(pts_closed)
    with pts_out:
        print('Contour terminÃƒÆ’Ã‚Â© et chargÃƒÆ’Ã‚Â© depuis l\'interface widgets.')
    _points_for_widget_input.clear()
    _refresh_points_display()

def _on_reset(b):
    _points_for_widget_input.clear()
    _refresh_points_display()

add_point_btn.on_click(_on_add_point)
finish_btn.on_click(_on_finish)
reset_btn.on_click(_on_reset)

# Affichage de la mini-interface widgets pour saisie manuelle
display(widgets.VBox([
    widgets.HBox([x_input, y_input, add_point_btn]),
    widgets.HBox([finish_btn, reset_btn]),
    pts_out
]))

_refresh_points_display()

# Note: la saisie via `input()` (console) et la fonction de contour prÃƒÆ’Ã‚Â©dÃƒÆ’Ã‚Â©fini ont ÃƒÆ’Ã‚Â©tÃƒÆ’Ã‚Â© supprimÃƒÆ’Ã‚Â©es
# pour simplifier l'interface. Utilisez la Textarea ou l'interface pointÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬ËœparÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Ëœpoint ciÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Ëœdessus.


In [3]:
# Calcul du c_moyen par subdivision spanwise (N subdivisions, c_moy_i = 0.5*(cr+ct))
def compute_cmoy_from_contour(polygon_pts, N=5, plot=True):
    import numpy as np
    pts = np.array(polygon_pts, dtype=float)
    pts = close_polygon(pts)
    y_min = np.min(pts[:,1]); y_max = np.max(pts[:,1])
    ys = np.linspace(y_min, y_max, N+1)
    cmoy_list = []
    for i in range(N):
        y0 = ys[i]; y1 = ys[i+1]
        xs0 = intersections_with_line_y(pts, y0)
        xs1 = intersections_with_line_y(pts, y1)
        if len(xs0) >= 2 and len(xs1) >= 2:
            # cr = corde ÃƒÆ’Ã‚Â  y0 (infÃƒÆ’Ã‚Â©rieure de la subdivision), ct = corde ÃƒÆ’Ã‚Â  y1 (supÃƒÆ’Ã‚Â©rieure)
            cr = np.max(xs0) - np.min(xs0)
            ct = np.max(xs1) - np.min(xs1)
            cm_i = 0.5 * (cr + ct)
        else:
            # si pas de corde dÃƒÆ’Ã‚Â©tectÃƒÆ’Ã‚Â©e ÃƒÆ’Ã‚Â  ce niveau, on met zÃƒÆ’Ã‚Â©ro
            cr = 0.0; ct = 0.0; cm_i = 0.0
        cmoy_list.append(cm_i)
    # Moyenne telle que demandÃƒÆ’Ã‚Â©e: somme des cmoy_i divisÃƒÆ’Ã‚Â©e par le nombre de subdivisions N
    cmoy = float(np.sum(cmoy_list) / float(N))
    if plot:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7,4))
        ax.plot(pts[:,0], pts[:,1], '-k')
        # tracer les cordes moyennes comme segments en leur y moyenne
        for i in range(N):
            y0 = ys[i]; y1 = ys[i+1]
            midy = 0.5*(y0+y1)
            cm_i = cmoy_list[i]
            # tracer segment centrÃƒÆ’Ã‚Â© sur x=0 de longueur cm_i (pour visualisation)
            # mieux: placer segment entre les intersections interpolÃƒÆ’Ã‚Â©es si disponibles
            xs_lo = intersections_with_line_y(pts, y0)
            xs_hi = intersections_with_line_y(pts, y1)
            if len(xs_lo) >= 2 and len(xs_hi) >= 2:
                xl = 0.5*(np.min(xs_lo) + np.min(xs_hi))
                xr = xl + cm_i
                ax.plot([xl, xr], [midy, midy], '-C1')
                ax.plot([0.5*(xl+xr)], [midy], 'rx')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_title(f'c_moy par subdivision (N={N}) - c_moy moyen = {cmoy:.4f} m')
        ax.legend()
        plt.show()
    return {'cmoy_list': cmoy_list, 'cmoy': cmoy, 'N': N, 'ys': ys}

# Si un contour a dÃƒÆ’Ã‚Â©jÃƒÆ’Ã‚Â  ÃƒÆ’Ã‚Â©tÃƒÆ’Ã‚Â© saisi, exÃƒÆ’Ã‚Â©cuter un exemple avec N=6
if 'wing_contour' in globals():
    try:
        res_cm = compute_cmoy_from_contour(wing_contour, N=5, plot=True)
        print('c_moy par bande :', res_cm['cmoy_list'])
        print('c_moy moyen (somme/N) :', res_cm['cmoy'])
    except Exception as e:
        print('Erreur compute_cmoy_from_contour :', e)

In [4]:
# 4 Cellule: calcul des Reynolds min/max ÃƒÆ’Ã‚Â  partir de c_moy (valeur calculÃƒÆ’Ã‚Â©e dans la cellule prÃƒÆ’Ã‚Â©cÃƒÆ’Ã‚Â©dente)
import ipywidgets as widgets
from IPython.display import display, clear_output

rho_widget = widgets.FloatText(value=1.225, description='rho (kg/m3):', step=0.001)
mu_widget = widgets.FloatText(value=1.81e-5, description='mu (Pa.s):', step=1e-7)
vmin_widget = widgets.FloatText(value=5.0, description='V min (m/s):', step=0.1)
vmax_widget = widgets.FloatText(value=20.0, description='V max (m/s):', step=0.1)
compute_btn = widgets.Button(description='Calculer Re')
out_re = widgets.Output()

display(widgets.VBox([rho_widget, mu_widget, vmin_widget, vmax_widget, compute_btn, out_re]))

def _calc_re(b):
    with out_re:
        clear_output()
        if 'wing_contour' not in globals():
            print('Aucun contour dÃƒÆ’Ã‚Â©fini. ExÃƒÆ’Ã‚Â©cuter la cellule de saisie du contour dabord.')
            return
        try:
            res = compute_cmoy_from_contour(wing_contour, N=6, plot=False)
            c_moy = float(res.get('cmoy', 0.0))
        except Exception as e:
            print('Erreur lors du calcul de c_moy :', e)
            return
        try:
            rho = float(rho_widget.value)
            mu = float(mu_widget.value)
            vmin = float(vmin_widget.value)
            vmax = float(vmax_widget.value)
        except Exception as e:
            print('Valeurs numÃƒÆ’Ã‚Â©riques invalides :', e)
            return
        if mu == 0.0:
            print('mu ne doit pas ÃƒÆ’Ã‚Âªtre zÃƒÆ’Ã‚Â©ro.')
            return
        Re_min = (vmin * rho * c_moy) / mu
        Re_max = (vmax * rho * c_moy) / mu
        print(f'c_moy utilisÃƒÆ’Ã‚Â© = {c_moy:.6g} m')
        print(f'Re_min = {Re_min:.6g}')
        print(f'Re_max = {Re_max:.6g}')

compute_btn.on_click(_calc_re)


In [5]:
# === Cellule 5 : Calculs aerodynamiques + TracÃƒÆ’Ã‚Â© Mt vs DeltaX ===
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

# N bandes fixe
N_USED_FIXED = 100

# Constantes air standard (utilisees pour convertir V -> Re)
RHO_AIR_DEFAULT = 1.225   # kg/m3
MU_AIR_DEFAULT = 1.81e-5  # Pa.s


def _available_profiles_from_folder(folder='.'):
    p = Path(folder)
    files = sorted(p.glob('NACA *.txt'))
    out = []
    for f in files:
        name = f.stem.replace(' ','').upper()
        if name.startswith('NACA'):
            out.append(name)
    return out


def _resolve_profile_file(profile_name, folder='.'):
    p = Path(folder)
    profile = str(profile_name).replace(' ','').upper()
    if not profile.startswith('NACA'):
        profile = 'NACA' + profile

    if len(profile) >= 8 and profile[4:].isdigit():
        candidate = p / f"NACA {profile[4:]}.txt"
        if candidate.exists():
            return candidate

    for f in p.glob('NACA *.txt'):
        if f.stem.replace(' ','').upper() == profile:
            return f

    raise FileNotFoundError(f"Fichier polaire introuvable pour {profile}")


def _load_polar_file(file_path):
    re_data = {}
    current_re = None

    with open(file_path, 'r', encoding='utf-8') as fh:
        for raw in fh:
            line = raw.strip()
            if not line:
                continue

            if line.startswith('Mach') and 'Re' in line:
                m = re.search(r'Re\s*=\s*([0-9]+(?:\.[0-9]+)?)', line)
                if m:
                    current_re = float(m.group(1))
                    re_data.setdefault(current_re, [])
                continue

            if current_re is None:
                continue

            if line.startswith('Nom') or line.startswith('Fini de surface') or line.startswith('ÃƒÅ½Ã‚Â±') or line.startswith('[Ãƒâ€šÃ‚Â°]'):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            try:
                alpha = float(parts[0])
                cz = float(parts[1])
                cx = float(parts[2])
                cm = float(parts[3])
                re_data[current_re].append((alpha, cz, cx, cm))
            except ValueError:
                continue

    if not re_data:
        raise ValueError(f"Aucune donnee polaire lisible dans {file_path}")

    for re_key in list(re_data.keys()):
        df = pd.DataFrame(re_data[re_key], columns=['alpha', 'Cl', 'Cd', 'Cm'])
        df = df.sort_values('alpha').drop_duplicates(subset='alpha', keep='last').reset_index(drop=True)
        re_data[re_key] = df

    return re_data


def _interp_coeffs_at_alpha(df_alpha, alpha_target):
    x = df_alpha['alpha'].to_numpy(dtype=float)
    cl = df_alpha['Cl'].to_numpy(dtype=float)
    cd = df_alpha['Cd'].to_numpy(dtype=float)
    cm = df_alpha['Cm'].to_numpy(dtype=float)
    a = float(alpha_target)
    return {
        'Cl': float(np.interp(a, x, cl))
        , 'Cd': float(np.interp(a, x, cd))
        , 'Cm': float(np.interp(a, x, cm))
    }


def _interp_coeffs_for_re_alpha(re_tables, re_target, alpha_target):
    re_sorted = sorted(re_tables.keys())
    re_t = float(re_target)

    if re_t <= re_sorted[0]:
        c = _interp_coeffs_at_alpha(re_tables[re_sorted[0]], alpha_target)
        c['Re_used'] = re_sorted[0]
        return c
    if re_t >= re_sorted[-1]:
        c = _interp_coeffs_at_alpha(re_tables[re_sorted[-1]], alpha_target)
        c['Re_used'] = re_sorted[-1]
        return c

    for i in range(len(re_sorted) - 1):
        re0 = re_sorted[i]
        re1 = re_sorted[i + 1]
        if re0 <= re_t <= re1:
            c0 = _interp_coeffs_at_alpha(re_tables[re0], alpha_target)
            c1 = _interp_coeffs_at_alpha(re_tables[re1], alpha_target)
            t = (re_t - re0) / (re1 - re0) if re1 != re0 else 0.0
            return {
                'Cl': float(c0['Cl'] + t * (c1['Cl'] - c0['Cl'])),
                'Cd': float(c0['Cd'] + t * (c1['Cd'] - c0['Cd'])),
                'Cm': float(c0['Cm'] + t * (c1['Cm'] - c0['Cm'])),
                'Re_used': (re0, re1),
            }

    c = _interp_coeffs_at_alpha(re_tables[re_sorted[-1]], alpha_target)
    c['Re_used'] = re_sorted[-1]
    return c


def _simple_finite_lookup(alpha_inf_deg, cl_inf, cd_inf, AR):
    if AR <= 0.0 or np.isnan(AR):
        raise ValueError('Aspect ratio AR invalide.')

    alpha_inf_rad = np.deg2rad(float(alpha_inf_deg))
    if abs(alpha_inf_rad) < 1e-12:
        alpha_fini_deg = float(alpha_inf_deg)
    else:
        alpha_fini_deg = float(alpha_inf_deg) + (float(cl_inf) / (np.pi * AR)) * (180.0 / np.pi)

    cd_fini = float(cd_inf) + (float(cl_inf) ** 2) / (np.pi * AR)

    return {
        'Cd_fini_calc': float(cd_fini),
        'alpha_fini_deg': float(alpha_fini_deg),
    }


def _compute_moments_from_df(df, Cl, V, rho, Cm):
    h_list = df['h'].values
    Mr_list = []
    Mt_list = []
    q = 0.5 * float(rho) * (float(V) ** 2)
    for i in range(len(df)):
        h = float(df.at[i, 'h'])
        cm_i = float(df.at[i, 'cm'])
        lever_arm = float(np.sum(h_list[:i])) + 0.5 * h
        L_local = float(Cl) * float(cm_i) * q * h
        Mr_i = L_local * lever_arm
        Mt_i = float(Cm) * h * (cm_i ** 2) * q
        Mr_list.append(Mr_i)
        Mt_list.append(Mt_i)
    out = df.copy()
    out['Mr_i'] = Mr_list
    out['Mt_i'] = Mt_list
    out['L_calc_i'] = float(Cl) * out['h'] * out['cm'] * q
    return out


# --- Fonction de tracÃƒÆ’Ã‚Â© (anciennement cellule 6) ---
def plot_mt_versus_deltax():
    """Trace Mt en fonction de DeltaX pour les 3 cas (min, mid, max) depuis les rÃƒÆ’Ã‚Â©sultats de cette cellule."""
    # ParamÃƒÆ’Ã‚Â¨tres par dÃƒÆ’Ã‚Â©faut prÃƒÆ’Ã‚Â©levÃƒÆ’Ã‚Â©s depuis les widgets existants
    try:
        N_used = int(N_USED_FIXED)
        rho_val = float(rho_widget.value)
    except Exception:
        N_used = 10
        rho_val = 1.225

    # Valeurs par dÃƒÆ’Ã‚Â©faut pour Cl/Cm/V si rien d'autre
    Cl_min_val = 0.9
    Cm_min_val = 0.2741
    u_min_val = 5.0
    Cl_max_val = 1.1
    Cm_max_val = 0.2741
    u_max_val = 20.0

    # Tenter d'extraire vitesses et Cm depuis le rÃƒÆ’Ã‚Â©sultat de calcul
    use_from_cell5 = False
    if 'cell6_lookup_results' in globals():
        try:
            prev = globals().get('cell6_lookup_results')
            df_prev = prev.get('summary') if isinstance(prev, dict) else None
            if df_prev is not None and hasattr(df_prev, 'columns'):
                # rÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â©rer les 3 premiÃƒÆ’Ã‚Â¨res vitesses
                if 'V_mps' in df_prev.columns:
                    v_list = df_prev['V_mps'].astype(float).tolist()
                else:
                    v_list = None
                # rÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â©rer Cm prÃƒÆ’Ã‚Â©fÃƒÆ’Ã‚Â©rentiellement depuis la colonne 'Cm_f_table' puis 'Cm_i' puis 'Cm'
                cm_col = None
                for c in ('Cm_f_table', 'Cm_i', 'Cm'):
                    if c in df_prev.columns:
                        cm_col = c
                        break
                cm_list = df_prev[cm_col].astype(float).tolist() if cm_col is not None else None
                if v_list and cm_list and len(v_list) >= 3 and len(cm_list) >= 3:
                    u_min_val, u_mid_val, u_max_val = v_list[0], v_list[1], v_list[2]
                    Cm_min_val, Cm_mid_val, Cm_max_val = cm_list[0], cm_list[1], cm_list[2]
                    # essayer aussi de rÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â©rer Cl si disponible
                    if 'Cl_f_table' in df_prev.columns:
                        cl_list = df_prev['Cl_f_table'].astype(float).tolist()
                        Cl_min_val, Cl_mid_val, Cl_max_val = cl_list[0], cl_list[1], cl_list[2]
                    use_from_cell5 = True
        except Exception:
            use_from_cell5 = False

    # Si non trouvÃƒÆ’Ã‚Â©, tenter widgets (compatibilitÃƒÆ’Ã‚Â© ancienne)
    if not use_from_cell5:
        try:
            Cl_min_val = float(Cl_min_widget.value)
            Cm_min_val = float(Cm_min_widget.value)
            u_min_val = float(vmin_widget.value)
            Cl_max_val = float(Cl_max_widget.value)
            Cm_max_val = float(Cm_max_widget.value)
            u_max_val = float(vmax_widget.value)
        except Exception:
            pass

    # Calculer valeurs intermÃƒÆ’Ã‚Â©diaires (mid)
    try:
        Cl_mid_val
    except NameError:
        Cl_mid_val = 0.5 * (Cl_min_val + Cl_max_val)
    try:
        Cm_mid_val
    except NameError:
        Cm_mid_val = 0.5 * (Cm_min_val + Cm_max_val)
    try:
        u_mid_val
    except NameError:
        u_mid_val = 0.5 * (u_min_val + u_max_val)

    # VÃƒÆ’Ã‚Â©rifier prÃƒÆ’Ã‚Â©requis
    if 'wing_contour' not in globals():
        print('Aucun contour dÃƒÆ’Ã‚Â©tectÃƒÆ’Ã‚Â©. Chargez un contour avant d\'exÃƒÆ’Ã‚Â©cuter ce tracÃƒÆ’Ã‚Â©.')
        return

    # --- RÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â©rer cmoy en prioritÃƒÆ’Ã‚Â© depuis res_cm si disponible ---
    cmoy = None
    cmoy_source = None
    if 'res_cm' in globals() and isinstance(res_cm, dict) and ('cmoy' in res_cm):
        try:
            cmoy = float(res_cm.get('cmoy', 0.0))
            cmoy_source = 'res_cm'
        except Exception:
            cmoy = None
    if cmoy is None and 'cmoy' in globals():
        try:
            cmoy = float(globals().get('cmoy', 0.0))
            cmoy_source = 'variable globale cmoy'
        except Exception:
            cmoy = None
    # fallback: recalculer
    if cmoy is None:
        try:
            res_cmoy = compute_cmoy_from_contour(wing_contour, N=N_used, plot=False)
            cmoy = float(res_cmoy.get('cmoy', 0.0))
            cmoy_source = 'recalculÃƒÆ’Ã‚Â© (compute_cmoy_from_contour)'
        except Exception as e:
            print('Erreur calcul c_moy :', e)
            cmoy = 0.0
            cmoy_source = 'fallback 0.0'

    # dÃƒÆ’Ã‚Â©coupage en bandes pour rÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â©rer h_i et cm_i
    try:
        base = compute_cmoyen_bands(wing_contour, N_used, origin=wing_origin, L_total=None, plot=False)
        if base is None:
            raise RuntimeError('DÃƒÆ’Ã‚Â©coupage en bandes renvoie None')
        Ls = np.array(base['Ls'], dtype=float)
        bands = base['bands']
        df_bands = _format_bands_to_df(bands, Ls, wing_origin)
    except Exception as e:
        print('Erreur lors du dÃƒÆ’Ã‚Â©coupage en bandes :', e)
        df_bands = None

    if df_bands is None or df_bands.shape[0] == 0:
        print('Impossible de calculer Mt_fini_total : pas de bandes valides.')
        return

    # Calcul du terme commun sum(h_i * cm_i^2)
    sum_term = np.sum(df_bands['h'].values * (df_bands['cm'].values ** 2))

    # --- Obtenir Mt_fini_total depuis df_min/df_mid/df_max si disponibles ---
    Mt_fini_total_min = None
    Mt_fini_total_mid = None
    Mt_fini_total_max = None

    if 'df_min' in globals() and hasattr(df_min, 'columns') and 'Mt_fini_i' in df_min.columns:
        try:
            Mt_fini_total_min = float(df_min['Mt_fini_i'].sum())
        except Exception:
            Mt_fini_total_min = None
    if Mt_fini_total_min is None:
        Mt_fini_total_min = Cm_min_val * 0.5 * rho_val * (u_min_val ** 2) * sum_term

    if 'df_mid' in globals() and hasattr(df_mid, 'columns') and 'Mt_fini_i' in df_mid.columns:
        try:
            Mt_fini_total_mid = float(df_mid['Mt_fini_i'].sum())
        except Exception:
            Mt_fini_total_mid = None
    if Mt_fini_total_mid is None:
        Mt_fini_total_mid = Cm_mid_val * 0.5 * rho_val * (u_mid_val ** 2) * sum_term

    if 'df_max' in globals() and hasattr(df_max, 'columns') and 'Mt_fini_i' in df_max.columns:
        try:
            Mt_fini_total_max = float(df_max['Mt_fini_i'].sum())
        except Exception:
            Mt_fini_total_max = None
    if Mt_fini_total_max is None:
        Mt_fini_total_max = Cm_max_val * 0.5 * rho_val * (u_max_val ** 2) * sum_term

    # Calcul de B (envergure approximative)
    ys = wing_contour[:-1,1]
    B = float(np.max(ys) - np.min(ys))

    # DÃƒÆ’Ã‚Â©terminer la plage horizontale (DeltaX)
    xs_all = wing_contour[:-1,0]
    x_min = float(np.min(xs_all))
    x_max = float(np.max(xs_all))
    if np.isclose(x_min, x_max):
        x_min = x_min - 0.1
        x_max = x_max + 0.1

    DeltaX = np.linspace(x_min, x_max, 401)

    # Calcul des valeurs Mt pour MIN, MID et MAX
    Mt_vals_min = Mt_fini_total_min - Cl_min_val * cmoy * B * 0.5 * rho_val * (u_min_val ** 2) * DeltaX
    Mt_vals_mid = Mt_fini_total_mid - Cl_mid_val * cmoy * B * 0.5 * rho_val * (u_mid_val ** 2) * DeltaX
    Mt_vals_max = Mt_fini_total_max - Cl_max_val * cmoy * B * 0.5 * rho_val * (u_max_val ** 2) * DeltaX

    # TracÃƒÆ’Ã‚Â©
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(DeltaX, Mt_vals_min, '-C0', label=f'CAS MIN: Cm={Cm_min_val:.4f}, V={u_min_val:.2f} m/s')
    ax.plot(DeltaX, Mt_vals_mid, '-C2', label=f'CAS MID: Cm={Cm_mid_val:.4f}, V={u_mid_val:.2f} m/s')
    ax.plot(DeltaX, Mt_vals_max, '--C1', label=f'CAS MAX: Cm={Cm_max_val:.4f}, V={u_max_val:.2f} m/s')
    ax.axvline(0.0, color='k', linewidth=0.6)
    ax.axhline(0.0, color='k', linewidth=0.6)

    # DÃƒÆ’Ã‚Â©tection d'intersections entre paires
    intersections = []
    def find_intersections(y1, y2):
        diff = y1 - y2
        sign_changes = np.where(np.sign(diff[:-1]) * np.sign(diff[1:]) < 0)[0]
        pts = []
        for idx in sign_changes:
            x0 = DeltaX[idx]; x1 = DeltaX[idx+1]
            y0 = diff[idx]; y1v = diff[idx+1]
            if (y1v - y0) == 0:
                root_x = x0
            else:
                root_x = x0 - y0 * (x1 - x0) / (y1v - y0)
            Mt_at_idx = y1[idx]; Mt_at_next = y1[idx+1]
            Mt_root = Mt_at_idx + (Mt_at_next - Mt_at_idx) * ((root_x - x0) / (x1 - x0)) if (x1 - x0) != 0 else Mt_at_idx
            pts.append((root_x, Mt_root))
        return pts

    pts_mm = find_intersections(Mt_vals_min, Mt_vals_mid)
    for (xi, yi) in pts_mm:
        intersections.append((xi, yi, 'MIN-MID'))
    pts_mx = find_intersections(Mt_vals_mid, Mt_vals_max)
    for (xi, yi) in pts_mx:
        intersections.append((xi, yi, 'MID-MAX'))
    pts_Mm = find_intersections(Mt_vals_min, Mt_vals_max)
    for (xi, yi) in pts_Mm:
        intersections.append((xi, yi, 'MIN-MAX'))

    # Marquer les intersections sur le graphe
    if len(intersections) == 0:
        ann_text = 'Aucune intersection dÃƒÆ’Ã‚Â©tectÃƒÆ’Ã‚Â©e entre les cas dans la plage DeltaX.'
        ax.text(0.01, 0.95, ann_text, transform=ax.transAxes, fontsize=9, va='top')
        print(ann_text)
    else:
        added_labels = set()
        for (xi, yi, tag) in intersections:
            label = f'Intersection {tag}' if tag not in added_labels else None
            ax.plot(xi, yi, 'o', color='red', markersize=4, label=label)
            added_labels.add(tag)

    ax.legend()
    ax.set_xlabel('DeltaX (m)')
    ax.set_ylabel('Mt (N.m)')
    ax.set_title('Mt en fonction de DeltaX ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â cas MIN, MID et MAX (avec intersections)')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlim(x_min, x_max)
    plt.show()

    # Afficher rÃƒÆ’Ã‚Â©sultats
    print(f'cmoy used = {cmoy:.6f} m (source: {cmoy_source})')
    print(f'Parameters MIN: Cl={Cl_min_val:.2f}, Cm={Cm_min_val:.4f}, V={u_min_val:.2f} m/s')
    print(f'Mt (MIN) at DeltaX=0 = {Mt_fini_total_min:.2f} N.m')
    print(f'Parameters MID: Cl={Cl_mid_val:.2f}, Cm={Cm_mid_val:.4f}, V={u_mid_val:.2f} m/s')
    print(f'Mt (MID) at DeltaX=0 = {Mt_fini_total_mid:.2f} N.m')
    print(f'Parameters MAX: Cl={Cl_max_val:.2f}, Cm={Cm_max_val:.4f}, V={u_max_val:.2f} m/s')
    print(f'Mt (MAX) at DeltaX=0 = {Mt_fini_total_max:.2f} N.m')

    if len(intersections) > 0:
        print('\nIntersections dÃƒÆ’Ã‚Â©tectÃƒÆ’Ã‚Â©es:')
        for (xi, yi, tag) in intersections:
            print(f' - {tag}: DeltaX = {xi:.4f} m, Mt = {yi:.2f} N.m')


# --- Widgets ---
profiles_list = _available_profiles_from_folder('.')
if len(profiles_list) == 0:
    profiles_list = ['NACA0012']

profile_widget = widgets.Dropdown(
    options=profiles_list,
    value=profiles_list[0],
    description='Profil:'
 )

alpha1_widget = widgets.FloatText(value=2.0, description='alpha 1 (deg):', step=0.1)
alpha2_widget = widgets.FloatText(value=6.0, description='alpha 2 (deg):', step=0.1)
alpha3_widget = widgets.FloatText(value=10.0, description='alpha 3 (deg):', step=0.1)

v1_widget = widgets.FloatText(value=5.0, description='V1 (m/s):', step=0.1)
v2_widget = widgets.FloatText(value=10.0, description='V2 (m/s):', step=0.1)
v3_widget = widgets.FloatText(value=15.0, description='V3 (m/s):', step=0.1)

compute_moments_btn = widgets.Button(description='Calculer (infini + mÃƒÆ’Ã‚Â©thode angle corrigÃƒÆ’Ã‚Â©)', button_style='primary')
out_moments = widgets.Output()

display(widgets.VBox([
    widgets.HTML(f'<b>N bandes fixe : {N_USED_FIXED}</b>'),
    profile_widget,
    widgets.HBox([alpha1_widget, alpha2_widget, alpha3_widget]),
    widgets.HBox([v1_widget, v2_widget, v3_widget]),
    compute_moments_btn,
    out_moments,
]))


# --- Definitions locales de secours ---
if 'compute_cmoyen_bands' not in globals():
    def polygon_area_centroid_local(poly):
        x = np.asarray([p[0] for p in poly], dtype=float)
        y = np.asarray([p[1] for p in poly], dtype=float)
        if x.size == 0:
            return 0.0, np.nan, np.nan
        x2 = np.append(x, x[0]); y2 = np.append(y, y[0])
        A = 0.5 * np.sum(x2[:-1] * y2[1:] - y2[:-1] * x2[1:])
        if np.isclose(A, 0.0):
            return 0.0, np.nan, np.nan
        Cx = (1.0 / (6.0 * A)) * np.sum((x2[:-1] + x2[1:]) * (x2[:-1] * y2[1:] - x2[1:] * y2[:-1]))
        Cy = (1.0 / (6.0 * A)) * np.sum((y2[:-1] + y2[1:]) * (x2[:-1] * y2[1:] - x2[1:] * y2[:-1]))
        return abs(A), Cx, Cy

    def compute_cmoyen_bands(polygon_pts, Ny, origin=(0.0, 0.0), L_total=None, plot=False):
        pts = np.array(polygon_pts, dtype=float)
        try:
            pts = close_polygon(pts)
        except Exception:
            if not np.allclose(pts[0], pts[-1]):
                pts = np.vstack([pts, pts[0]])
        y_min = np.min(pts[:, 1]); y_max = np.max(pts[:, 1])
        if Ny < 1:
            raise ValueError('Ny must be >= 1')
        ys = np.linspace(y_min, y_max, Ny + 1)
        bands = []
        total_area = 0.0
        for i in range(Ny):
            y0 = ys[i]; y1 = ys[i + 1]
            xs0 = intersections_with_line_y(pts, y0)
            xs1 = intersections_with_line_y(pts, y1)
            if len(xs0) < 2 or len(xs1) < 2:
                continue
            x_left0, x_right0 = np.min(xs0), np.max(xs0)
            x_left1, x_right1 = np.min(xs1), np.max(xs1)
            poly = np.array([[x_left0, y0], [x_right0, y0], [x_right1, y1], [x_left1, y1]])
            A, cx, cy = polygon_area_centroid_local(poly)
            if A <= 0 or np.isnan(cx):
                continue
            bands.append({'poly': poly, 'area': A, 'cx': cx, 'cy': cy, 'y0': y0, 'y1': y1})
            total_area += A
        if total_area == 0.0:
            return None
        if L_total is None:
            L_total = 1.0
        areas = np.array([b['area'] for b in bands])
        Ls = L_total * (areas / np.sum(areas))
        return {'bands': bands, 'total_area': total_area, 'Ls': Ls, 'ys': ys}
    globals()['compute_cmoyen_bands'] = compute_cmoyen_bands

def _format_bands_to_df(bands, Ls, origin):
    rows = []
    x0, _ = origin
    for ib, b in enumerate(bands):
        y0b = b['y0']; y1b = b['y1']
        h = float(y1b - y0b)
        Ai = float(b.get('area', 0.0))
        cx = float(b.get('cx', x0)) if not pd.isna(b.get('cx', np.nan)) else float(x0)
        cm_i = None
        try:
            if 'wing_contour' in globals():
                xs0 = intersections_with_line_y(wing_contour, y0b)
                xs1 = intersections_with_line_y(wing_contour, y1b)
                if len(xs0) >= 2 and len(xs1) >= 2:
                    cr = float(np.max(xs0) - np.min(xs0))
                    ct = float(np.max(xs1) - np.min(xs1))
                    cm_i = 0.5 * (cr + ct)
        except Exception:
            cm_i = None
        if cm_i is None:
            cm_i = float(cx - x0)
        Li = float(Ls[ib])
        rows.append({'i': ib, 'y0': y0b, 'y1': y1b, 'h': h, 'area': Ai, 'cx': cx, 'cm': cm_i, 'L_i': Li})
    return pd.DataFrame(rows)

def _on_compute(_):
    with out_moments:
        clear_output()

        missing = []
        if 'wing_contour' not in globals():
            missing.append('wing_contour (executer la cellule de saisie du contour)')
        if 'wing_origin' not in globals():
            missing.append('wing_origin (executer la cellule de saisie du contour)')
        if 'compute_cmoy_from_contour' not in globals():
            missing.append('compute_cmoy_from_contour (executer la cellule de calcul c_moy)')
        if missing:
            print('Impossible d executer : elements manquants dans le noyau:')
            for m in missing:
                print(' -', m)
            return

        profile = str(profile_widget.value).replace(' ', '').upper()
        alphas = [float(alpha1_widget.value), float(alpha2_widget.value), float(alpha3_widget.value)]
        speeds = [float(v1_widget.value), float(v2_widget.value), float(v3_widget.value)]

        if any(v <= 0.0 for v in speeds):
            print('Les vitesses doivent etre strictement positives.')
            return

        try:
            c_moy = float(compute_cmoy_from_contour(wing_contour, N=N_USED_FIXED, plot=False).get('cmoy', 0.0))
            base_split = compute_cmoyen_bands(wing_contour, N_USED_FIXED, origin=wing_origin, L_total=None, plot=False)
        except Exception as e:
            print('Erreur geometrie (cmoy / decoupage):', e)
            return

        if base_split is None:
            print('Aucune aire detectee dans le decoupage (verifier le contour / N).')
            return

        total_area = float(base_split['total_area'])
        ys = wing_contour[:-1, 1]
        b = float(np.max(ys) - np.min(ys))
        AR = (b ** 2) / total_area if total_area > 0 else np.nan

        try:
            polar_file = _resolve_profile_file(profile, folder='.')
            polar_tables = _load_polar_file(polar_file)
        except Exception as e:
            print('Erreur chargement polaires:', e)
            return

        print(f'Profil choisi: {profile} | Fichier: {polar_file.name}')
        print(f'c_moy = {c_moy:.4f} m | S = {total_area:.4f} m2 | b = {b:.4f} m | AR = {AR:.3f}')
        print(f'Constantes air utilisees pour Re: rho={RHO_AIR_DEFAULT:.3f} kg/m3, mu={MU_AIR_DEFAULT:.3e} Pa.s')

        rows = []
        details = {}

        for idx, (alpha_deg, V) in enumerate(zip(alphas, speeds), start=1):
            Re_case = (RHO_AIR_DEFAULT * V * c_moy) / MU_AIR_DEFAULT
            coeffs_inf = _interp_coeffs_for_re_alpha(polar_tables, Re_case, alpha_deg)

            Cl_inf = float(coeffs_inf['Cl'])
            Cd_inf = float(coeffs_inf['Cd'])
            Cm_inf = float(coeffs_inf['Cm'])

            try:
                corr = _simple_finite_lookup(alpha_deg, Cl_inf, Cd_inf, AR)
            except Exception as e:
                print(f'Cas {idx}: erreur calcul angle corrige:', e)
                continue

            alpha_fini = float(corr['alpha_fini_deg'])
            Cd_fini_calc = float(corr['Cd_fini_calc'])

            # On relit les tables au nouvel angle corrige pour les autres valeurs
            coeffs_fini = _interp_coeffs_for_re_alpha(polar_tables, Re_case, alpha_fini)
            Cl_f_table = float(coeffs_fini['Cl'])
            Cd_fini_table = float(coeffs_fini['Cd'])
            Cm_f_table = float(coeffs_fini['Cm'])

            q = 0.5 * RHO_AIR_DEFAULT * (V ** 2)
            L_inf_total = Cl_inf * q * total_area
            Fd_inf_total = Cd_inf * q * total_area
            L_fini_total = Cl_inf * q * total_area
            Fd_fini_total = Cd_fini_calc * q * total_area

            try:
                split_inf = compute_cmoyen_bands(wing_contour, N_USED_FIXED, origin=wing_origin, L_total=L_inf_total, plot=False)
                split_fini = compute_cmoyen_bands(wing_contour, N_USED_FIXED, origin=wing_origin, L_total=L_fini_total, plot=False)
            except Exception as e:
                print(f'Erreur decoupage cas {idx}:', e)
                continue

            if split_inf is None or split_fini is None:
                print(f'Cas {idx}: decoupage invalide, ignore.')
                continue

            Ls_inf = np.array(split_inf['Ls'], dtype=float)
            Ls_fini = np.array(split_fini['Ls'], dtype=float)

            df_inf = _format_bands_to_df(split_inf['bands'], Ls_inf, wing_origin)
            df_fini = _format_bands_to_df(split_fini['bands'], Ls_fini, wing_origin)

            df_inf = _compute_moments_from_df(df_inf, Cl=Cl_inf, V=V, rho=RHO_AIR_DEFAULT, Cm=Cm_inf)
            df_fini = _compute_moments_from_df(df_fini, Cl=Cl_inf, V=V, rho=RHO_AIR_DEFAULT, Cm=Cm_f_table)

            Mr_inf_total = float(df_inf['Mr_i'].sum())
            Mt_inf_total = float(df_inf['Mt_i'].sum())
            Mr_fini_total = float(df_fini['Mr_i'].sum())
            Mt_fini_total = float(df_fini['Mt_i'].sum())

            re_used = coeffs_inf.get('Re_used')
            if isinstance(re_used, tuple):
                re_used_txt = f"{int(re_used[0])}-{int(re_used[1])}"
            else:
                re_used_txt = str(int(re_used))

            rows.append({
                'cas': idx,
                'alpha_i_deg': alpha_deg,
                'alpha_f_deg': alpha_fini,
                'V_mps': V,
                'Re_calc': Re_case,
                'Re_table': re_used_txt,
                'Cl_i': Cl_inf,
                'Cd_i': Cd_inf,
                'Cm_i': Cm_inf,
                'Cl_f_table': Cl_f_table,
                'Cd_f_calc': Cd_fini_calc,
                'Cd_f_table': Cd_fini_table,
                'Cm_f_table': Cm_f_table,
                'L_i_N': float(df_inf['L_calc_i'].sum()),
                'Fd_i_N': Fd_inf_total,
                'Mr_i_Nm': Mr_inf_total,
                'Mt_i_Nm': Mt_inf_total,
                'L_f_N': float(df_fini['L_calc_i'].sum()),
                'Fd_f_N': Fd_fini_total,
                'Mr_f_Nm': Mr_fini_total,
                'Mt_f_Nm': Mt_fini_total,
            })

            details[f'cas_{idx}'] = {
                'infinite': df_inf,
                'finite': df_fini,
            }

        if not rows:
            print('Aucun cas valide calcule.')
            return

        df_summary = pd.DataFrame(rows)

        df_inf = df_summary[[
            'cas', 'alpha_i_deg', 'V_mps', 'Re_calc', 'Re_table',
            'Cl_i', 'Cd_i', 'Cm_i',
            'L_i_N', 'Fd_i_N', 'Mr_i_Nm', 'Mt_i_Nm'
        ]].copy()

        df_fini = df_summary[[
            'cas', 'alpha_i_deg', 'alpha_f_deg', 'V_mps', 'Re_calc', 'Re_table',
            'Cd_f_calc', 'Cm_f_table',
            'L_f_N', 'Fd_f_N', 'Mr_f_Nm', 'Mt_f_Nm'
        ]].copy()

        display(HTML('<h3>Resultats section infinie</h3>'))
        display(df_inf.style.format({
            'alpha_i_deg': '{:.2f}',
            'V_mps': '{:.2f}',
            'Re_calc': '{:.0f}',
            'Cl_i': '{:.4f}',
            'Cd_i': '{:.5f}',
            'Cm_i': '{:.4f}',
            'L_i_N': '{:.2f}',
            'Fd_i_N': '{:.2f}',
            'Mr_i_Nm': '{:.2f}',
            'Mt_i_Nm': '{:.2f}',
        }))

        display(HTML('<h3>Resultats aile finie - methode angle corrige</h3>'))
        display(df_fini.style.format({
            'alpha_i_deg': '{:.2f}',
            'alpha_f_deg': '{:.2f}',
            'V_mps': '{:.2f}',
            'Re_calc': '{:.0f}',
            'Cd_f_calc': '{:.5f}',
            'Cm_f_table': '{:.4f}',
            'L_f_N': '{:.2f}',
            'Fd_f_N': '{:.2f}',
            'Mr_f_Nm': '{:.2f}',
            'Mt_f_Nm': '{:.2f}',
        }))

        globals()['cell6_lookup_results'] = {
            'profile': profile,
            'polar_file': str(polar_file),
            'summary': df_summary,
            'summary_inf': df_inf,
            'summary_fini': df_fini,
            'details': details,
            'constants': {'rho': RHO_AIR_DEFAULT, 'mu': MU_AIR_DEFAULT},
            'geometry': {'c_moy': c_moy, 'S': total_area, 'b': b, 'AR': AR},
        }
        print('\nVariable creee: cell6_lookup_results')
        
        # Auto-trigger plot
        if 'plot_mt_versus_deltax' in globals():
            try:
                plot_mt_versus_deltax()
            except Exception as e:
                print(f'TracÃƒÆ’Ã‚Â© automatique ÃƒÆ’Ã‚Â©chouÃƒÆ’Ã‚Â©: {e}')


print('Cellule 5+6 prete : calcul (profil + 3 alphas + 3 vitesses) + tracÃƒÆ’Ã‚Â© Mt vs DeltaX.')

# Brancher le bouton
compute_moments_btn.on_click(_on_compute)

# Enregistrer la fonction de tracÃƒÆ’Ã‚Â© dans les globales (pour accÃƒÆ’Ã‚Â¨s externe si besoin)
globals()['plot_mt_versus_deltax'] = plot_mt_versus_deltax


Cellule 5+6 prete : calcul (profil + 3 alphas + 3 vitesses) + tracÃƒÆ’Ã‚Â© Mt vs DeltaX.


In [6]:
# === Cellule 6 : Section infinie + angle de finesse ===
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import numpy as np
import matplotlib.pyplot as plt

N_HYDRO_FIXED = 100

# Valeurs par defaut si non definies dans des cellules precedentes
if 'DEFAULT_SAFRAN' not in globals():
    DEFAULT_SAFRAN = "0.00,0.00\n0.08,0.00\n0.08,0.65\n0.00,0.65\n0.00,0.00"
if 'DEFAULT_QUILLE' not in globals():
    DEFAULT_QUILLE = "0.00,0.00\n0.20,0.00\n0.20,1.20\n0.00,1.20\n0.00,0.00"

# --- Widgets ---
safran_pts8_widget = widgets.Textarea(value=DEFAULT_SAFRAN, description='Safran pts:', rows=5)
quille_pts8_widget = widgets.Textarea(value=DEFAULT_QUILLE, description='Quille pts:', rows=5)
rho8_widget = widgets.FloatText(value=1025.0, description='rho eau:', step=1.0)
vmin8_widget = widgets.FloatText(value=1.0, description='V min (m/s):', step=0.1)
vmax8_widget = widgets.FloatText(value=3.0, description='V max (m/s):', step=0.1)
alpha_inf_min8_widget = widgets.FloatText(value=2.0, description='a min (deg):', step=0.5)
alpha_inf_max8_widget = widgets.FloatText(value=6.0, description='a max (deg):', step=0.5)
eoswald8_widget = widgets.FloatText(value=0.9, description='e oswald:', step=0.05)

s_cl_inf_min8 = widgets.FloatText(value=0.4, description='Cl s min:')
s_cm_inf_min8 = widgets.FloatText(value=0.02, description='Cm s min:')
s_cd_inf_min8 = widgets.FloatText(value=0.03, description='Cd s min:')

s_cl_inf_max8 = widgets.FloatText(value=0.55, description='Cl s max:')
s_cm_inf_max8 = widgets.FloatText(value=0.03, description='Cm s max:')
s_cd_inf_max8 = widgets.FloatText(value=0.04, description='Cd s max:')

q_cl_inf_min8 = widgets.FloatText(value=0.5, description='Cl q min:')
q_cm_inf_min8 = widgets.FloatText(value=0.02, description='Cm q min:')
q_cd_inf_min8 = widgets.FloatText(value=0.03, description='Cd q min:')

q_cl_inf_max8 = widgets.FloatText(value=0.65, description='Cl q max:')
q_cm_inf_max8 = widgets.FloatText(value=0.03, description='Cm q max:')
q_cd_inf_max8 = widgets.FloatText(value=0.04, description='Cd q max:')

plot8_btn = widgets.Button(description='Afficher contours', button_style='info')
compute8_btn = widgets.Button(description='Calcul section infinie et alpha_fini', button_style='success')
out8 = widgets.Output()

# --- Helpers ---
def _parse_point_local(s):
    s = str(s).strip()
    if not s:
        return None
    parts = s.split(',') if ',' in s else s.split()
    if len(parts) < 2:
        return None
    try:
        return (float(parts[0]), float(parts[1]))
    except Exception:
        return None


def _close_polygon_local(pts):
    arr = np.array(pts, dtype=float)
    if arr.shape[0] == 0:
        return arr
    if abs(arr[0, 0] - arr[-1, 0]) > 1e-9 or abs(arr[0, 1] - arr[-1, 1]) > 1e-9:
        arr = np.vstack([arr, arr[0:1]])
    return arr


def _parse_points_text(txt):
    parser = globals().get('parse_point', _parse_point_local)
    pts = []
    for i, line in enumerate(txt.strip().split('\n'), start=1):
        raw = line.strip()
        if not raw:
            continue
        p = parser(raw)
        if p is None:
            raise ValueError(f'Ligne invalide ({i}): {raw!r}')
        pts.append(p)
    if len(pts) < 3:
        raise ValueError(f'Au moins 3 points requis. Trouve {len(pts)}')
    closer = globals().get('close_polygon', _close_polygon_local)
    return closer(pts)


def _auto_origin(contour):
    xs = contour[:-1, 0]
    return (float(np.mean(xs)), 0.0)


def _plot_contours(s, os, q, oq):
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    for idx, (name, contour, origin) in enumerate([('Safran', s, os), ('Quille', q, oq)]):
        ax = axs[idx]
        ax.plot(contour[:, 0], contour[:, 1], 'b-', linewidth=2, label='Contour')
        ax.plot(origin[0], origin[1], 'r*', markersize=15, label='Origine')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_title(name)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='best')
    fig.suptitle('Contours hydrodynamiques')
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def compute_hydro_appendage_from_coeffs(name, contour, origin, Cl, Cm, Cd, rho, V):
    res_cmoy = compute_cmoy_from_contour(contour, N=10, plot=False)
    c_moy = float(res_cmoy.get('cmoy', 0.0))

    split0 = compute_cmoyen_bands(contour, N_HYDRO_FIXED, origin=origin, L_total=None, plot=False)
    if split0 is None:
        raise RuntimeError(f"Decoupage impossible pour {name}.")

    S = float(split0['total_area'])
    qdyn = 0.5 * float(rho) * (float(V) ** 2)
    L_total = float(Cl) * qdyn * S

    split = compute_cmoyen_bands(contour, N_HYDRO_FIXED, origin=origin, L_total=L_total, plot=False)
    if split is None:
        raise RuntimeError(f"Repartition impossible pour {name}.")

    rows = []
    h_prev = 0.0
    for b in split['bands']:
        y0 = float(b['y0'])
        y1 = float(b['y1'])
        h = y1 - y0

        xs0 = intersections_with_line_y(contour, y0)
        xs1 = intersections_with_line_y(contour, y1)
        if len(xs0) >= 2 and len(xs1) >= 2:
            cr = float(np.max(xs0) - np.min(xs0))
            ct = float(np.max(xs1) - np.min(xs1))
            cm_i = 0.5 * (cr + ct)
        else:
            cx = float(b.get('cx', origin[0]))
            cm_i = float(cx - origin[0])

        L_i = float(Cl) * qdyn * cm_i * h
        lever = h_prev + 0.5 * h
        Mr_i = L_i * lever
        Mt_i = float(Cm) * qdyn * h * (cm_i ** 2)

        rows.append({'h': h, 'cm': cm_i, 'L_i': L_i, 'Mr_i': Mr_i, 'Mt_i': Mt_i})
        h_prev += h

    df = pd.DataFrame(rows)
    b_span = float(np.max(contour[:-1, 1]) - np.min(contour[:-1, 1]))
    AR = (b_span ** 2) / S if S > 0 else np.nan
    Fd = float(Cd) * qdyn * S

    return {
        'name': name,
        'S': S,
        'b': b_span,
        'AR': AR,
        'c_moy': c_moy,
        'Cl': float(Cl),
        'Cm': float(Cm),
        'Cd': float(Cd),
        'V': float(V),
        'rho': float(rho),
        'L_tot': float(df['L_i'].sum()),
        'Mr_tot': float(df['Mr_i'].sum()),
        'Mt_tot': float(df['Mt_i'].sum()),
        'Fd_tot': Fd,
        'df': df,
    }


def _finite_corr(cl_inf, cd_inf, alpha_inf_deg, AR, e):
    if e <= 0.0:
        raise ValueError('e Oswald doit etre > 0')
    if AR <= 0.0 or np.isnan(AR):
        raise ValueError('AR invalide')

    alpha_inf_rad = np.deg2rad(float(alpha_inf_deg))
    if abs(alpha_inf_rad) < 1e-12:
        cl_fini = float(cl_inf)
    else:
        a_inf = float(cl_inf) / alpha_inf_rad
        a_fini = a_inf / (1.0 + a_inf / (np.pi * AR * e))
        cl_fini = a_fini * alpha_inf_rad

    cd_fini = float(cd_inf) + (cl_fini ** 2) / (np.pi * AR * e)
    alpha_fini_deg = float(alpha_inf_deg) + (float(cl_inf) / (np.pi * AR * e)) * (180.0 / np.pi)

    return {
        'Cl_fini': cl_fini,
        'Cd_fini': cd_fini,
        'alpha_fini_deg': alpha_fini_deg,
    }


def _print_inf(title, rs, rq):
    display(HTML(f"<h3>{title}</h3>"))
    for r in [rs, rq]:
        print(f"[{r['name']}] S={r['S']:.4f} m2, b={r['b']:.4f} m, AR={r['AR']:.3f}, c_moy={r['c_moy']:.4f} m")
        print(f"  Cl={r['Cl']:.4f}, Cm={r['Cm']:.4f}, Cd={r['Cd']:.4f}, V={r['V']:.3f} m/s")
        print(f"  L={r['L_tot']:.3f} N, Mr={r['Mr_tot']:.3f} N.m, Mt={r['Mt_tot']:.3f} N.m, Fd={r['Fd_tot']:.3f} N")
    print('')
    print('[TOTAL]')
    print(f"  L={rs['L_tot'] + rq['L_tot']:.3f} N")
    print(f"  Mr={rs['Mr_tot'] + rq['Mr_tot']:.3f} N.m")
    print(f"  Mt={rs['Mt_tot'] + rq['Mt_tot']:.3f} N.m")
    print(f"  Fd={rs['Fd_tot'] + rq['Fd_tot']:.3f} N")


def _on_plot8(_):
    with out8:
        clear_output()
        try:
            s = _parse_points_text(safran_pts8_widget.value)
            q = _parse_points_text(quille_pts8_widget.value)
            os = _auto_origin(s)
            oq = _auto_origin(q)
            _plot_contours(s, os, q, oq)
            print('Contours affiches.')
        except Exception as e:
            print('Erreur affichage:', e)


def _on_compute8(_):
    with out8:
        clear_output()

        required = ['compute_cmoy_from_contour', 'compute_cmoyen_bands', 'intersections_with_line_y']
        miss = [n for n in required if n not in globals()]
        if miss:
            print('Fonctions manquantes dans le noyau:')
            for m in miss:
                print(' -', m)
            print('Execute les cellules 1 a 7 avant.')
            return

        try:
            rho = float(rho8_widget.value)
            vmin = float(vmin8_widget.value)
            vmax = float(vmax8_widget.value)
            ainf_min = float(alpha_inf_min8_widget.value)
            ainf_max = float(alpha_inf_max8_widget.value)
            e = float(eoswald8_widget.value)

            s = _parse_points_text(safran_pts8_widget.value)
            q = _parse_points_text(quille_pts8_widget.value)
            os = _auto_origin(s)
            oq = _auto_origin(q)

            _plot_contours(s, os, q, oq)

            # Cas MIN (section infinie)
            s_inf_min = compute_hydro_appendage_from_coeffs('safran', s, os, s_cl_inf_min8.value, s_cm_inf_min8.value, s_cd_inf_min8.value, rho, vmin)
            q_inf_min = compute_hydro_appendage_from_coeffs('quille', q, oq, q_cl_inf_min8.value, q_cm_inf_min8.value, q_cd_inf_min8.value, rho, vmin)
            _print_inf('CAS MIN - Section infinie', s_inf_min, q_inf_min)

            corr_s_min = _finite_corr(s_inf_min['Cl'], s_inf_min['Cd'], ainf_min, s_inf_min['AR'], e)
            corr_q_min = _finite_corr(q_inf_min['Cl'], q_inf_min['Cd'], ainf_min, q_inf_min['AR'], e)
            print('Nouveaux angles d\'attaque (MIN):')
            print(f"  Safran: alpha_fini={corr_s_min['alpha_fini_deg']:.3f} deg")
            print(f"  Quille: alpha_fini={corr_q_min['alpha_fini_deg']:.3f} deg")
            print('')

            # Cas MAX (section infinie)
            s_inf_max = compute_hydro_appendage_from_coeffs('safran', s, os, s_cl_inf_max8.value, s_cm_inf_max8.value, s_cd_inf_max8.value, rho, vmax)
            q_inf_max = compute_hydro_appendage_from_coeffs('quille', q, oq, q_cl_inf_max8.value, q_cm_inf_max8.value, q_cd_inf_max8.value, rho, vmax)
            _print_inf('CAS MAX - Section infinie', s_inf_max, q_inf_max)

            corr_s_max = _finite_corr(s_inf_max['Cl'], s_inf_max['Cd'], ainf_max, s_inf_max['AR'], e)
            corr_q_max = _finite_corr(q_inf_max['Cl'], q_inf_max['Cd'], ainf_max, q_inf_max['AR'], e)
            print('Nouveaux angles d\'attaque (MAX):')
            print(f"  Safran: alpha_fini={corr_s_max['alpha_fini_deg']:.3f} deg")
            print(f"  Quille: alpha_fini={corr_q_max['alpha_fini_deg']:.3f} deg")

            # Store for cell 9
            global hydro_step8
            hydro_step8 = {
                'safran_contour': s,
                'quille_contour': q,
                'safran_origin': os,
                'quille_origin': oq,
                's_inf_min': s_inf_min,
                'q_inf_min': q_inf_min,
                's_inf_max': s_inf_max,
                'q_inf_max': q_inf_max,
                'corr_s_min': corr_s_min,
                'corr_q_min': corr_q_min,
                'corr_s_max': corr_s_max,
                'corr_q_max': corr_q_max,
                'rho': rho,
                'vmin': vmin,
                'vmax': vmax,
                'e_oswald': e,
            }

        except Exception as e:
            print('Erreur calcul:', str(e))
            import traceback
            traceback.print_exc()


display(widgets.VBox([
    widgets.HTML('<b>Cellule 8 - Contours et calcul section infinie</b>'),
    widgets.HBox([safran_pts8_widget, quille_pts8_widget]),
    widgets.HTML('<b>Parametres generaux</b>'),
    widgets.HBox([rho8_widget, vmin8_widget, vmax8_widget]),
    widgets.HBox([alpha_inf_min8_widget, alpha_inf_max8_widget, eoswald8_widget]),
    widgets.HTML('<b>Coefficients - Safran</b>'),
    widgets.HBox([s_cl_inf_min8, s_cm_inf_min8, s_cd_inf_min8]),
    widgets.HBox([s_cl_inf_max8, s_cm_inf_max8, s_cd_inf_max8]),
    widgets.HTML('<b>Coefficients - Quille</b>'),
    widgets.HBox([q_cl_inf_min8, q_cm_inf_min8, q_cd_inf_min8]),
    widgets.HBox([q_cl_inf_max8, q_cm_inf_max8, q_cd_inf_max8]),
    widgets.HBox([plot8_btn, compute8_btn]),
    out8,
]))

print('Cellule 8 prete: afficher les contours puis calculer la section infinie et le nouvel angle d\'attaque.')
plot8_btn.on_click(_on_plot8)
compute8_btn.on_click(_on_compute8)

Cellule 8 prete: afficher les contours puis calculer la section infinie et le nouvel angle d'attaque.


In [7]:
# === CELLULE 7 : POLAIRE V_B(AWA) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â PROFIL INFINI + MODE HYBRIDE (VOILE vs SPY) ===
# Calcul: Pour chaque AWA (0-360Ãƒâ€šÃ‚Â°), trouve V_B tel que F_prop = R_hydro
# MODE HYBRIDE: Compare VOILE et SPY ÃƒÆ’Ã‚Â  chaque AWA, utilise le max(V_B)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  Polaire V_B(AWA) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Profil infini + MODE HYBRIDE (VOILE/SPY)\n")

# ===== Fonctions pour polaire =====

def calc_aero_simple(V_AW, beta_AW_deg, alpha_trim_deg, polar_tables, A_sail, c_moy, 
                     TWA_deg, rho_air=1.225, mu_air=1.81e-5):
    """Calcule Cl, Cd par interpolation simple + forces aÃƒÆ’Ã‚Â©ro (profil INFINI)."""
    V_AW = float(V_AW)
    beta_AW_deg = float(beta_AW_deg)
    alpha_trim_deg = float(alpha_trim_deg)
    TWA_deg = float(TWA_deg)
    
    alpha_eff_deg = alpha_trim_deg
    Re = (rho_air * V_AW * c_moy) / mu_air
    
    try:
        coeffs = _interp_coeffs_for_re_alpha(polar_tables, Re, alpha_eff_deg)
        Cl = float(coeffs.get('Cl', 0.0))
        Cd = float(coeffs.get('Cd', 0.0))
    except Exception as e:
        Cl, Cd = 0.0, 0.0
    
    q_dyn = 0.5 * rho_air * V_AW**2
    L_aero = Cl * q_dyn * A_sail
    D_aero = Cd * q_dyn * A_sail
    
    angle1 = alpha_trim_deg + beta_AW_deg + 90.0 - TWA_deg
    angle2 = TWA_deg - alpha_trim_deg - beta_AW_deg
    F_prop = L_aero * np.cos(np.radians(angle1)) - D_aero * np.cos(np.radians(angle2))
    
    return {'F_prop': float(F_prop), 'Cl': float(Cl), 'Cd': float(Cd)}


def calc_apparent_wind(V_TW, V_B, TWA_deg):
    """Calcule vent apparent vectoriel."""
    V_TW = float(V_TW)
    V_B = float(V_B)
    TWA_deg = float(TWA_deg)
    
    V_TW_x = 0.0
    V_TW_y = V_TW
    
    angle_rot_rad = np.radians(90.0 - TWA_deg)
    V_B_x = V_B * np.cos(angle_rot_rad)
    V_B_y = V_B * np.sin(angle_rot_rad)
    
    V_AW_x = V_TW_x + V_B_x
    V_AW_y = V_TW_y + V_B_y
    V_AW = np.sqrt(V_AW_x**2 + V_AW_y**2)
    
    if abs(V_AW_y) < 1e-9:
        beta_AW_rad = 0.0
    else:
        beta_AW_rad = np.arctan(V_AW_x / V_AW_y)
    
    beta_AW_deg = np.degrees(beta_AW_rad)
    
    return {'V_AW': float(V_AW), 'beta_AW': float(beta_AW_deg)}


def calc_hydro_drag(V_B, S_wetted, CR_friction, Rcoque_coeff, rho_water):
    """Calcule rÃƒÆ’Ã‚Â©sistance hydrodynamique."""
    V_B = float(V_B)
    q_dyn_water = 0.5 * rho_water * V_B**2
    C_R_total = float(CR_friction) + float(Rcoque_coeff)
    R_hydro = C_R_total * q_dyn_water * S_wetted
    return float(R_hydro)


def solve_equilibrium_7(V_TW, AWA_deg, polar_tables, alpha_trim_deg, A_sail, c_moy,
                        rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                        max_iterations=50, tolerance=0.5, V_B_max_factor=2.0):
    """RÃƒÆ’Ã‚Â©sout F_prop (VOILE) = R_hydro par dichotomie."""
    V_TW = float(V_TW)
    AWA_deg = float(AWA_deg)
    
    V_B_min = max(0.1, float(V_TW) * 0.05)
    V_B_max = float(V_TW) * float(V_B_max_factor)
    
    converged = False
    
    for it in range(int(max_iterations)):
        V_B_mid = 0.5 * (V_B_min + V_B_max)
        
        aw_info = calc_apparent_wind(V_TW, V_B_mid, AWA_deg)
        aero_info = calc_aero_simple(aw_info['V_AW'], aw_info['beta_AW'],
                                     alpha_trim_deg, polar_tables, A_sail, c_moy,
                                     AWA_deg, rho_air=rho_air)
        R_hydro = calc_hydro_drag(V_B_mid, S_wetted, CR_friction, Rcoque_coeff, rho_water)
        
        F_prop = aero_info['F_prop']
        error = F_prop - R_hydro
        
        if abs(error) < tolerance:
            converged = True
            return {'V_B': float(V_B_mid), 'converged': True, 'aw': aw_info, 'mode': 'voile'}
        
        if error > 0:
            V_B_min = V_B_mid
        else:
            V_B_max = V_B_mid
    
    V_B_solution = 0.5 * (V_B_min + V_B_max)
    aw_final = calc_apparent_wind(V_TW, V_B_solution, AWA_deg)
    return {'V_B': float(V_B_solution), 'converged': converged, 'aw': aw_final, 'mode': 'voile'}


def calc_spy_equilibrium(V_TW, AWA_deg, rho_air, rho_water, S_wetted, CR_friction, 
                         Rcoque_coeff, Cx_spy=1.0, S_spy=1.0):
    """RÃƒÆ’Ã‚Â©sout F_spy = R_hydro pour mode SPY (arriÃƒÆ’Ã‚Â¨re uniquement)."""
    V_TW = float(V_TW)
    AWA_deg = float(AWA_deg)
    Cx_spy = float(Cx_spy)
    S_spy = float(S_spy)
    
    # SPY actif seulement entre 90Ãƒâ€šÃ‚Â° et 270Ãƒâ€šÃ‚Â°
    if AWA_deg < 90.0 or AWA_deg > 270.0:
        return None
    
    sin_90_CAP = np.sin(np.radians(90.0 - AWA_deg))
    abs_sin_90_CAP = abs(sin_90_CAP)
    
    if abs_sin_90_CAP < 0.01:
        return None
    
    V_B_min = 0.1
    V_B_max = V_TW * 2.0
    
    for it in range(50):
        V_B_mid = 0.5 * (V_B_min + V_B_max)
        F_spy = Cx_spy * 0.5 * rho_air * V_B_mid**2 * S_spy
        F_prop_spy = F_spy * abs_sin_90_CAP
        R_hydro = calc_hydro_drag(V_B_mid, S_wetted, CR_friction, Rcoque_coeff, rho_water)
        error = F_prop_spy - R_hydro
        
        if abs(error) < 0.5:
            return {'V_B': float(V_B_mid), 'mode': 'spy'}
        
        if error > 0:
            V_B_min = V_B_mid
        else:
            V_B_max = V_B_mid
    
    V_B_solution = 0.5 * (V_B_min + V_B_max)
    return {'V_B': float(V_B_solution), 'mode': 'spy'}


def solve_equilibrium_hybrid(V_TW, AWA_deg, polar_tables, alpha_trim_deg, A_sail, c_moy,
                            rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                            Cx_spy=1.0, S_spy=1.0, max_iterations=50, tolerance=0.5, V_B_max_factor=2.0):
    """RÃƒÆ’Ã‚Â©sout ÃƒÆ’Ã‚Â©quilibre HYBRIDE: compare VOILE et SPY, retourne max(V_B)."""
    eq_voile = solve_equilibrium_7(V_TW, AWA_deg, polar_tables, alpha_trim_deg, A_sail, c_moy,
                                  rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                                  max_iterations, tolerance, V_B_max_factor)
    V_B_voile = eq_voile['V_B'] if eq_voile else 0.0
    
    eq_spy = calc_spy_equilibrium(V_TW, AWA_deg, rho_air, rho_water, S_wetted, CR_friction, 
                                 Rcoque_coeff, Cx_spy=Cx_spy, S_spy=S_spy)
    V_B_spy = eq_spy['V_B'] if eq_spy else 0.0
    
    if V_B_spy > V_B_voile:
        eq_final = eq_spy
        eq_final['V_B'] = V_B_spy
        aw_final = calc_apparent_wind(V_TW, V_B_spy, AWA_deg) if V_B_spy > 0 else {'V_AW': 0, 'beta_AW': 0}
        eq_final['aw'] = aw_final
    else:
        eq_final = eq_voile
        eq_final['V_B'] = V_B_voile
    
    eq_final['V_B_voile'] = V_B_voile
    eq_final['V_B_spy'] = V_B_spy
    
    return eq_final


def compute_polaire_7(V_TW, profile_file, alpha_trim_deg, A_sail, c_moy,
                      S_wetted, CR_friction, Rcoque_coeff,
                      rho_air=1.225, rho_water=1025.0, n_twa=180,
                      Cx_spy=1.0, S_spy=1.0, use_hybrid=True):
    """Calcule polaire V_B vs TWA avec mode hybride."""
    try:
        polar_tables = _load_polar_file(profile_file)
    except Exception as e:
        print(f"ÃƒÂ¢Ã‚ÂÃ…â€™ Erreur chargement polaire: {e}")
        return None
    
    n_calc = (n_twa // 2) + 1
    TWAs_calc = np.linspace(0, 180, n_calc)
    rows = []
    
    for twa in TWAs_calc:
        try:
            if use_hybrid:
                eq = solve_equilibrium_hybrid(V_TW, twa, polar_tables, alpha_trim_deg, A_sail, c_moy,
                                             rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                                             Cx_spy=Cx_spy, S_spy=S_spy)
            else:
                eq = solve_equilibrium_7(V_TW, twa, polar_tables, alpha_trim_deg, A_sail, c_moy,
                                        rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff)
            
            V_B = eq['V_B']
            aw = eq.get('aw', {'V_AW': 0, 'beta_AW': 0})
            vmg = V_B * abs(np.cos(np.radians(twa)))
            
            row_dict = {
                'TWA': float(twa),
                'V_B': float(V_B),
                'V_B_voile': float(eq.get('V_B_voile', 0.0)),
                'V_B_spy': float(eq.get('V_B_spy', 0.0)),
                'mode': eq.get('mode', 'voile'),
                'V_AW': float(aw.get('V_AW', 0.0)),
                'beta_AW': float(aw.get('beta_AW', 0.0)),
                'VMG': float(vmg),
                'converged': eq.get('converged', True),
            }
            rows.append(row_dict)
        except Exception as ex:
            print(f"  ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â  TWA={twa}Ãƒâ€šÃ‚Â°: {ex}")
            continue
    
    if not rows:
        return None
    
    df_first = pd.DataFrame(rows)
    
    # SymÃƒÆ’Ã‚Â©trie pour 180-360Ãƒâ€šÃ‚Â°
    rows_sym = []
    for _, row in df_first.iterrows():
        twa_orig = float(row['TWA'])
        if 0 < twa_orig < 180:
            row_sym = row.copy()
            row_sym['TWA'] = 360.0 - twa_orig
            row_sym['VMG'] = float(row['V_B']) * abs(np.cos(np.radians(360.0 - twa_orig)))
            rows_sym.append(row_sym)
    
    if rows_sym:
        df_sym = pd.DataFrame(rows_sym)
        df_polaire = pd.concat([df_first, df_sym], ignore_index=True)
        df_polaire = df_polaire.sort_values('TWA').reset_index(drop=True)
    else:
        df_polaire = df_first
    
    return df_polaire


# ===== Widgets =====

profile_widget7 = widgets.Dropdown(
    options=profiles_list if 'profiles_list' in globals() else ['NACA0012'],
    value=(profiles_list[0] if 'profiles_list' in globals() and len(profiles_list) > 0 else 'NACA0012'),
    description='Profil:',
    style={'description_width': '80px'}
)

V_TW_widget7 = widgets.FloatText(
    value=10.0,
    description='V_TW (m/s):',
    step=0.5,
    min=1.0,
    max=30.0,
    style={'description_width': '80px'}
)

alpha_trim_widget7 = widgets.FloatText(
    value=0.0,
    description='ÃƒÅ½Ã‚Â±_trim (deg):',
    step=0.5,
    min=-15.0,
    max=20.0,
    style={'description_width': '80px'}
)

A_sail_widget7 = widgets.FloatText(
    value=A_sail_default if 'A_sail_default' in globals() else 1.0,
    description='A_sail (mÃƒâ€šÃ‚Â²):',
    step=0.1,
    min=0.1,
    max=10.0,
    style={'description_width': '80px'}
)

c_moy_widget7 = widgets.FloatText(
    value=c_moy_default if 'c_moy_default' in globals() else 0.5,
    description='cÃƒÅ’Ã¢â‚¬Å¾ (m):',
    step=0.05,
    min=0.1,
    max=2.0,
    style={'description_width': '80px'}
)

S_wetted_widget7 = widgets.FloatText(
    value=S_wetted_default if 'S_wetted_default' in globals() else 1.5,
    description='S mouillÃƒÆ’Ã‚Â©e (mÃƒâ€šÃ‚Â²):',
    step=0.1,
    min=0.1,
    max=10.0,
    style={'description_width': '80px'}
)

CR_friction_widget7 = widgets.FloatText(
    value=0.003,
    description='CR friction:',
    step=0.0001,
    min=0.001,
    max=0.01,
    style={'description_width': '80px'}
)

Rcoque_coeff_widget7 = widgets.FloatText(
    value=0.002,
    description='Rcoque coeff:',
    step=0.0001,
    min=0.0,
    max=0.05,
    style={'description_width': '80px'}
)

rho_air_widget7 = widgets.FloatText(
    value=1.225,
    description='ÃƒÂÃ‚Â air (kg/mÃƒâ€šÃ‚Â³):',
    step=0.01,
    min=1.0,
    max=1.3,
    style={'description_width': '80px'}
)

rho_water_widget7 = widgets.FloatText(
    value=1025.0,
    description='ÃƒÂÃ‚Â eau (kg/mÃƒâ€šÃ‚Â³):',
    step=1.0,
    min=1020.0,
    max=1030.0,
    style={'description_width': '80px'}
)

n_twa_widget7 = widgets.IntSlider(
    value=180,
    min=36,
    max=360,
    step=36,
    description='N points TWA:',
    style={'description_width': '80px'},
    continuous_update=False
)

Cx_spy_widget7 = widgets.FloatText(
    value=1.0,
    description='Cx (SPY):',
    step=0.1,
    min=0.1,
    max=2.0,
    style={'description_width': '80px'}
)

S_spy_widget7 = widgets.FloatText(
    value=1.0,
    description='S (SPY mÃƒâ€šÃ‚Â²):',
    step=0.1,
    min=0.1,
    max=10.0,
    style={'description_width': '80px'}
)

polaire_btn7 = widgets.Button(
    description='Tracer polaire V_B(TWA)',
    button_style='success',
    tooltip='Calculate and plot boat speed polaire',
    icon='calculator'
)

out7 = widgets.Output()


def _on_polaire7(_):
    """Callback: calcul et tracÃƒÆ’Ã‚Â© polaire."""
    with out7:
        clear_output()
        try:
            # Fonction de sÃƒÆ’Ã‚Â©curitÃƒÆ’Ã‚Â© pour extraire les valeurs des widgets
            def safe_get_float(widget, default=0.0):
                """RÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â¨re float widget ou retourne default si None."""
                try:
                    val = widget.value
                    if val is None:
                        return default
                    return float(val)
                except (TypeError, ValueError, AttributeError):
                    return default
            
            def safe_get_int(widget, default=0):
                """RÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â¨re int widget ou retourne default si None."""
                try:
                    val = widget.value
                    if val is None:
                        return default
                    return int(val)
                except (TypeError, ValueError, AttributeError):
                    return default
            
            def safe_get_str(widget, default=''):
                """RÃƒÆ’Ã‚Â©cupÃƒÆ’Ã‚Â¨re string widget ou retourne default si None."""
                try:
                    val = widget.value
                    if val is None:
                        return default
                    return str(val).strip()
                except (TypeError, AttributeError):
                    return default
            
            # Extraction sÃƒÆ’Ã‚Â©curisÃƒÆ’Ã‚Â©e de toutes les valeurs
            V_TW = safe_get_float(V_TW_widget7, 10.0)
            alpha_trim = safe_get_float(alpha_trim_widget7, 0.0)
            profile = safe_get_str(profile_widget7, 'NACA0012').upper()
            A_sail = safe_get_float(A_sail_widget7, 1.0)
            c_moy = safe_get_float(c_moy_widget7, 0.5)
            S_wetted = safe_get_float(S_wetted_widget7, 1.0)
            CR_friction = safe_get_float(CR_friction_widget7, 0.001)
            Rcoque_coeff = safe_get_float(Rcoque_coeff_widget7, 0.05)
            rho_air = safe_get_float(rho_air_widget7, 1.225)
            rho_water = safe_get_float(rho_water_widget7, 1025.0)
            n_twa = safe_get_int(n_twa_widget7, 180)
            Cx_spy = safe_get_float(Cx_spy_widget7, 1.0)
            S_spy = safe_get_float(S_spy_widget7, 1.0)
            
            # Validation des paramÃƒÆ’Ã‚Â¨tres critiques
            if V_TW <= 0.0 or n_twa <= 0:
                print("ÃƒÂ¢Ã‚ÂÃ…â€™ Erreur: V_TW et n_twa doivent ÃƒÆ’Ã‚Âªtre > 0")
                return
            
            if n_twa % 36 != 0:
                n_twa = max(36, min(360, ((n_twa + 18) // 36) * 36))
                print(f"ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â n_twa ajustÃƒÆ’Ã‚Â© ÃƒÆ’Ã‚Â  {n_twa}")
            
            profile_file = _resolve_profile_file(profile, folder='.')
            
            print(f"ÃƒÂ¢Ã‚ÂÃ‚Â³ Calcul polaire V_B(TWA) - MODE HYBRIDE (VOILE/SPY)...")
            print(f"   Profil: {profile}, ÃƒÅ½Ã‚Â±_trim={alpha_trim}Ãƒâ€šÃ‚Â°, V_TW={V_TW:.1f} m/s")
            print(f"   N points: {n_twa}, SPY: Cx={Cx_spy}, S={S_spy:.2f} mÃƒâ€šÃ‚Â²\n")
            
            df_polaire = compute_polaire_7(
                V_TW=V_TW,
                profile_file=str(profile_file),
                alpha_trim_deg=alpha_trim,
                A_sail=A_sail,
                c_moy=c_moy,
                S_wetted=S_wetted,
                CR_friction=CR_friction,
                Rcoque_coeff=Rcoque_coeff,
                rho_air=rho_air,
                rho_water=rho_water,
                n_twa=n_twa,
                Cx_spy=Cx_spy,
                S_spy=S_spy,
                use_hybrid=True
            )
            
            if df_polaire is None or df_polaire.shape[0] == 0:
                print("ÃƒÂ¢Ã‚ÂÃ…â€™ Polaire vide!")
                return
            
            # Tracer 4 panneaux
            fig = plt.figure(figsize=(18, 6))
            
            # Panel 1: Rose polaire
            ax1 = plt.subplot(141, projection='polar')
            TWAs_rad = np.radians(df_polaire['TWA'].values)
            V_Bs = df_polaire['V_B'].values
            ax1.plot(TWAs_rad, V_Bs, 'b-', linewidth=2.5)
            ax1.fill(TWAs_rad, V_Bs, alpha=0.2, color='blue')
            ax1.set_theta_zero_location('N')
            ax1.set_theta_direction(-1)
            ax1.set_title(f'Rose V_B (ÃƒÅ½Ã‚Â±_trim={alpha_trim}Ãƒâ€šÃ‚Â°)', fontsize=11, weight='bold')
            ax1.grid(True, alpha=0.3)
            
            # Panel 2: V_B vs TWA (tous)
            ax2 = plt.subplot(142)
            ax2.plot(df_polaire['TWA'], df_polaire['V_B_voile'], 'b--', linewidth=1.5, alpha=0.6, label='VOILE')
            ax2.plot(df_polaire['TWA'], df_polaire['V_B_spy'], 'r--', linewidth=1.5, alpha=0.6, label='SPY')
            ax2.plot(df_polaire['TWA'], df_polaire['V_B'], 'g-', linewidth=2.5, marker='o', markersize=3, label='MAX (choisi)')
            idx_max = df_polaire['V_B'].idxmax()
            V_B_max = df_polaire.loc[idx_max, 'V_B']
            TWA_max = df_polaire.loc[idx_max, 'TWA']
            ax2.plot(TWA_max, V_B_max, 'k*', markersize=20, label=f'Max: {V_B_max:.2f} m/s @ {TWA_max:.0f}Ãƒâ€šÃ‚Â°')
            ax2.set_xlabel('TWA (deg)', fontsize=10, weight='bold')
            ax2.set_ylabel('V_B (m/s)', fontsize=10, weight='bold')
            ax2.set_title('Vitesse bateau (hybride)', fontsize=11, weight='bold')
            ax2.grid(True, alpha=0.3)
            ax2.set_xlim(0, 360)
            ax2.legend(fontsize=8, loc='best')
            
            # Panel 3: DiffÃƒÆ’Ã‚Â©rence VOILE vs SPY
            ax3 = plt.subplot(143)
            df_polaire['diff'] = df_polaire['V_B_spy'] - df_polaire['V_B_voile']
            colors = ['red' if x > 0 else 'blue' for x in df_polaire['diff']]
            ax3.bar(df_polaire['TWA'], df_polaire['diff'], color=colors, alpha=0.6, width=2)
            ax3.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
            ax3.set_xlabel('TWA (deg)', fontsize=10, weight='bold')
            ax3.set_ylabel('ÃƒÅ½Ã¢â‚¬ÂV_B (SPY - VOILE) m/s', fontsize=10, weight='bold')
            ax3.set_title('SPY meilleur? (rouge=oui)', fontsize=11, weight='bold')
            ax3.grid(True, alpha=0.3, axis='y')
            ax3.set_xlim(0, 360)
            
            # Panel 4: VMG vs TWA
            ax4 = plt.subplot(144)
            ax4.plot(df_polaire['TWA'], df_polaire['VMG'], 'g-', linewidth=2.5, marker='s', markersize=3, alpha=0.7)
            idx_vmg = df_polaire['VMG'].idxmax()
            VMG_max = df_polaire.loc[idx_vmg, 'VMG']
            TWA_vmg = df_polaire.loc[idx_vmg, 'TWA']
            ax4.plot(TWA_vmg, VMG_max, 'r*', markersize=20, label=f'Max: {VMG_max:.2f} m/s @ {TWA_vmg:.0f}Ãƒâ€šÃ‚Â°')
            ax4.set_xlabel('TWA (deg)', fontsize=10, weight='bold')
            ax4.set_ylabel('VMG (m/s)', fontsize=10, weight='bold')
            ax4.set_title('Vitesse utile', fontsize=11, weight='bold')
            ax4.grid(True, alpha=0.3)
            ax4.set_xlim(0, 360)
            ax4.legend(fontsize=10, loc='best')
            
            plt.tight_layout()
            display(fig)
            plt.close(fig)
            
            # Stats
            n_spy = (df_polaire['mode'] == 'spy').sum()
            print(f"\nÃƒÂ¢Ã…â€œÃ¢â‚¬Â¦ Polaire calculÃƒÆ’Ã‚Â©e:")
            print(f"   ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢ {df_polaire.shape[0]} points")
            print(f"   ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢ Mode VOILE: {len(df_polaire) - n_spy} points")
            print(f"   ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢ Mode SPY: {n_spy} points ({100*n_spy/len(df_polaire):.1f}%)")
            print(f"   ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢ V_B range: {df_polaire['V_B'].min():.2f} ÃƒÆ’Ã‚Â  {df_polaire['V_B'].max():.2f} m/s")
            print(f"   ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢ VMG max: {VMG_max:.2f} m/s @ TWA={TWA_vmg:.0f}Ãƒâ€šÃ‚Â°")
            
            globals()['polaire_results_7'] = df_polaire
            
        except Exception as e:
            print(f"ÃƒÂ¢Ã‚ÂÃ…â€™ Erreur: {e}")
            import traceback
            traceback.print_exc()


polaire_btn7.on_click(_on_polaire7)

# Afficher interface
display(widgets.VBox([
    widgets.HTML('<h3>ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  POLAIRE V_B(TWA) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â MODE HYBRIDE (VOILE/SPY)</h3>'),
    widgets.HTML('<b>Profil INFINI simple + comparaison SPY</b>'),
    widgets.HTML(''),
    widgets.HBox([profile_widget7, V_TW_widget7, alpha_trim_widget7]),
    widgets.HBox([A_sail_widget7, c_moy_widget7, S_wetted_widget7]),
    widgets.HBox([CR_friction_widget7, Rcoque_coeff_widget7]),
    widgets.HBox([rho_air_widget7, rho_water_widget7, n_twa_widget7]),
    widgets.HBox([Cx_spy_widget7, S_spy_widget7]),
    widgets.HBox([polaire_btn7]),
    out7,
]))

print("ÃƒÂ¢Ã…â€œÃ¢â‚¬Å“ Cellule 7 prÃƒÆ’Ã‚Âªte! (polaire V_B avec mode VOILE/SPY hybride)")

ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã…Â  Polaire V_B(AWA) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Profil infini + MODE HYBRIDE (VOILE/SPY)



ÃƒÂ¢Ã…â€œÃ¢â‚¬Å“ Cellule 7 prÃƒÆ’Ã‚Âªte! (polaire V_B avec mode VOILE/SPY hybride)


In [8]:
# === CELLULE 8 : POLAIRE V_B(CAP) - VENT REEL TW + DERIVE + GITE ===
# Calcul: pour chaque CAP (0-360 deg), trouve V_B tel que F_prop(CAP) = R_hydro(V_B).
# Ajout derive: Faero_lateral(CAP) = Fhydro_derive(psi, V_B), profil derive NACA0012.
# Convention: CAP est l'angle de la direction du bateau par rapport au vent reel TW.
# Le vent apparent n'est pas utilise dans cette cellule.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import struct
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("Polaire V_B(CAP) - Vent reel TW + derive + gite\n")

# ===== Fonctions pour polaire =====

def normalize_cap_for_half_polar(CAP_deg):
    """Ramene un CAP 0..360 deg sur le demi-plan symetrique 0..180 deg."""
    cap = float(CAP_deg) % 360.0
    if cap > 180.0:
        cap = 360.0 - cap
    return cap


def project_real_wind_forces_on_boat_axis(F_portance, F_trainee, CAP_deg):
    """Projette portance et trainee dans l'axe du bateau selon le CAP."""
    cap = normalize_cap_for_half_polar(CAP_deg)
    F_portance = float(F_portance)
    F_trainee = float(F_trainee)

    if np.isclose(cap, 90.0, atol=1e-9):
        F_prop = F_portance
        case = 'CAP = 90 deg'
    elif cap < 90.0:
        F_prop = (
            F_portance * np.cos(np.radians(90.0 - cap))
            - F_trainee * np.cos(np.radians(cap))
        )
        case = '0 deg <= CAP < 90 deg'
    else:
        F_prop = (
            F_portance * np.cos(np.radians(cap - 90.0))
            + F_trainee * np.cos(np.radians(180.0 - cap))
        )
        case = '90 deg < CAP <= 180 deg'

    return float(F_prop), case


def project_real_wind_side_force(F_portance, F_trainee, CAP_deg):
    """Force aero laterale a equilibrer par la portance de derive."""
    cap = normalize_cap_for_half_polar(CAP_deg)
    F_portance = float(F_portance)
    F_trainee = float(F_trainee)

    if np.isclose(cap, 90.0, atol=1e-9):
        F_aero = F_trainee
        case = 'CAP = 90 deg'
    elif cap < 90.0:
        F_aero = (
            F_portance * np.cos(np.radians(cap))
            + F_trainee * np.cos(np.radians(90.0 - cap))
        )
        case = '0 deg <= CAP < 90 deg'
    else:
        # Signe negatif: meme magnitude que la projection, sens lateral oppose.
        F_aero_magnitude = (
            F_portance * np.cos(np.radians(180.0 - cap))
            + F_trainee * np.cos(np.radians(cap - 90.0))
        )
        F_aero = -F_aero_magnitude
        case = '90 deg < CAP <= 180 deg'

    return float(F_aero), case


def calc_aero_simple(V_TW, CAP_deg, alpha_trim_deg, polar_tables, A_sail, c_moy,
                     rho_air=1.225, mu_air=1.81e-5):
    """Calcule Cl, Cd et forces aero avec le vent reel TW uniquement."""
    V_TW = float(V_TW)
    CAP_deg = float(CAP_deg)
    alpha_trim_deg = float(alpha_trim_deg)

    alpha_eff_deg = alpha_trim_deg
    Re = (rho_air * V_TW * c_moy) / mu_air

    try:
        coeffs = _interp_coeffs_for_re_alpha(polar_tables, Re, alpha_eff_deg)
        Cl = float(coeffs.get('Cl', 0.0))
        Cd = float(coeffs.get('Cd', 0.0))
    except Exception:
        Cl, Cd = 0.0, 0.0

    q_dyn = 0.5 * rho_air * V_TW**2
    L_aero = Cl * q_dyn * A_sail
    D_aero = Cd * q_dyn * A_sail
    F_prop, projection_case = project_real_wind_forces_on_boat_axis(L_aero, D_aero, CAP_deg)
    F_aero_lat, lateral_case = project_real_wind_side_force(L_aero, D_aero, CAP_deg)

    return {
        'F_prop': float(F_prop),
        'F_aero_lat': float(F_aero_lat),
        'F_aero_lat_abs': float(abs(F_aero_lat)),
        'L_aero': float(L_aero),
        'D_aero': float(D_aero),
        'Cl': float(Cl),
        'Cd': float(Cd),
        'Re': float(Re),
        'projection_case': projection_case,
        'lateral_case': lateral_case,
    }


def calc_hydro_drag(V_B, S_wetted, CR_friction, Rcoque_coeff, rho_water):
    """Calcule resistance hydrodynamique longitudinale."""
    V_B = float(V_B)
    q_dyn_water = 0.5 * rho_water * V_B**2
    C_R_total = float(CR_friction) + float(Rcoque_coeff)
    R_hydro = C_R_total * q_dyn_water * S_wetted
    return float(R_hydro)


def _zero_appendage_forces(name):
    """Retourne un paquet d'efforts nul pour un appendice hydrodynamique."""
    out = {
        'name': name,
        'alpha_deg': 0.0,
        'V_rel': 0.0,
        'V_lateral': 0.0,
        'Re': 0.0,
        'Cl': 0.0,
        'Cd': 0.0,
        'L': 0.0,
        'D': 0.0,
        'F_lat': 0.0,
        'R_long': 0.0,
    }
    out.update({
        f'alpha_{name}_deg': 0.0,
        f'V_{name}': 0.0,
        f'Re_{name}': 0.0,
        f'Cl_{name}': 0.0,
        f'Cd_{name}': 0.0,
        f'L_{name}': 0.0,
        f'D_{name}': 0.0,
        f'F_hydro_{name}': 0.0,
        f'R_long_{name}': 0.0,
    })
    return out


def calc_appendage_hydro_forces(name, V_B, psi_deg, appendage_tables, S_appendage, c_appendage,
                                  rho_water=1025.0, mu_water=1.05e-3):
    """Calcule portance, trainee, force laterale et trainee longitudinale d'un appendice.

    Le safran fixe et la quille utilisent le meme ecoulement relatif: V_rel = V_B / cos(psi),
    et le meme angle d'attaque alpha = psi.
    """
    V_B = float(V_B)
    psi_abs_deg = abs(float(psi_deg))
    S_appendage = float(S_appendage)
    c_appendage = float(c_appendage)
    rho_water = float(rho_water)
    mu_water = float(mu_water)

    if V_B <= 0.0 or S_appendage <= 0.0 or c_appendage <= 0.0:
        return _zero_appendage_forces(name)

    psi_rad = np.radians(psi_abs_deg)
    cos_psi = max(float(np.cos(psi_rad)), 1e-9)
    sin_psi = float(np.sin(psi_rad))
    V_lateral = V_B * np.tan(psi_rad)
    V_rel = max(float(np.sqrt(V_B**2 + V_lateral**2)), float(V_B / cos_psi))
    Re_appendage = (rho_water * V_rel * c_appendage) / mu_water if mu_water != 0.0 else 0.0

    try:
        coeffs = _interp_coeffs_for_re_alpha(appendage_tables, Re_appendage, psi_abs_deg)
        Cl_appendage = float(coeffs.get('Cl', 0.0))
        Cd_appendage = float(coeffs.get('Cd', 0.0))
    except Exception:
        Cl_appendage, Cd_appendage = 0.0, 0.0

    q_dyn_water = 0.5 * rho_water * V_rel**2
    L_appendage = Cl_appendage * q_dyn_water * S_appendage
    D_appendage = Cd_appendage * q_dyn_water * S_appendage

    F_lat = D_appendage * sin_psi + L_appendage * cos_psi
    R_long = D_appendage * cos_psi - L_appendage * sin_psi

    out = {
        'name': name,
        'alpha_deg': float(psi_abs_deg),
        'V_rel': float(V_rel),
        'V_lateral': float(V_lateral),
        'Re': float(Re_appendage),
        'Cl': float(Cl_appendage),
        'Cd': float(Cd_appendage),
        'L': float(L_appendage),
        'D': float(D_appendage),
        'F_lat': float(F_lat),
        'R_long': float(R_long),
    }
    out.update({
        f'alpha_{name}_deg': float(psi_abs_deg),
        f'V_{name}': float(V_rel),
        f'Re_{name}': float(Re_appendage),
        f'Cl_{name}': float(Cl_appendage),
        f'Cd_{name}': float(Cd_appendage),
        f'L_{name}': float(L_appendage),
        f'D_{name}': float(D_appendage),
        f'F_hydro_{name}': float(F_lat),
        f'R_long_{name}': float(R_long),
    })
    return out


def _sum_appendage_forces(results):
    return {
        'F_lat_total': float(sum(r.get('F_lat', 0.0) for r in results.values())),
        'R_long_total': float(sum(r.get('R_long', 0.0) for r in results.values())),
        'L_total': float(sum(r.get('L', 0.0) for r in results.values())),
        'D_total': float(sum(r.get('D', 0.0) for r in results.values())),
    }


def calc_appendages_for_psi(V_B, psi_deg, appendage_tables, appendages,
                            rho_water=1025.0, mu_water=1.05e-3):
    results = {}
    for app in appendages:
        name = app['name']
        tables = app.get('tables', appendage_tables)
        results[name] = calc_appendage_hydro_forces(
            name=name,
            V_B=V_B,
            psi_deg=psi_deg,
            appendage_tables=tables,
            S_appendage=app['S'],
            c_appendage=app['c'],
            rho_water=rho_water,
            mu_water=mu_water,
        )
    totals = _sum_appendage_forces(results)
    return results, totals


def solve_appendage_lateral_equilibrium(F_aero_lat, V_B, appendage_tables, appendages,
                                        rho_water=1025.0, mu_water=1.05e-3,
                                        psi_max_deg=20.0, tolerance=0.5, max_iterations=60):
    """Resout |Faero_lat| = F_lat_quille + F_lat_safran pour une vitesse V_B donnee."""
    F_aero_lat = float(F_aero_lat)
    target = abs(F_aero_lat)
    sign = 1.0 if F_aero_lat >= 0.0 else -1.0

    def pack(psi_abs, converged, reason):
        results, totals = calc_appendages_for_psi(V_B, psi_abs, appendage_tables, appendages,
                                                  rho_water=rho_water, mu_water=mu_water)
        return {
            'psi_deg': float(sign * psi_abs),
            'converged': bool(converged),
            'reason': reason,
            'appendages': results,
            **totals,
        }

    if V_B <= 0.0:
        return pack(0.0, target <= tolerance, 'vitesse nulle')

    if target <= tolerance:
        return pack(0.0, True, 'target nul')

    psi_min = 0.0
    psi_max = float(psi_max_deg)
    max_pack = pack(psi_max, False, 'force laterale trop grande pour psi_max')
    if max_pack['F_lat_total'] < target:
        return max_pack

    psi_mid = 0.0
    mid_pack = None
    converged = False
    for _ in range(int(max_iterations)):
        psi_mid = 0.5 * (psi_min + psi_max)
        mid_pack = pack(psi_mid, False, 'max_iterations')
        error = mid_pack['F_lat_total'] - target
        if abs(error) < tolerance:
            converged = True
            break
        if error < 0.0:
            psi_min = psi_mid
        else:
            psi_max = psi_mid

    if mid_pack is None:
        mid_pack = pack(psi_mid, False, 'max_iterations')
    mid_pack['converged'] = converged
    mid_pack['reason'] = 'ok' if converged else 'max_iterations'
    return mid_pack
def compute_aero_application_height_from_wing(N=120):
    """Hauteur du centre d'effort aero depuis le contour de voile local a la cellule 8."""
    try:
        return float(max(cell8_get_sail_geometry()["h_aero_m"], 1e-6))
    except Exception:
        return 1.0


# ===== Hydrostatique STL pour moment redressant de coque =====

CELL8_STL_FOLDER = Path("test_coque_stl")
CELL8_DEFAULT_STL_PATH = Path("test.coque.stl")
CELL8_STL_UNIT_SCALE = {"m": 1.0, "cm": 0.01, "mm": 0.001}
CELL8_RHO_CARBON_DEFAULT = 1500.0
CELL8_FOIL_VOLUME_SHAPE_FACTOR = 0.68
CELL8_DEFAULT_SAIL_CONTOUR_TEXT = "0.0,0.0\n0.2,0.0\n0.463,0.313\n0.463,0.713\n0.09,2.4\n-0.2,2.4\n-0.375,0.418\n-0.2,0.0\n0.0,0.0"
CELL8_RIGHTING_CACHE = {}
CELL8_APPENDAGE_CONTOURS = {}
CELL8_SAIL_GEOMETRY = {}


def cell8_available_stl_options():
    candidates = []
    if CELL8_STL_FOLDER.exists():
        candidates.extend(sorted(p for p in CELL8_STL_FOLDER.iterdir() if p.is_file() and p.suffix.lower() == ".stl"))
    if CELL8_DEFAULT_STL_PATH.exists():
        candidates.append(CELL8_DEFAULT_STL_PATH)
    unique = []
    seen = set()
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    if not unique:
        return [("Aucun STL trouve", "")]
    return [(str(path), str(path)) for path in unique]


def cell8_load_stl_triangles(path, unit="mm"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"STL introuvable: {path}")
    scale = CELL8_STL_UNIT_SCALE.get(str(unit).lower(), 1.0)
    raw = path.read_bytes()
    triangles = []
    if raw[:5].lower() == b"solid" and b"facet" in raw[:4096].lower():
        text = raw.decode("utf-8", errors="ignore")
        verts = []
        for line in text.splitlines():
            parts = line.strip().split()
            if len(parts) == 4 and parts[0].lower() == "vertex":
                verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
        if len(verts) >= 3 and len(verts) % 3 == 0:
            return np.asarray(verts, dtype=float).reshape((-1, 3, 3)) * scale
    if len(raw) < 84:
        raise ValueError("Fichier STL invalide ou trop court.")
    n_tri = struct.unpack("<I", raw[80:84])[0]
    offset = 84
    for _ in range(int(n_tri)):
        if offset + 50 > len(raw):
            break
        coords = struct.unpack("<12fH", raw[offset:offset + 50])
        triangles.append([coords[3:6], coords[6:9], coords[9:12]])
        offset += 50
    if not triangles:
        raise ValueError("Aucun triangle STL lisible.")
    return np.asarray(triangles, dtype=float) * scale


def cell8_signed_distance_to_water(points, waterline_level, vertical_idx=2, up_sign=1.0):
    pts = np.asarray(points, dtype=float)
    return (pts[..., vertical_idx] - float(waterline_level)) * float(up_sign)


def cell8_triangle_area(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0
    cross = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    return float(0.5 * np.linalg.norm(cross, axis=1).sum())


def cell8_surface_area_centroid(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    cross = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    areas = 0.5 * np.linalg.norm(cross, axis=1)
    area = float(areas.sum())
    if area <= 1e-18:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    centroid = (areas[:, None] * tris.mean(axis=1)).sum(axis=0) / area
    return area, centroid


def cell8_clean_polygon_vertices(poly, tol):
    cleaned = []
    for p in poly:
        p = np.asarray(p, dtype=float)
        if not cleaned or np.linalg.norm(p - cleaned[-1]) > tol:
            cleaned.append(p)
    if len(cleaned) > 1 and np.linalg.norm(cleaned[0] - cleaned[-1]) <= tol:
        cleaned.pop()
    return cleaned


def cell8_line_plane_intersection(p0, p1, d0, d1):
    denom = float(d0 - d1)
    if abs(denom) < 1e-18:
        return 0.5 * (np.asarray(p0, dtype=float) + np.asarray(p1, dtype=float))
    t = float(d0) / denom
    return np.asarray(p0, dtype=float) + t * (np.asarray(p1, dtype=float) - np.asarray(p0, dtype=float))


def cell8_clip_triangle_under_water(tri, waterline_level, vertical_idx, up_sign, eps):
    poly = [np.asarray(p, dtype=float) for p in tri]
    output = []
    for i in range(3):
        p0 = poly[i]
        p1 = poly[(i + 1) % 3]
        d0 = float(cell8_signed_distance_to_water(p0, waterline_level, vertical_idx, up_sign))
        d1 = float(cell8_signed_distance_to_water(p1, waterline_level, vertical_idx, up_sign))
        in0 = d0 <= eps
        in1 = d1 <= eps
        if in0 and in1:
            output.append(p1)
        elif in0 and not in1:
            output.append(cell8_line_plane_intersection(p0, p1, d0, d1))
        elif not in0 and in1:
            output.append(cell8_line_plane_intersection(p0, p1, d0, d1))
            output.append(p1)
    return cell8_clean_polygon_vertices(output, eps * 10.0)


def cell8_triangulate_fan(poly):
    if len(poly) < 3:
        return []
    return [np.asarray([poly[0], poly[i], poly[i + 1]], dtype=float) for i in range(1, len(poly) - 1)]


def cell8_dedupe_points(points, tol):
    unique = []
    for p in points:
        p = np.asarray(p, dtype=float)
        if not any(np.linalg.norm(p - q) <= tol for q in unique):
            unique.append(p)
    return unique


def cell8_triangle_waterline_segments(tri, waterline_level, vertical_idx, up_sign, eps):
    tri = np.asarray(tri, dtype=float)
    d = cell8_signed_distance_to_water(tri, waterline_level, vertical_idx, up_sign)
    segments = []
    points = []
    for i in range(3):
        p0 = tri[i]
        p1 = tri[(i + 1) % 3]
        d0 = float(d[i])
        d1 = float(d[(i + 1) % 3])
        on0 = abs(d0) <= eps
        on1 = abs(d1) <= eps
        if on0 and on1:
            segments.append((p0.copy(), p1.copy()))
        elif d0 * d1 < -eps * eps:
            points.append(cell8_line_plane_intersection(p0, p1, d0, d1))
        elif on0:
            points.append(p0.copy())
        elif on1:
            points.append(p1.copy())
    points = cell8_dedupe_points(points, eps * 10.0)
    if len(points) == 2:
        segments.append((points[0], points[1]))
    return [(a, b) for a, b in segments if np.linalg.norm(a - b) > eps]


def cell8_point_key(point, tol):
    return tuple(np.round(np.asarray(point, dtype=float) / tol).astype(np.int64))


def cell8_merge_segments_to_loops(segments, tol):
    points = []
    key_to_id = {}
    edges = set()

    def add_point(p):
        key = cell8_point_key(p, tol)
        if key not in key_to_id:
            key_to_id[key] = len(points)
            points.append(np.asarray(p, dtype=float))
        return key_to_id[key]

    for a, b in segments:
        ia = add_point(a)
        ib = add_point(b)
        if ia != ib:
            edges.add(tuple(sorted((ia, ib))))

    adjacency = {i: set() for i in range(len(points))}
    for ia, ib in edges:
        adjacency[ia].add(ib)
        adjacency[ib].add(ia)

    loops = []
    unused = set(edges)
    while unused:
        ia, ib = unused.pop()
        loop = [ia, ib]
        previous = ia
        current = ib
        while True:
            candidates = []
            for nxt in adjacency[current]:
                edge = tuple(sorted((current, nxt)))
                if edge in unused:
                    candidates.append(nxt)
            if not candidates:
                break
            nxt = candidates[0]
            if len(candidates) > 1:
                for cand in candidates:
                    if cand != previous:
                        nxt = cand
                        break
            unused.remove(tuple(sorted((current, nxt))))
            if nxt == loop[0]:
                break
            loop.append(nxt)
            previous, current = current, nxt
        if len(loop) >= 3 and loop[0] in adjacency[loop[-1]]:
            loops.append(np.asarray([points[i] for i in loop], dtype=float))
    return loops


def cell8_polygon_area_2d(points2d):
    pts = np.asarray(points2d, dtype=float)
    if len(pts) < 3:
        return 0.0
    x = pts[:, 0]
    y = pts[:, 1]
    return float(0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y))


def cell8_ear_clip_indices(points2d, eps=1e-12):
    pts = np.asarray(points2d, dtype=float)
    n = len(pts)
    if n < 3:
        return []
    if n == 3:
        return [[0, 1, 2]]
    vertices = list(range(n))
    if cell8_polygon_area_2d(pts) < 0.0:
        vertices.reverse()
    triangles = []
    guard = 0
    def cross2(u, v):
        return u[0] * v[1] - u[1] * v[0]
    while len(vertices) > 3 and guard < n * n:
        guard += 1
        ear_found = False
        for pos in range(len(vertices)):
            i_prev = vertices[pos - 1]
            i_curr = vertices[pos]
            i_next = vertices[(pos + 1) % len(vertices)]
            a = pts[i_prev]
            b = pts[i_curr]
            c = pts[i_next]
            if cross2(b - a, c - a) <= eps:
                continue
            contains = False
            for j in vertices:
                if j in (i_prev, i_curr, i_next):
                    continue
                p = pts[j]
                if (cross2(b - a, p - a) >= -eps and cross2(c - b, p - b) >= -eps and cross2(a - c, p - c) >= -eps):
                    contains = True
                    break
            if contains:
                continue
            triangles.append([i_prev, i_curr, i_next])
            del vertices[pos]
            ear_found = True
            break
        if not ear_found:
            return [[vertices[0], vertices[i], vertices[i + 1]] for i in range(1, len(vertices) - 1)]
    if len(vertices) == 3:
        triangles.append(vertices[:])
    return triangles


def cell8_triangulate_cap_loops(loops, vertical_idx, tol):
    cap = []
    axes = [0, 1, 2]
    axes.remove(vertical_idx)
    for loop in loops:
        loop = cell8_clean_polygon_vertices(loop, tol)
        if len(loop) < 3:
            continue
        pts2d = np.asarray(loop, dtype=float)[:, axes]
        for ids in cell8_ear_clip_indices(pts2d, eps=tol * tol):
            cap.append(np.asarray([loop[i] for i in ids], dtype=float))
    if cap:
        return np.asarray(cap, dtype=float)
    return np.zeros((0, 3, 3), dtype=float)


def cell8_signed_mesh_volume(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0
    return float((np.einsum("ij,ij->i", tris[:, 0], np.cross(tris[:, 1], tris[:, 2])) / 6.0).sum())


def cell8_orient_triangles_outward(triangles):
    tris = np.asarray(triangles, dtype=float).copy()
    if tris.size == 0:
        return tris
    reference = tris.reshape((-1, 3)).mean(axis=0)
    centers = tris.mean(axis=1)
    normals = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    dots = np.einsum("ij,ij->i", normals, centers - reference)
    flip = dots < 0.0
    tmp = tris[flip, 1].copy()
    tris[flip, 1] = tris[flip, 2]
    tris[flip, 2] = tmp
    if cell8_signed_mesh_volume(tris) < 0.0:
        tmp = tris[:, 1].copy()
        tris[:, 1] = tris[:, 2]
        tris[:, 2] = tmp
    return tris


def cell8_mesh_volume_and_centroid(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    vols = np.einsum("ij,ij->i", tris[:, 0], np.cross(tris[:, 1], tris[:, 2])) / 6.0
    volume = float(vols.sum())
    if abs(volume) < 1e-18:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    centroids = tris.sum(axis=1) / 4.0
    centroid = (vols[:, None] * centroids).sum(axis=0) / volume
    if volume < 0.0:
        volume = -volume
    return float(volume), centroid


def cell8_analyze_immersion(triangles, waterline_level, vertical_idx, up_sign, eps, merge_tol):
    shell = []
    segments = []
    for tri in triangles:
        clipped = cell8_clip_triangle_under_water(tri, waterline_level, vertical_idx, up_sign, eps)
        shell.extend(cell8_triangulate_fan(clipped))
        segments.extend(cell8_triangle_waterline_segments(tri, waterline_level, vertical_idx, up_sign, eps))
    shell = np.asarray(shell, dtype=float) if shell else np.zeros((0, 3, 3), dtype=float)
    loops = cell8_merge_segments_to_loops(segments, merge_tol)
    cap = cell8_triangulate_cap_loops(loops, vertical_idx, merge_tol)
    closed = np.vstack([shell, cap]) if len(cap) else shell.copy()
    closed = cell8_orient_triangles_outward(closed)
    volume, B = cell8_mesh_volume_and_centroid(closed)
    return {"volume_m3": float(volume), "center_buoyancy_m": B, "wet_area_m2": cell8_triangle_area(shell), "loops": loops}


def cell8_rotate_points_about_axis(points, origin, axis_vector, angle_deg):
    pts = np.asarray(points, dtype=float)
    origin = np.asarray(origin, dtype=float)
    axis = np.asarray(axis_vector, dtype=float)
    axis = axis / max(np.linalg.norm(axis), 1e-18)
    angle = np.radians(float(angle_deg))
    rel = pts - origin
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    cross = np.cross(axis, rel)
    dot = np.einsum("...i,i->...", rel, axis)[..., None]
    return origin + rel * cos_a + cross * sin_a + axis * dot * (1.0 - cos_a)


def cell8_rotate_triangles_about_axis(triangles, origin, axis_vector, angle_deg):
    shape = triangles.shape
    rotated = cell8_rotate_points_about_axis(triangles.reshape((-1, 3)), origin, axis_vector, angle_deg)
    return rotated.reshape(shape)


def cell8_find_waterline_for_volume(triangles, target_volume, vertical_idx, up_sign, eps, merge_tol, max_iter=50):
    verts = triangles.reshape((-1, 3))
    h = verts[:, vertical_idx] * up_sign
    span = max(float(h.max() - h.min()), eps)
    lo_h = float(h.min() - 0.05 * span)
    hi_h = float(h.max() + 0.05 * span)
    hi_res = cell8_analyze_immersion(triangles, hi_h / up_sign, vertical_idx, up_sign, eps, merge_tol)
    if target_volume >= hi_res["volume_m3"]:
        return hi_h / up_sign, hi_res
    for _ in range(int(max_iter)):
        mid_h = 0.5 * (lo_h + hi_h)
        res = cell8_analyze_immersion(triangles, mid_h / up_sign, vertical_idx, up_sign, eps, merge_tol)
        if res["volume_m3"] < target_volume:
            lo_h = mid_h
        else:
            hi_h = mid_h
    level = 0.5 * (lo_h + hi_h) / up_sign
    return level, cell8_analyze_immersion(triangles, level, vertical_idx, up_sign, eps, merge_tol)


def cell8_polygon_area_centroid_2d(points):
    pts = np.asarray(points, dtype=float)
    if len(pts) < 3:
        return 0.0, np.array([np.nan, np.nan])
    if not np.allclose(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])
    x = pts[:, 0]
    y = pts[:, 1]
    cross = x[:-1] * y[1:] - x[1:] * y[:-1]
    area_signed = 0.5 * float(cross.sum())
    if abs(area_signed) <= 1e-18:
        return 0.0, np.array([np.nan, np.nan])
    cx = float(((x[:-1] + x[1:]) * cross).sum() / (6.0 * area_signed))
    cy = float(((y[:-1] + y[1:]) * cross).sum() / (6.0 * area_signed))
    return abs(area_signed), np.array([cx, cy])


def cell8_parse_contour_points_2d(text, label="contour"):
    points = []
    for raw_line in str(text).splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        line = line.replace(";", ",").replace("\t", ",").replace(" ", ",")
        parts = [part for part in line.split(",") if part]
        if len(parts) < 2:
            continue
        points.append((float(parts[0]), float(parts[1])))
    if len(points) < 3:
        raise ValueError(f"{label}: au moins 3 points x,y sont necessaires.")
    pts = np.asarray(points, dtype=float)
    if not np.allclose(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])
    return pts


def cell8_sail_geometry_from_text(text, label="voile"):
    pts = cell8_parse_contour_points_2d(text, label=label)
    area, centroid = cell8_polygon_area_centroid_2d(pts)
    y_min = float(np.min(pts[:, 1]))
    y_max = float(np.max(pts[:, 1]))
    span = y_max - y_min
    if area <= 0.0 or span <= 0.0:
        raise ValueError(f"{label}: aire ou hauteur nulle.")
    return {
        "points": pts,
        "area_m2": float(area),
        "span_m": float(span),
        "c_moy_m": float(area / span),
        "centroid": centroid,
        "h_aero_m": float(max(centroid[1] - y_min, 1e-6)),
        "y_min_m": y_min,
        "y_max_m": y_max,
    }


def cell8_get_sail_geometry():
    geom = CELL8_SAIL_GEOMETRY.get("voile", None)
    if geom is None:
        geom = cell8_sail_geometry_from_text(CELL8_DEFAULT_SAIL_CONTOUR_TEXT, label="voile cellule 8")
        CELL8_SAIL_GEOMETRY["voile"] = geom
    return geom


def cell8_apply_sail_contour(textarea, area_widget, chord_widget, output_widget=None):
    try:
        geom = cell8_sail_geometry_from_text(textarea.value, label="voile")
        area_widget.value = geom["area_m2"]
        chord_widget.value = geom["c_moy_m"]
        CELL8_SAIL_GEOMETRY["voile"] = geom
        CELL8_RIGHTING_CACHE.clear()
        if output_widget is not None:
            with output_widget:
                clear_output(wait=True)
                print(f"voile: A={geom['area_m2']:.4f} m2, c_moy={geom['c_moy_m']:.4f} m, h_aero={geom['h_aero_m']:.4f} m")
        return geom
    except Exception as exc:
        if output_widget is not None:
            with output_widget:
                clear_output(wait=True)
                print(f"Erreur voile: {exc}")
        return None


def cell8_contour_geometry_from_text(text, label="contour"):
    pts = cell8_parse_contour_points_2d(text, label=label)
    area, centroid = cell8_polygon_area_centroid_2d(pts)
    y_min = float(np.min(pts[:, 1]))
    y_max = float(np.max(pts[:, 1]))
    height = y_max - y_min
    if area <= 0.0 or height <= 0.0:
        raise ValueError(f"{label}: aire ou hauteur nulle.")
    return {
        "points": pts,
        "area_m2": float(area),
        "height_m": float(height),
        "centroid": centroid,
        "centroid_offset_m": float(centroid[1] - y_min),
        "mean_chord_m": float(area / height),
    }


def cell8_appendage_vertical_centroid_offset(key, height_m):
    geom = CELL8_APPENDAGE_CONTOURS.get(key, None)
    height_m = max(float(height_m), 1e-9)
    if geom is not None:
        offset = float(geom.get("centroid_offset_m", 0.5 * height_m))
        if np.isfinite(offset):
            return float(np.clip(offset, 0.0, height_m))
    return 0.5 * height_m


def cell8_apply_appendage_contour(textarea, area_widget, height_widget, key, label, output_widget=None):
    try:
        geom = cell8_contour_geometry_from_text(textarea.value, label=label)
        area_widget.value = geom["area_m2"]
        height_widget.value = geom["height_m"]
        CELL8_APPENDAGE_CONTOURS[key] = geom
        if output_widget is not None:
            with output_widget:
                clear_output(wait=True)
                print(f"{label}: S={geom['area_m2']:.4f} m2, h={geom['height_m']:.4f} m, c_moy={geom['mean_chord_m']:.4f} m")
        return geom
    except Exception as exc:
        if output_widget is not None:
            with output_widget:
                clear_output(wait=True)
                print(f"Erreur {label}: {exc}")
        return None


def cell8_get_sail_area_centroid():
    geom = cell8_get_sail_geometry()
    return float(geom["area_m2"]), np.asarray(geom["centroid"], dtype=float)


def cell8_foil_volume_from_area_span(area_m2, span_m, thickness_ratio=0.12):
    area_m2 = max(float(area_m2), 0.0)
    span_m = max(float(span_m), 1e-9)
    chord_eff = area_m2 / span_m
    return float(CELL8_FOIL_VOLUME_SHAPE_FACTOR * thickness_ratio * area_m2 * chord_eff)


def cell8_make_height_point(reference, vertical_idx, up_sign, height_value):
    point = np.asarray(reference, dtype=float).copy()
    point[vertical_idx] = float(height_value) / float(up_sign)
    return point


def cell8_estimate_complete_boat_cg(triangles, S_derive, hauteur_quille, m_quille_kg, S_safran, hauteur_safran, hull_thickness_mm, sail_skin_thickness_mm, rho_carbon=1500.0):
    vertices = triangles.reshape((-1, 3))
    reference = vertices.mean(axis=0)
    vertical_idx = 2
    up_sign = 1.0
    h = vertices[:, vertical_idx] * up_sign
    h_bottom = float(h.min())
    h_top = float(h.max())
    masses = []
    points = []
    hull_area, hull_centroid = cell8_surface_area_centroid(triangles)
    hull_mass = float(rho_carbon) * hull_area * max(float(hull_thickness_mm), 0.0) * 1e-3
    hull_height = float(hull_centroid[vertical_idx]) if np.all(np.isfinite(hull_centroid)) else 0.5 * (h_bottom + h_top)
    masses.append(hull_mass)
    points.append(cell8_make_height_point(reference, vertical_idx, up_sign, hull_height))
    derive_volume = cell8_foil_volume_from_area_span(S_derive, hauteur_quille)
    derive_cg_offset = cell8_appendage_vertical_centroid_offset("derive", hauteur_quille)
    masses.append(float(rho_carbon) * derive_volume)
    points.append(cell8_make_height_point(reference, vertical_idx, up_sign, h_bottom - derive_cg_offset))
    masses.append(max(float(m_quille_kg), 0.0))
    points.append(cell8_make_height_point(reference, vertical_idx, up_sign, h_bottom - float(hauteur_quille)))
    rudder_volume = cell8_foil_volume_from_area_span(S_safran, hauteur_safran)
    rudder_cg_offset = cell8_appendage_vertical_centroid_offset("safran", hauteur_safran)
    masses.append(float(rho_carbon) * rudder_volume)
    points.append(cell8_make_height_point(reference, vertical_idx, up_sign, h_bottom - rudder_cg_offset))
    sail_geom = cell8_get_sail_geometry()
    sail_area = float(sail_geom["area_m2"])
    sail_mass = float(rho_carbon) * sail_area * max(float(sail_skin_thickness_mm), 0.0) * 1e-3
    sail_height = h_top + max(float(sail_geom["h_aero_m"]), 0.0)
    masses.append(sail_mass)
    points.append(cell8_make_height_point(reference, vertical_idx, up_sign, sail_height))
    masses = np.asarray(masses, dtype=float)
    points = np.asarray(points, dtype=float)
    total_mass = float(masses.sum())
    if total_mass <= 1e-12:
        return reference, total_mass
    return (masses[:, None] * points).sum(axis=0) / total_mass, total_mass


def cell8_build_hull_righting_curve(stl_path, draft_m, rho_water, g, S_derive, hauteur_quille, m_quille_kg, S_safran, hauteur_safran, hull_thickness_mm, sail_skin_thickness_mm, theta_max_deg, step_deg=2.0):
    derive_cg_offset = cell8_appendage_vertical_centroid_offset("derive", hauteur_quille)
    rudder_cg_offset = cell8_appendage_vertical_centroid_offset("safran", hauteur_safran)
    sail_geom = cell8_get_sail_geometry()
    sail_centroid = np.asarray(sail_geom["centroid"], dtype=float)
    key = (str(stl_path), float(draft_m), float(rho_water), float(S_derive), float(hauteur_quille), float(m_quille_kg), float(S_safran), float(hauteur_safran), float(derive_cg_offset), float(rudder_cg_offset), float(sail_geom["area_m2"]), float(sail_geom["c_moy_m"]), float(sail_geom["h_aero_m"]), float(sail_centroid[0]), float(sail_centroid[1]), float(hull_thickness_mm), float(sail_skin_thickness_mm), float(theta_max_deg), float(step_deg))
    if key in CELL8_RIGHTING_CACHE:
        return CELL8_RIGHTING_CACHE[key]
    triangles = cell8_load_stl_triangles(stl_path, "mm")
    vertices = triangles.reshape((-1, 3))
    extent = vertices.max(axis=0) - vertices.min(axis=0)
    eps = max(float(extent.max()) * 1e-9, 1e-12)
    merge_tol = max(float(extent.max()) * 1e-7, 1e-9)
    vertical_idx = 2
    up_sign = 1.0
    long_vec = np.array([1.0, 0.0, 0.0])
    up_vec = np.array([0.0, 0.0, 1.0])
    G, total_mass = cell8_estimate_complete_boat_cg(triangles, S_derive, hauteur_quille, m_quille_kg, S_safran, hauteur_safran, hull_thickness_mm, sail_skin_thickness_mm, CELL8_RHO_CARBON_DEFAULT)
    h = vertices[:, vertical_idx] * up_sign
    waterline_level = (float(h.min()) + max(float(draft_m), 0.0)) / up_sign
    upright = cell8_analyze_immersion(triangles, waterline_level, vertical_idx, up_sign, eps, merge_tol)
    target_volume = max(float(upright["volume_m3"]), 0.0)
    theta_values = np.arange(0.0, max(float(theta_max_deg), 0.0) + float(step_deg), float(step_deg))
    moments = []
    gz_values = []
    for theta in theta_values:
        rotated = cell8_rotate_triangles_about_axis(triangles, G, long_vec, theta)
        _, res = cell8_find_waterline_for_volume(rotated, target_volume, vertical_idx, up_sign, eps, merge_tol)
        B = res["center_buoyancy_m"]
        if np.all(np.isfinite(B)):
            gz_signed = float(np.dot(long_vec, np.cross(B - G, up_vec)))
            gz_abs = abs(gz_signed)
        else:
            gz_abs = 0.0
        gz_values.append(gz_abs)
        moments.append(float(rho_water) * target_volume * float(g) * gz_abs)
    curve = {
        "theta_deg": theta_values,
        "M_hull": np.asarray(moments, dtype=float),
        "GZ_abs": np.asarray(gz_values, dtype=float),
        "G": G,
        "mass_model_kg": total_mass,
        "target_volume_m3": target_volume,
        "displacement_kg": float(rho_water) * target_volume,
        "stl_path": str(stl_path),
    }
    CELL8_RIGHTING_CACHE[key] = curve
    return curve


def cell8_hull_righting_value(curve, theta_deg):
    if curve is None:
        return 0.0
    theta = abs(float(theta_deg))
    return float(np.interp(theta, curve["theta_deg"], curve["M_hull"], left=0.0, right=float(curve["M_hull"][-1])))

def solve_heel_equilibrium(F_aero_lat, F_derive, h_aero, h_derive,
                           m_quille_kg, hauteur_quille, theta_max_deg=45.0,
                           g=9.81, tolerance=0.05, max_iterations=60,
                           F_safran=0.0, h_safran=0.0, hull_righting_curve=None):
    """Resout M_aero(theta) = M_derive(theta) + M_safran(theta) + M_quille/theta ou M_hull_STL(theta)."""
    F_aero_lat = abs(float(F_aero_lat))
    F_derive = abs(float(F_derive))
    F_safran = abs(float(F_safran))
    h_aero = max(float(h_aero), 0.0)
    h_derive = max(float(h_derive), 0.0)
    h_safran = max(float(h_safran), 0.0)
    m_quille_kg = max(float(m_quille_kg), 0.0)
    hauteur_quille = max(float(hauteur_quille), 0.0)
    theta_max_deg = max(float(theta_max_deg), 0.0)
    M_quille_coeff = m_quille_kg * float(g) * hauteur_quille

    def moments(theta_deg):
        theta_rad = np.radians(theta_deg)
        cos_t = np.cos(theta_rad)
        sin_t = np.sin(theta_rad)
        M_aero = F_aero_lat * cos_t * h_aero
        M_derive = F_derive * cos_t * h_derive
        M_safran = F_safran * cos_t * h_safran
        M_hull = cell8_hull_righting_value(hull_righting_curve, theta_deg) if hull_righting_curve is not None else 0.0
        M_quille = 0.0 if hull_righting_curve is not None else M_quille_coeff * sin_t
        residual = M_aero - M_derive - M_safran - M_quille - M_hull
        return float(residual), float(M_aero), float(M_derive), float(M_safran), float(M_quille), float(M_hull)

    if theta_max_deg <= 0.0 or (F_aero_lat <= 0.0 and F_derive <= 0.0 and F_safran <= 0.0):
        residual, M_aero, M_derive, M_safran, M_quille, M_hull = moments(0.0)
        return {'theta_deg': 0.0, 'converged': True, 'reason': 'theta nul',
                'M_aero': M_aero, 'M_derive': M_derive, 'M_safran': M_safran,
                'M_quille': M_quille, 'M_hull': M_hull, 'heel_residual': residual}

    residual_0, M_aero_0, M_derive_0, M_safran_0, M_quille_0, M_hull_0 = moments(0.0)
    if residual_0 <= 0.0:
        return {'theta_deg': 0.0, 'converged': True, 'reason': 'stable a theta=0',
                'M_aero': M_aero_0, 'M_derive': M_derive_0, 'M_safran': M_safran_0,
                'M_quille': M_quille_0, 'M_hull': M_hull_0, 'heel_residual': residual_0}

    residual_max, M_aero_max, M_derive_max, M_safran_max, M_quille_max, M_hull_max = moments(theta_max_deg)
    if residual_max > 0.0:
        return {'theta_deg': theta_max_deg, 'converged': False, 'reason': 'theta_max atteint',
                'M_aero': M_aero_max, 'M_derive': M_derive_max, 'M_safran': M_safran_max,
                'M_quille': M_quille_max, 'M_hull': M_hull_max, 'heel_residual': residual_max}

    lo = 0.0
    hi = theta_max_deg
    theta_mid = 0.0
    M_aero = M_derive = M_safran = M_quille = M_hull = residual = 0.0
    converged = False
    for _ in range(int(max_iterations)):
        theta_mid = 0.5 * (lo + hi)
        residual, M_aero, M_derive, M_safran, M_quille, M_hull = moments(theta_mid)
        if abs(residual) < float(tolerance):
            converged = True
            break
        if residual > 0.0:
            lo = theta_mid
        else:
            hi = theta_mid
    if not converged:
        theta_mid = 0.5 * (lo + hi)
        residual, M_aero, M_derive, M_safran, M_quille, M_hull = moments(theta_mid)
    return {'theta_deg': float(theta_mid), 'converged': converged,
            'reason': 'ok' if converged else 'max_iterations',
            'M_aero': M_aero, 'M_derive': M_derive, 'M_safran': M_safran,
            'M_quille': M_quille, 'M_hull': M_hull, 'heel_residual': residual}

def solve_speed_from_propulsion(F_prop, S_wetted, CR_friction, Rcoque_coeff, rho_water):
    """Vitesse issue de F_prop = 0.5*rho*V^2*S*(CR+Rcoque)."""
    F_prop = max(float(F_prop), 0.0)
    C_R_total = float(CR_friction) + float(Rcoque_coeff)
    denom = 0.5 * float(rho_water) * float(S_wetted) * C_R_total
    if F_prop <= 0.0 or denom <= 0.0:
        return 0.0
    return float(np.sqrt(F_prop / denom))


def apply_heel_correction_to_equilibrium(eq, aero, derive_tables, S_derive,
                                         hauteur_quille, h_aero, S_wetted,
                                         CR_friction, Rcoque_coeff, rho_water,
                                         mu_water=1.05e-3, m_quille_kg=1.0,
                                         theta_max_deg=45.0, g=9.81,
                                         psi_max_deg=20.0, n_iter=60,
                                         S_safran=0.052, c_safran=None,
                                         safran_tables=None,
                                         hauteur_safran=0.65,
                                         hull_righting_curve=None):
    """Couple V_B, derive, safran et gite.

    Pour chaque essai de vitesse, le meme ecoulement relatif V_B/cos(psi) sert a la quille
    et au safran fixe. La resistance longitudinale des deux appendices est ajoutee a la coque.
    """
    V_B_raw = float(eq.get('V_B', 0.0))
    hauteur_quille = max(float(hauteur_quille), 1e-6)
    hauteur_safran = max(float(hauteur_safran), 1e-6)
    S_derive = max(float(S_derive), 1e-9)
    S_safran = max(float(S_safran), 1e-9)
    c_derive_eff = max(float(S_derive) / hauteur_quille, 1e-6)
    c_safran_eff = max(float(c_safran), 1e-6) if c_safran is not None else max(float(S_safran) / hauteur_safran, 1e-6)
    h_derive = 0.5 * hauteur_quille
    h_safran = 0.5 * hauteur_safran
    F_aero_lat = float(aero.get('F_aero_lat', 0.0))
    F_prop_base = max(float(eq.get('F_prop_spy', aero.get('F_prop', 0.0))), 0.0)

    if safran_tables is None:
        safran_tables = derive_tables

    appendages = [
        {'name': 'derive', 'S': S_derive, 'c': c_derive_eff, 'h': h_derive, 'tables': derive_tables},
        {'name': 'safran', 'S': S_safran, 'c': c_safran_eff, 'h': h_safran, 'tables': safran_tables},
    ]

    def evaluate_speed(V_B_eval):
        lateral = solve_appendage_lateral_equilibrium(
            F_aero_lat=F_aero_lat,
            V_B=V_B_eval,
            appendage_tables=derive_tables,
            appendages=appendages,
            rho_water=rho_water,
            mu_water=mu_water,
            psi_max_deg=psi_max_deg,
        )
        derive = lateral['appendages'].get('derive', _zero_appendage_forces('derive'))
        safran = lateral['appendages'].get('safran', _zero_appendage_forces('safran'))
        heel = solve_heel_equilibrium(
            abs(F_aero_lat), derive['F_lat'], h_aero, h_derive,
            m_quille_kg, hauteur_quille, theta_max_deg=theta_max_deg, g=g,
            F_safran=safran['F_lat'], h_safran=h_safran,
            hull_righting_curve=hull_righting_curve,
        )
        theta_rad = np.radians(float(heel['theta_deg']))
        F_prop_gite = max(0.0, F_prop_base * np.cos(theta_rad))
        R_hull = calc_hydro_drag(V_B_eval, S_wetted, CR_friction, Rcoque_coeff, rho_water)
        R_appendices_long = float(lateral['R_long_total'])
        R_total = R_hull + R_appendices_long
        residual = F_prop_gite - R_total
        return {
            'V_B': float(V_B_eval),
            'residual': float(residual),
            'F_prop_gite': float(F_prop_gite),
            'R_hydro_gite': float(R_hull),
            'R_appendices_long': float(R_appendices_long),
            'R_total': float(R_total),
            'lateral': lateral,
            'derive': derive,
            'safran': safran,
            'heel': heel,
        }

    if F_prop_base <= 0.0:
        evaluated = evaluate_speed(0.0)
        return {'V_B_raw': float(V_B_raw), 'V_B_gite': 0.0,
                'F_prop_base': float(F_prop_base), 'F_prop_gite': 0.0,
                'R_hydro_gite': 0.0, 'R_appendices_long': 0.0, 'R_total': 0.0,
                'derive': evaluated['derive'], 'safran': evaluated['safran'],
                'appendages_total': evaluated['lateral'], 'heel': evaluated['heel'],
                'c_derive_eff': float(c_derive_eff), 'h_derive': float(h_derive),
                'c_safran_eff': float(c_safran_eff), 'h_safran': float(h_safran),
                'speed_converged': True, 'speed_reason': 'F_prop nul'}

    lo = 0.0
    hi = max(0.1, V_B_raw * 1.5, 1.0)
    eval_lo = evaluate_speed(lo)
    eval_hi = evaluate_speed(hi)
    for _ in range(25):
        if eval_hi['residual'] <= 0.0:
            break
        hi *= 1.5
        eval_hi = evaluate_speed(hi)

    if eval_lo['residual'] <= 0.0:
        final = eval_lo
        speed_converged = True
        speed_reason = 'stable a V=0'
    elif eval_hi['residual'] > 0.0:
        final = eval_hi
        speed_converged = False
        speed_reason = 'borne haute atteinte'
    else:
        final = eval_hi
        speed_converged = False
        speed_reason = 'max_iterations'
        for _ in range(int(n_iter)):
            mid = 0.5 * (lo + hi)
            eval_mid = evaluate_speed(mid)
            if abs(eval_mid['residual']) < 0.5:
                final = eval_mid
                speed_converged = True
                speed_reason = 'ok'
                break
            if eval_mid['residual'] > 0.0:
                lo = mid
            else:
                hi = mid
            final = eval_mid

    return {'V_B_raw': float(V_B_raw), 'V_B_gite': float(final['V_B']),
            'F_prop_base': float(F_prop_base), 'F_prop_gite': float(final['F_prop_gite']),
            'R_hydro_gite': float(final['R_hydro_gite']),
            'R_appendices_long': float(final['R_appendices_long']),
            'R_total': float(final['R_total']),
            'derive': final['derive'], 'safran': final['safran'],
            'appendages_total': final['lateral'], 'heel': final['heel'],
            'c_derive_eff': float(c_derive_eff), 'h_derive': float(h_derive),
            'c_safran_eff': float(c_safran_eff), 'h_safran': float(h_safran),
            'speed_converged': bool(speed_converged), 'speed_reason': speed_reason}
def solve_equilibrium_7(V_TW, CAP_deg, polar_tables, alpha_trim_deg, A_sail, c_moy,
                        rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                        max_iterations=50, tolerance=0.5, V_B_max_factor=2.0):
    """Resout F_prop(CAP) = R_hydro(V_B) par dichotomie."""
    V_TW = float(V_TW)
    CAP_deg = float(CAP_deg)

    aero_info = calc_aero_simple(V_TW, CAP_deg, alpha_trim_deg, polar_tables,
                                 A_sail, c_moy, rho_air=rho_air)
    F_prop = float(aero_info['F_prop'])

    if F_prop <= 0.0:
        return {
            'V_B': 0.0,
            'converged': True,
            'aero': aero_info,
            'R_hydro': 0.0,
            'mode': 'voile',
            'reason': 'F_prop <= 0',
        }

    V_B_min = 0.0
    V_B_max = max(0.1, float(V_TW) * float(V_B_max_factor))

    for _ in range(20):
        R_high = calc_hydro_drag(V_B_max, S_wetted, CR_friction, Rcoque_coeff, rho_water)
        if F_prop - R_high <= 0.0:
            break
        V_B_max *= 1.5

    converged = False
    V_B_mid = 0.0
    for _ in range(int(max_iterations)):
        V_B_mid = 0.5 * (V_B_min + V_B_max)
        R_hydro = calc_hydro_drag(V_B_mid, S_wetted, CR_friction, Rcoque_coeff, rho_water)
        error = F_prop - R_hydro

        if abs(error) < tolerance:
            converged = True
            break

        if error > 0.0:
            V_B_min = V_B_mid
        else:
            V_B_max = V_B_mid

    V_B_solution = float(V_B_mid if converged else 0.5 * (V_B_min + V_B_max))
    R_solution = calc_hydro_drag(V_B_solution, S_wetted, CR_friction, Rcoque_coeff, rho_water)
    return {
        'V_B': V_B_solution,
        'converged': converged,
        'aero': aero_info,
        'R_hydro': float(R_solution),
        'mode': 'voile',
        'reason': 'ok' if converged else 'max_iterations',
    }


def calc_spy_equilibrium(V_TW, CAP_deg, rho_air, rho_water, S_wetted, CR_friction,
                         Rcoque_coeff, Cx_spy=1.0, S_spy=1.0):
    """Resout F_spy(CAP) = R_hydro(V_B) pour le mode SPY."""
    V_TW = float(V_TW)
    cap = normalize_cap_for_half_polar(CAP_deg)
    Cx_spy = float(Cx_spy)
    S_spy = float(S_spy)

    if cap < 90.0:
        return None

    q_dyn_air = 0.5 * rho_air * V_TW**2
    F_spy = Cx_spy * q_dyn_air * S_spy
    F_prop_spy = F_spy * np.cos(np.radians(180.0 - cap))

    if F_prop_spy <= 0.0:
        return None

    V_B_min = 0.0
    V_B_max = max(0.1, V_TW * 2.0)

    for _ in range(20):
        R_high = calc_hydro_drag(V_B_max, S_wetted, CR_friction, Rcoque_coeff, rho_water)
        if F_prop_spy - R_high <= 0.0:
            break
        V_B_max *= 1.5

    converged = False
    V_B_mid = 0.0
    for _ in range(50):
        V_B_mid = 0.5 * (V_B_min + V_B_max)
        R_hydro = calc_hydro_drag(V_B_mid, S_wetted, CR_friction, Rcoque_coeff, rho_water)
        error = F_prop_spy - R_hydro

        if abs(error) < 0.5:
            converged = True
            break

        if error > 0.0:
            V_B_min = V_B_mid
        else:
            V_B_max = V_B_mid

    V_B_solution = float(V_B_mid if converged else 0.5 * (V_B_min + V_B_max))
    return {
        'V_B': V_B_solution,
        'mode': 'spy',
        'converged': converged,
        'F_prop_spy': float(F_prop_spy),
    }


def solve_equilibrium_hybrid(V_TW, CAP_deg, polar_tables, alpha_trim_deg, A_sail, c_moy,
                             rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                             Cx_spy=1.0, S_spy=1.0, max_iterations=50,
                             tolerance=0.5, V_B_max_factor=2.0):
    """Compare VOILE et SPY, retourne le mode donnant la plus grande vitesse."""
    eq_voile = solve_equilibrium_7(V_TW, CAP_deg, polar_tables, alpha_trim_deg, A_sail, c_moy,
                                   rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                                   max_iterations, tolerance, V_B_max_factor)
    V_B_voile = eq_voile['V_B'] if eq_voile else 0.0

    eq_spy = calc_spy_equilibrium(V_TW, CAP_deg, rho_air, rho_water, S_wetted, CR_friction,
                                  Rcoque_coeff, Cx_spy=Cx_spy, S_spy=S_spy)
    V_B_spy = eq_spy['V_B'] if eq_spy else 0.0

    if V_B_spy > V_B_voile:
        eq_final = eq_spy
        eq_final['V_B'] = V_B_spy
        # La derive reste basee sur les efforts lateraux aero du modele voile.
        eq_final['aero'] = eq_voile.get('aero', {}) if eq_voile else {}
    else:
        eq_final = eq_voile
        eq_final['V_B'] = V_B_voile

    eq_final['V_B_voile'] = V_B_voile
    eq_final['V_B_spy'] = V_B_spy
    return eq_final


def compute_polaire_7(V_TW, profile_file, alpha_trim_deg, A_sail, c_moy,
                      S_wetted, CR_friction, Rcoque_coeff,
                      rho_air=1.225, rho_water=1025.0, n_cap=180,
                      Cx_spy=1.0, S_spy=1.0, use_hybrid=True,
                      S_derive=0.2, c_derive=None, mu_water=1.05e-3,
                      psi_max_deg=20.0, m_quille_kg=1.0, hauteur_quille=1.0,
                      theta_max_deg=45.0, g=9.81,
                      S_safran=0.052, c_safran=None, hauteur_safran=0.65,
                      derive_profile='NACA0012', safran_profile='NACA0012',
                      use_hull_righting=False, hull_stl_path='', hull_draft_m=0.25,
                      hull_thickness_mm=25.0, sail_skin_thickness_mm=3.0):
    """Calcule polaire V_B vs CAP avec profils de derive et safran configurables."""
    try:
        polar_tables = _load_polar_file(profile_file)
    except Exception as e:
        print(f"Erreur chargement polaire: {e}")
        return None

    try:
        derive_profile = str(derive_profile).replace(' ', '').upper()
        safran_profile = str(safran_profile).replace(' ', '').upper()
        derive_file = _resolve_profile_file(derive_profile, folder='.')
        safran_file = _resolve_profile_file(safran_profile, folder='.')
        derive_tables = _load_polar_file(derive_file)
        safran_tables = _load_polar_file(safran_file)
    except Exception as e:
        print(f"Erreur chargement polaires appendices: {e}")
        return None

    hauteur_quille = max(float(hauteur_quille), 1e-6)
    hauteur_safran = max(float(hauteur_safran), 1e-6)
    S_derive = max(float(S_derive), 1e-9)
    S_safran = max(float(S_safran), 1e-9)
    c_derive = S_derive / hauteur_quille
    c_safran = S_safran / hauteur_safran if c_safran is None else max(float(c_safran), 1e-6)
    h_aero = compute_aero_application_height_from_wing()
    hull_righting_curve = None
    if bool(use_hull_righting):
        try:
            hull_righting_curve = cell8_build_hull_righting_curve(
                stl_path=hull_stl_path,
                draft_m=hull_draft_m,
                rho_water=rho_water,
                g=g,
                S_derive=S_derive,
                hauteur_quille=hauteur_quille,
                m_quille_kg=m_quille_kg,
                S_safran=S_safran,
                hauteur_safran=hauteur_safran,
                hull_thickness_mm=hull_thickness_mm,
                sail_skin_thickness_mm=sail_skin_thickness_mm,
                theta_max_deg=theta_max_deg,
            )
            print(f"Moment coque STL actif: {Path(hull_stl_path).name}, deplacement={hull_righting_curve['displacement_kg']:.2f} kg, masse modele={hull_righting_curve['mass_model_kg']:.2f} kg")
        except Exception as exc:
            print(f"Erreur moment coque STL: {exc}")
            return None

    n_calc = (int(n_cap) // 2) + 1
    CAPs_calc = np.linspace(0.0, 180.0, n_calc)
    rows = []

    for cap in CAPs_calc:
        try:
            if use_hybrid:
                eq = solve_equilibrium_hybrid(V_TW, cap, polar_tables, alpha_trim_deg, A_sail, c_moy,
                                              rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff,
                                              Cx_spy=Cx_spy, S_spy=S_spy)
            else:
                eq = solve_equilibrium_7(V_TW, cap, polar_tables, alpha_trim_deg, A_sail, c_moy,
                                         rho_air, rho_water, S_wetted, CR_friction, Rcoque_coeff)

            aero = eq.get('aero', {})
            heel_result = apply_heel_correction_to_equilibrium(
                eq=eq,
                aero=aero,
                derive_tables=derive_tables,
                S_derive=S_derive,
                hauteur_quille=hauteur_quille,
                h_aero=h_aero,
                S_wetted=S_wetted,
                CR_friction=CR_friction,
                Rcoque_coeff=Rcoque_coeff,
                rho_water=rho_water,
                mu_water=mu_water,
                m_quille_kg=m_quille_kg,
                theta_max_deg=theta_max_deg,
                g=g,
                psi_max_deg=psi_max_deg,
                S_safran=S_safran,
                c_safran=c_safran,
                safran_tables=safran_tables,
                hauteur_safran=hauteur_safran,
                hull_righting_curve=hull_righting_curve,
            )
            V_B = float(heel_result['V_B_gite'])
            V_B_raw = float(heel_result['V_B_raw'])
            F_aero_lat = float(aero.get('F_aero_lat', 0.0))
            derive = heel_result['derive']
            safran = heel_result['safran']
            appendages_total = heel_result['appendages_total']
            heel = heel_result['heel']

            cap_rad = np.radians(cap)
            vmg_signed = V_B * np.cos(cap_rad)
            vmg = abs(vmg_signed)
            vmg_pres = max(vmg_signed, 0.0)
            vmg_portant = max(-vmg_signed, 0.0)

            rows.append({
                'CAP': float(cap),
                'TWA': float(cap),
                'V_B': V_B,
                'V_B_raw': V_B_raw,
                'V_B_voile': float(eq.get('V_B_voile', 0.0)),
                'V_B_spy': float(eq.get('V_B_spy', 0.0)),
                'mode': eq.get('mode', 'voile'),
                'F_prop': float(heel_result['F_prop_base']),
                'F_prop_gite': float(heel_result['F_prop_gite']),
                'F_aero_lat': F_aero_lat,
                'F_aero_lat_abs': abs(F_aero_lat),
                'L_aero': float(aero.get('L_aero', np.nan)),
                'D_aero': float(aero.get('D_aero', np.nan)),
                'R_hydro': float(heel_result['R_hydro_gite']),
                'R_appendices_long': float(heel_result['R_appendices_long']),
                'R_total': float(heel_result['R_total']),
                'Cl': float(aero.get('Cl', np.nan)),
                'Cd': float(aero.get('Cd', np.nan)),
                'Re': float(aero.get('Re', np.nan)),
                'projection_case': aero.get('projection_case', ''),
                'lateral_case': aero.get('lateral_case', ''),
                'derive_profile': derive_profile,
                'safran_profile': safran_profile,
                'psi_deg': float(appendages_total['psi_deg']),
                'theta_deg': float(heel['theta_deg']),
                'V_lateral': float(np.sign(appendages_total['psi_deg']) * derive['V_lateral']),
                'V_derive': float(derive['V_derive']),
                'F_hydro_derive': float(derive['F_hydro_derive']),
                'F_hydro_safran': float(safran['F_hydro_safran']),
                'F_hydro_appendices': float(appendages_total['F_lat_total']),
                'R_long_derive': float(derive['R_long_derive']),
                'R_long_safran': float(safran['R_long_safran']),
                'M_aero': float(heel['M_aero']),
                'M_derive': float(heel['M_derive']),
                'M_safran': float(heel['M_safran']),
                'M_quille': float(heel['M_quille']),
                'M_hull': float(heel.get('M_hull', 0.0)),
                'righting_mode': 'coque_stl' if hull_righting_curve is not None else 'quille_simple',
                'heel_residual': float(heel['heel_residual']),
                'heel_converged': bool(heel['converged']),
                'heel_reason': heel['reason'],
                'h_aero': float(h_aero),
                'h_derive': float(heel_result['h_derive']),
                'h_safran': float(heel_result['h_safran']),
                'hauteur_quille': float(hauteur_quille),
                'hauteur_safran': float(hauteur_safran),
                'c_derive_eff': float(heel_result['c_derive_eff']),
                'c_safran_eff': float(heel_result['c_safran_eff']),
                'L_derive': float(derive['L_derive']),
                'D_derive': float(derive['D_derive']),
                'Cl_derive': float(derive['Cl_derive']),
                'Cd_derive': float(derive['Cd_derive']),
                'Re_derive': float(derive['Re_derive']),
                'alpha_derive_deg': float(np.sign(appendages_total['psi_deg']) * derive['alpha_derive_deg']),
                'L_safran': float(safran['L_safran']),
                'D_safran': float(safran['D_safran']),
                'Cl_safran': float(safran['Cl_safran']),
                'Cd_safran': float(safran['Cd_safran']),
                'Re_safran': float(safran['Re_safran']),
                'V_safran': float(safran['V_safran']),
                'alpha_safran_deg': float(np.sign(appendages_total['psi_deg']) * safran['alpha_safran_deg']),
                'derive_converged': bool(appendages_total['converged']),
                'derive_reason': appendages_total['reason'],
                'VMG': float(vmg),
                'VMG_signed': float(vmg_signed),
                'VMG_pres': float(vmg_pres),
                'VMG_portant': float(vmg_portant),
                'converged': eq.get('converged', True),
                'speed_converged': bool(heel_result['speed_converged']),
                'speed_reason': heel_result['speed_reason'],
            })
        except Exception as ex:
            print(f"  CAP={cap} deg: {ex}")
            continue

    if not rows:
        return None

    df_first = pd.DataFrame(rows)

    rows_sym = []
    for _, row in df_first.iterrows():
        cap_orig = float(row['CAP'])
        if 0.0 < cap_orig < 180.0:
            row_sym = row.copy()
            row_sym['CAP'] = 360.0 - cap_orig
            row_sym['TWA'] = row_sym['CAP']
            cap_sym_rad = np.radians(row_sym['CAP'])
            vmg_signed_sym = float(row['V_B']) * np.cos(cap_sym_rad)
            row_sym['VMG_signed'] = float(vmg_signed_sym)
            row_sym['VMG'] = float(abs(vmg_signed_sym))
            row_sym['VMG_pres'] = float(max(vmg_signed_sym, 0.0))
            row_sym['VMG_portant'] = float(max(-vmg_signed_sym, 0.0))
            row_sym['F_aero_lat'] = -float(row['F_aero_lat'])
            row_sym['V_lateral'] = -float(row['V_lateral'])
            row_sym['psi_deg'] = -float(row['psi_deg'])
            row_sym['alpha_derive_deg'] = -float(row.get('alpha_derive_deg', 0.0))
            row_sym['alpha_safran_deg'] = -float(row.get('alpha_safran_deg', 0.0))
            rows_sym.append(row_sym)

    if rows_sym:
        df_sym = pd.DataFrame(rows_sym)
        df_polaire = pd.concat([df_first, df_sym], ignore_index=True)
        df_polaire = df_polaire.sort_values('CAP').reset_index(drop=True)
    else:
        df_polaire = df_first

    return df_polaire


# ===== Widgets =====

profile_widget7 = widgets.Dropdown(
    options=profiles_list if 'profiles_list' in globals() else ['NACA0012'],
    value=(profiles_list[0] if 'profiles_list' in globals() and len(profiles_list) > 0 else 'NACA0012'),
    description='Profil:',
    style={'description_width': '90px'}
)

appendage_profiles7 = profiles_list if 'profiles_list' in globals() and len(profiles_list) > 0 else ['NACA0012']
derive_profile_widget7 = widgets.Dropdown(
    options=appendage_profiles7,
    value=('NACA0012' if 'NACA0012' in appendage_profiles7 else appendage_profiles7[0]),
    description='Profil derive:',
    style={'description_width': '90px'}
)
safran_profile_widget7 = widgets.Dropdown(
    options=appendage_profiles7,
    value=('NACA0012' if 'NACA0012' in appendage_profiles7 else appendage_profiles7[0]),
    description='Profil safran:',
    style={'description_width': '90px'}
)

cell8_initial_sail_geom7 = cell8_get_sail_geometry()
V_TW_widget7 = widgets.FloatText(value=10.0, description='V_TW (m/s):', step=0.5, min=1.0, max=30.0, style={'description_width': '90px'})
alpha_trim_widget7 = widgets.FloatText(value=0.0, description='alpha_trim:', step=0.5, min=-15.0, max=20.0, style={'description_width': '90px'})
A_sail_widget7 = widgets.FloatText(value=cell8_initial_sail_geom7["area_m2"], description='A_sail (m2):', step=0.1, min=0.1, max=10.0, style={'description_width': '90px'})
c_moy_widget7 = widgets.FloatText(value=cell8_initial_sail_geom7["c_moy_m"], description='c_moy (m):', step=0.05, min=0.1, max=2.0, style={'description_width': '90px'})
sail_contour_textarea7 = widgets.Textarea(
    value=CELL8_DEFAULT_SAIL_CONTOUR_TEXT,
    description='Contour voile:',
    placeholder='x,y en m - y depuis le pied de voile vers le haut',
    layout=widgets.Layout(width='520px', height='130px'),
    style={'description_width': '105px'},
)
sail_contour_btn7 = widgets.Button(description='Appliquer voile', icon='check', layout=widgets.Layout(width='160px'))
sail_contour_out7 = widgets.Output(layout=widgets.Layout(width='340px'))
S_wetted_widget7 = widgets.FloatText(value=S_wetted_default if 'S_wetted_default' in globals() else 1.5, description='S mouillee:', step=0.1, min=0.1, max=10.0, style={'description_width': '90px'})
CR_friction_widget7 = widgets.FloatText(value=0.003, description='CR friction:', step=0.0001, min=0.001, max=0.01, style={'description_width': '90px'})
Rcoque_coeff_widget7 = widgets.FloatText(value=0.002, description='Rcoque coeff:', step=0.0001, min=0.0, max=0.05, style={'description_width': '90px'})
rho_air_widget7 = widgets.FloatText(value=1.225, description='rho air:', step=0.01, min=1.0, max=1.3, style={'description_width': '90px'})
rho_water_widget7 = widgets.FloatText(value=1025.0, description='rho eau:', step=1.0, min=1000.0, max=1030.0, style={'description_width': '90px'})
mu_water_widget7 = widgets.FloatText(value=1.05e-3, description='mu eau:', step=0.00005, style={'description_width': '90px'})
n_cap_widget7 = widgets.IntSlider(value=180, min=36, max=360, step=36, description='N CAP:', style={'description_width': '90px'}, continuous_update=False)
Cx_spy_widget7 = widgets.FloatText(value=1.0, description='Cx SPY:', step=0.1, min=0.1, max=2.0, style={'description_width': '90px'})
S_spy_widget7 = widgets.FloatText(value=1.0, description='S SPY (m2):', step=0.1, min=0.1, max=10.0, style={'description_width': '90px'})
S_derive_widget7 = widgets.FloatText(value=0.20, description='S derive:', step=0.05, min=0.01, max=10.0, style={'description_width': '90px'})
hauteur_quille_widget7 = widgets.FloatText(value=1.00, description='h quille:', step=0.05, min=0.05, max=10.0, style={'description_width': '90px'})
S_safran_widget7 = widgets.FloatText(value=0.052, description='S safran:', step=0.01, min=0.001, max=10.0, style={'description_width': '90px'})
hauteur_safran_widget7 = widgets.FloatText(value=0.65, description='h safran:', step=0.05, min=0.05, max=10.0, style={'description_width': '90px'})
derive_contour_textarea7 = widgets.Textarea(
    value='0.00,0.00\n0.20,0.00\n0.20,1.00\n0.00,1.00',
    description='Contour derive:',
    placeholder='x,y en m - y depuis la coque vers le bas',
    layout=widgets.Layout(width='380px', height='105px'),
    style={'description_width': '105px'},
)
derive_contour_btn7 = widgets.Button(description='Appliquer derive', icon='check', layout=widgets.Layout(width='165px'))
derive_contour_out7 = widgets.Output(layout=widgets.Layout(width='290px'))
safran_contour_textarea7 = widgets.Textarea(
    value='0.00,0.00\n0.08,0.00\n0.08,0.65\n0.00,0.65',
    description='Contour safran:',
    placeholder='x,y en m - y depuis la coque vers le bas',
    layout=widgets.Layout(width='380px', height='105px'),
    style={'description_width': '105px'},
)
safran_contour_btn7 = widgets.Button(description='Appliquer safran', icon='check', layout=widgets.Layout(width='165px'))
safran_contour_out7 = widgets.Output(layout=widgets.Layout(width='290px'))
m_quille_widget7 = widgets.FloatText(value=5.00, description='m quille:', step=0.5, min=0.0, max=500.0, style={'description_width': '90px'})
theta_max_widget7 = widgets.FloatText(value=45.0, description='theta max:', step=1.0, min=0.0, max=90.0, style={'description_width': '90px'})
psi_max_widget7 = widgets.FloatText(value=20.0, description='psi max:', step=1.0, min=1.0, max=45.0, style={'description_width': '90px'})

use_hull_righting_widget7 = widgets.Checkbox(value=False, description='Moment coque STL')
hull_stl_widget7 = widgets.Dropdown(
    options=cell8_available_stl_options(),
    description='Coque STL:',
    layout=widgets.Layout(width='430px'),
    style={'description_width': '90px'},
)
hull_stl_refresh_btn7 = widgets.Button(description='Rafraichir STL', layout=widgets.Layout(width='135px'))
hull_draft_widget7 = widgets.FloatText(value=0.25, description='Tirant eau:', step=0.01, min=0.0, max=10.0, style={'description_width': '90px'})
hull_thickness_widget7 = widgets.FloatText(value=25.0, description='Ep coque mm:', step=0.5, min=0.0, max=100.0, style={'description_width': '90px'})
sail_skin_thickness_widget7 = widgets.FloatText(value=3.0, description='Ep voile mm:', step=0.1, min=0.0, max=30.0, style={'description_width': '90px'})

def _refresh_hull_stl_widget7(_=None):
    current = getattr(hull_stl_widget7, 'value', '')
    options = cell8_available_stl_options()
    values = [value for _, value in options]
    hull_stl_widget7.options = options
    if current in values:
        hull_stl_widget7.value = current
    elif values:
        hull_stl_widget7.value = values[0]

hull_stl_refresh_btn7.on_click(_refresh_hull_stl_widget7)


def _apply_sail_contour7(_=None):
    return cell8_apply_sail_contour(
        sail_contour_textarea7,
        A_sail_widget7,
        c_moy_widget7,
        output_widget=sail_contour_out7,
    )


def _apply_derive_contour7(_=None):
    cell8_apply_appendage_contour(
        derive_contour_textarea7,
        S_derive_widget7,
        hauteur_quille_widget7,
        key="derive",
        label="derive",
        output_widget=derive_contour_out7,
    )


def _apply_safran_contour7(_=None):
    cell8_apply_appendage_contour(
        safran_contour_textarea7,
        S_safran_widget7,
        hauteur_safran_widget7,
        key="safran",
        label="safran",
        output_widget=safran_contour_out7,
    )


sail_contour_btn7.on_click(_apply_sail_contour7)
derive_contour_btn7.on_click(_apply_derive_contour7)
safran_contour_btn7.on_click(_apply_safran_contour7)

polaire_btn7 = widgets.Button(
    description='Tracer polaire V_B(CAP)',
    button_style='success',
    tooltip='Calculate and plot boat speed polar versus CAP with drift',
    icon='calculator'
)

out7 = widgets.Output()


def _on_polaire7(_):
    """Callback: calcul et trace polaire."""
    with out7:
        clear_output()
        try:
            def safe_get_float(widget, default=0.0):
                try:
                    val = widget.value
                    if val is None:
                        return default
                    return float(val)
                except (TypeError, ValueError, AttributeError):
                    return default

            def safe_get_int(widget, default=0):
                try:
                    val = widget.value
                    if val is None:
                        return default
                    return int(val)
                except (TypeError, ValueError, AttributeError):
                    return default

            def safe_get_str(widget, default=''):
                try:
                    val = widget.value
                    if val is None:
                        return default
                    return str(val).strip()
                except (TypeError, AttributeError):
                    return default

            sail_geom = _apply_sail_contour7()
            if sail_geom is None:
                print('Erreur: corriger le contour voile cellule 8 avant de lancer la polaire.')
                return

            V_TW = safe_get_float(V_TW_widget7, 10.0)
            alpha_trim = safe_get_float(alpha_trim_widget7, 0.0)
            profile = safe_get_str(profile_widget7, 'NACA0012').upper()
            derive_profile = safe_get_str(derive_profile_widget7, 'NACA0012').upper()
            safran_profile = safe_get_str(safran_profile_widget7, 'NACA0012').upper()
            A_sail = safe_get_float(A_sail_widget7, 1.0)
            c_moy = safe_get_float(c_moy_widget7, 0.5)
            S_wetted = safe_get_float(S_wetted_widget7, 1.0)
            CR_friction = safe_get_float(CR_friction_widget7, 0.001)
            Rcoque_coeff = safe_get_float(Rcoque_coeff_widget7, 0.05)
            rho_air = safe_get_float(rho_air_widget7, 1.225)
            rho_water = safe_get_float(rho_water_widget7, 1025.0)
            mu_water = safe_get_float(mu_water_widget7, 1.05e-3)
            n_cap = safe_get_int(n_cap_widget7, 180)
            Cx_spy = safe_get_float(Cx_spy_widget7, 1.0)
            S_spy = safe_get_float(S_spy_widget7, 1.0)
            S_derive = safe_get_float(S_derive_widget7, 0.2)
            hauteur_quille = safe_get_float(hauteur_quille_widget7, 1.0)
            m_quille = safe_get_float(m_quille_widget7, 5.0)
            S_safran = safe_get_float(S_safran_widget7, 0.052)
            hauteur_safran = safe_get_float(hauteur_safran_widget7, 0.65)
            theta_max = safe_get_float(theta_max_widget7, 45.0)
            psi_max = safe_get_float(psi_max_widget7, 20.0)
            use_hull_righting = bool(getattr(use_hull_righting_widget7, 'value', False))
            hull_stl_path = safe_get_str(hull_stl_widget7, '')
            hull_draft = safe_get_float(hull_draft_widget7, 0.25)
            hull_thickness = safe_get_float(hull_thickness_widget7, 25.0)
            sail_skin_thickness = safe_get_float(sail_skin_thickness_widget7, 3.0)

            if V_TW <= 0.0 or n_cap <= 0:
                print('Erreur: V_TW et n_cap doivent etre > 0')
                return
            if S_derive <= 0.0 or hauteur_quille <= 0.0 or S_safran <= 0.0 or hauteur_safran <= 0.0:
                print('Erreur: S_derive, hauteur_quille, S_safran et hauteur_safran doivent etre > 0')
                return
            if use_hull_righting:
                if not hull_stl_path:
                    print('Erreur: selectionner un STL de coque pour le moment redressant.')
                    return
                if not Path(hull_stl_path).exists():
                    print(f'Erreur: STL introuvable: {hull_stl_path}')
                    return
                if hull_draft <= 0.0:
                    print('Erreur: le tirant eau doit etre > 0 pour le moment coque STL.')
                    return

            if n_cap % 36 != 0:
                n_cap = max(36, min(360, ((n_cap + 18) // 36) * 36))
                print(f'n_cap ajuste a {n_cap}')

            profile_file = _resolve_profile_file(profile, folder='.')

            print('Calcul polaire V_B(CAP) - vent reel TW + quille/safran couples...')
            print(f'   Profil aero: {profile}, derive: {derive_profile}, safran: {safran_profile}, alpha_trim={alpha_trim} deg, V_TW={V_TW:.1f} m/s')
            print(f'   Voile cellule 8: A={A_sail:.3f} m2, c_moy={c_moy:.3f} m, h_aero={sail_geom["h_aero_m"]:.3f} m')
            print(f'   S_derive={S_derive:.3f} m2, h_quille={hauteur_quille:.3f} m, m_quille={m_quille:.2f} kg')
            print(f'   S_safran={S_safran:.3f} m2, h_safran={hauteur_safran:.3f} m, masse safran=0 kg')
            if CELL8_APPENDAGE_CONTOURS:
                active_contours = ', '.join(sorted(CELL8_APPENDAGE_CONTOURS.keys()))
                print(f'   Contours appendices appliques: {active_contours}')
            print(f'   theta_max={theta_max:.1f} deg, psi_max={psi_max:.1f} deg')
            if use_hull_righting:
                print(f'   Moment coque STL: {Path(hull_stl_path).name}, tirant={hull_draft:.3f} m, ep coque={hull_thickness:.1f} mm')
            print(f'   N points: {n_cap}, SPY: Cx={Cx_spy}, S={S_spy:.2f} m2\n')

            df_polaire = compute_polaire_7(
                V_TW=V_TW,
                profile_file=str(profile_file),
                alpha_trim_deg=alpha_trim,
                A_sail=A_sail,
                c_moy=c_moy,
                S_wetted=S_wetted,
                CR_friction=CR_friction,
                Rcoque_coeff=Rcoque_coeff,
                rho_air=rho_air,
                rho_water=rho_water,
                n_cap=n_cap,
                Cx_spy=Cx_spy,
                S_spy=S_spy,
                use_hybrid=True,
                S_derive=S_derive,
                c_derive=None,
                mu_water=mu_water,
                psi_max_deg=psi_max,
                m_quille_kg=m_quille,
                hauteur_quille=hauteur_quille,
                theta_max_deg=theta_max,
                S_safran=S_safran,
                hauteur_safran=hauteur_safran,
                derive_profile=derive_profile,
                safran_profile=safran_profile,
                use_hull_righting=use_hull_righting,
                hull_stl_path=hull_stl_path,
                hull_draft_m=hull_draft,
                hull_thickness_mm=hull_thickness,
                sail_skin_thickness_mm=sail_skin_thickness,
            )

            if df_polaire is None or df_polaire.shape[0] == 0:
                print('Polaire vide!')
                return

            fig = plt.figure(figsize=(28, 14))

            ax1 = plt.subplot(241, projection='polar')
            CAPs_rad = np.radians(df_polaire['CAP'].values)
            V_Bs = df_polaire['V_B'].values
            ax1.plot(CAPs_rad, V_Bs, 'b-', linewidth=2.5)
            ax1.fill(CAPs_rad, V_Bs, alpha=0.2, color='blue')
            ax1.set_theta_zero_location('N')
            ax1.set_theta_direction(-1)
            ax1.set_title(f'Rose V_B\nalpha_trim={alpha_trim} deg', fontsize=10, weight='bold')
            ax1.grid(True, alpha=0.3)

            ax2 = plt.subplot(242)
            ax2.plot(df_polaire['CAP'], df_polaire['V_B_voile'], 'b--', linewidth=1.4, alpha=0.6, label='VOILE')
            ax2.plot(df_polaire['CAP'], df_polaire['V_B_spy'], 'r--', linewidth=1.4, alpha=0.6, label='SPY')
            ax2.plot(df_polaire['CAP'], df_polaire['V_B_raw'], color='0.55', linestyle=':', linewidth=1.5, label='sans gite')
            ax2.plot(df_polaire['CAP'], df_polaire['V_B'], 'g-', linewidth=2.3, marker='o', markersize=3, label='couplee gite+appendices')
            idx_max = df_polaire['V_B'].idxmax()
            V_B_max = df_polaire.loc[idx_max, 'V_B']
            CAP_max = df_polaire.loc[idx_max, 'CAP']
            ax2.plot(CAP_max, V_B_max, 'k*', markersize=16, label=f'{V_B_max:.2f} m/s @ {CAP_max:.0f} deg')
            ax2.set_xlabel('CAP (deg)', fontsize=9, weight='bold')
            ax2.set_ylabel('V_B (m/s)', fontsize=9, weight='bold')
            ax2.set_title('Vitesse bateau', fontsize=10, weight='bold')
            ax2.grid(True, alpha=0.3)
            ax2.set_xlim(0, 360)
            ax2.legend(fontsize=8, loc='best')

            ax3 = plt.subplot(243)
            ax3.plot(df_polaire['CAP'], df_polaire['F_aero_lat_abs'], 'm-', linewidth=2.0, label='|Faero lat|')
            ax3.plot(df_polaire['CAP'], df_polaire['F_hydro_derive'], 'c--', linewidth=1.4, label='Fhydro derive')
            ax3.plot(df_polaire['CAP'], df_polaire['F_hydro_safran'], color='tab:orange', linestyle='--', linewidth=1.4, label='Fhydro safran')
            ax3.plot(df_polaire['CAP'], df_polaire['F_hydro_appendices'], color='tab:green', linewidth=1.8, label='Fhydro total')
            ax3.set_xlabel('CAP (deg)', fontsize=9, weight='bold')
            ax3.set_ylabel('Force laterale (N)', fontsize=9, weight='bold')
            ax3.set_title('Equilibre lateral', fontsize=10, weight='bold')
            ax3.grid(True, alpha=0.3)
            ax3.set_xlim(0, 360)
            ax3.legend(fontsize=8, loc='best')

            ax4 = plt.subplot(244)
            ax4.plot(df_polaire['CAP'], df_polaire['theta_deg'], color='tab:purple', linewidth=2.4, marker='o', markersize=3, label='theta gite')
            ax4.axhline(y=theta_max, color='r', linestyle=':', linewidth=1.2, label='theta max')
            ax4.set_xlabel('CAP (deg)', fontsize=9, weight='bold')
            ax4.set_ylabel('theta (deg)', fontsize=9, weight='bold')
            ax4.set_title('Angle de gite theta(CAP)', fontsize=10, weight='bold')
            ax4.grid(True, alpha=0.3)
            ax4.set_xlim(0, 360)
            ax4.legend(fontsize=8, loc='best')

            ax5 = plt.subplot(245)
            ax5.plot(df_polaire['CAP'], df_polaire['psi_deg'], 'k-', linewidth=2.0, marker='o', markersize=3, label='psi derive')
            ax5.axhline(y=0.0, color='0.5', linewidth=0.7)
            ax5.set_xlabel('CAP (deg)', fontsize=9, weight='bold')
            ax5.set_ylabel('psi (deg)', fontsize=9, weight='bold')
            ax5.set_title('Angle de derive psi(CAP)', fontsize=10, weight='bold')
            ax5.grid(True, alpha=0.3)
            ax5.set_xlim(0, 360)
            ax5.legend(fontsize=8, loc='best')

            ax6 = plt.subplot(246)
            ax6.plot(df_polaire['CAP'], df_polaire['V_lateral'], color='tab:orange', linewidth=2.0, label='V derive laterale')
            ax6.plot(df_polaire['CAP'], np.abs(df_polaire['V_lateral']), color='tab:red', linestyle='--', linewidth=1.5, label='|V derive|')
            ax6.axhline(y=0.0, color='k', linewidth=0.6)
            ax6.set_xlabel('CAP (deg)', fontsize=9, weight='bold')
            ax6.set_ylabel('Vitesse de derive (m/s)', fontsize=9, weight='bold')
            ax6.set_title('Vitesse de derive', fontsize=10, weight='bold')
            ax6.grid(True, alpha=0.3)
            ax6.set_xlim(0, 360)
            ax6.legend(fontsize=8, loc='best')

            ax7 = plt.subplot(247)
            ax7.plot(df_polaire['CAP'], df_polaire['VMG'], color='0.55', linewidth=1.4, label='VMG absolue')
            ax7.plot(df_polaire['CAP'], df_polaire['VMG_pres'], 'g-', linewidth=2.3, marker='s', markersize=3, alpha=0.8, label='VMG pres')
            ax7.plot(df_polaire['CAP'], df_polaire['VMG_portant'], color='tab:blue', linestyle='--', linewidth=1.8, label='VMG portant')
            idx_vmg_pres = df_polaire['VMG_pres'].idxmax()
            VMG_pres_max = df_polaire.loc[idx_vmg_pres, 'VMG_pres']
            CAP_vmg_pres = df_polaire.loc[idx_vmg_pres, 'CAP']
            ax7.plot(CAP_vmg_pres, VMG_pres_max, 'r*', markersize=16, label=f'Pres: {VMG_pres_max:.2f} m/s @ {CAP_vmg_pres:.0f} deg')
            ax7.set_xlabel('CAP (deg)', fontsize=10, weight='bold')
            ax7.set_ylabel('VMG (m/s)', fontsize=10, weight='bold')
            ax7.set_title('Vitesse utile: pres / portant', fontsize=11, weight='bold')
            ax7.grid(True, alpha=0.3)
            ax7.set_xlim(0, 360)
            ax7.legend(fontsize=8, loc='best')

            ax8 = plt.subplot(248)
            M_redressement = df_polaire['M_derive'] + df_polaire['M_safran'] + df_polaire['M_quille'] + df_polaire['M_hull']
            ax8.plot(df_polaire['CAP'], df_polaire['M_aero'], color='tab:purple', linewidth=2.0, label='M aero')
            ax8.plot(df_polaire['CAP'], df_polaire['M_derive'], color='c', linestyle=':', linewidth=1.2, label='M derive')
            ax8.plot(df_polaire['CAP'], df_polaire['M_safran'], color='tab:orange', linestyle=':', linewidth=1.2, label='M safran')
            ax8.plot(df_polaire['CAP'], df_polaire['M_hull'], color='tab:blue', linestyle='-.', linewidth=1.8, label='M coque STL')
            ax8.plot(df_polaire['CAP'], M_redressement, color='tab:green', linestyle='--', linewidth=1.8, label='M redressement total')
            ax8.set_xlabel('CAP (deg)', fontsize=9, weight='bold')
            ax8.set_ylabel('Moment (N.m)', fontsize=9, weight='bold')
            ax8.set_title('Equilibre moments gite', fontsize=10, weight='bold')
            ax8.grid(True, alpha=0.3)
            ax8.set_xlim(0, 360)
            ax8.legend(fontsize=8, loc='best')

            plt.tight_layout()
            display(fig)
            plt.close(fig)

            n_spy = (df_polaire['mode'] == 'spy').sum()
            n_derive_ok = int(df_polaire['derive_converged'].sum())
            n_heel_ok = int(df_polaire['heel_converged'].sum())
            n_speed_ok = int(df_polaire['speed_converged'].sum())
            print('\nPolaire calculee:')
            print(f'   - {df_polaire.shape[0]} points')
            print(f'   - Mode VOILE: {len(df_polaire) - n_spy} points')
            print(f'   - Mode SPY: {n_spy} points ({100*n_spy/len(df_polaire):.1f}%)')
            print(f"   - V_B range: {df_polaire['V_B'].min():.2f} a {df_polaire['V_B'].max():.2f} m/s")
            print(f"   - psi range: {df_polaire['psi_deg'].min():.2f} a {df_polaire['psi_deg'].max():.2f} deg")
            print(f"   - theta range: {df_polaire['theta_deg'].min():.2f} a {df_polaire['theta_deg'].max():.2f} deg")
            print(f'   - derive convergee: {n_derive_ok}/{len(df_polaire)} points')
            print(f'   - gite convergee: {n_heel_ok}/{len(df_polaire)} points')
            print(f'   - vitesse couplee convergee: {n_speed_ok}/{len(df_polaire)} points')
            idx_vmg = df_polaire['VMG'].idxmax()
            VMG_max = df_polaire.loc[idx_vmg, 'VMG']
            CAP_vmg = df_polaire.loc[idx_vmg, 'CAP']
            print(f'   - VMG absolue max: {VMG_max:.2f} m/s @ CAP={CAP_vmg:.0f} deg')
            print(f'   - VMG pres max: {VMG_pres_max:.2f} m/s @ CAP={CAP_vmg_pres:.0f} deg')

            globals()['polaire_results_7'] = df_polaire

        except Exception as e:
            print(f'Erreur: {e}')
            import traceback
            traceback.print_exc()


polaire_btn7.on_click(_on_polaire7)

# Afficher interface
display(widgets.VBox([
    widgets.HTML('<h3>POLAIRE V_B(CAP) - VENT REEL TW - QUILLE/SAFRAN + GITE + COQUE STL</h3>'),
    widgets.HTML('<b>Projection CAP, quille et safran couples dans V_B, psi et gite avec profils configurables</b>'),
    widgets.HBox([profile_widget7, V_TW_widget7, alpha_trim_widget7]),
    widgets.HTML('<b>Geometrie voile cellule 8</b>'),
    widgets.HBox([sail_contour_textarea7, widgets.VBox([sail_contour_btn7, sail_contour_out7])]),
    widgets.HBox([A_sail_widget7, c_moy_widget7, S_wetted_widget7]),
    widgets.HBox([CR_friction_widget7, Rcoque_coeff_widget7]),
    widgets.HBox([rho_air_widget7, rho_water_widget7, mu_water_widget7]),
    widgets.HBox([n_cap_widget7, Cx_spy_widget7, S_spy_widget7]),
    widgets.HTML('<b>Quille/derive et safran</b>'),
    widgets.HBox([derive_profile_widget7, safran_profile_widget7]),
    widgets.HBox([S_derive_widget7, hauteur_quille_widget7, m_quille_widget7]),
    widgets.HBox([S_safran_widget7, hauteur_safran_widget7]),
    widgets.HTML('<b>Contours appendices optionnels</b>'),
    widgets.HBox([derive_contour_textarea7, widgets.VBox([derive_contour_btn7, derive_contour_out7])]),
    widgets.HBox([safran_contour_textarea7, widgets.VBox([safran_contour_btn7, safran_contour_out7])]),
    widgets.HBox([theta_max_widget7, psi_max_widget7]),
    widgets.HTML('<b>Moment redressant hydrostatique de coque</b>'),
    widgets.HBox([use_hull_righting_widget7, hull_stl_widget7, hull_stl_refresh_btn7]),
    widgets.HBox([hull_draft_widget7, hull_thickness_widget7, sail_skin_thickness_widget7]),
    widgets.HBox([polaire_btn7]),
    out7,
]))

print('Cellule 8 prete: polaire V_B(CAP) avec contour voile local, quille/safran, trainee appendices, gite et option moment coque STL.')


Polaire V_B(CAP) - Vent reel TW + derive + gite



Cellule 8 prete: polaire V_B(CAP) avec contour voile local, quille/safran, trainee appendices, gite et option moment coque STL.


In [9]:
# === CELLULE 9 : HYDROSTATIQUE STL - VOLUME IMMERGE ET STABILITE ===
# Prototype autonome pour analyser une coque STL coupee par un plan d'eau.
# Les coques STL sont lues depuis test_coque_stl, avec test.coque.stl comme repli.
# Reglages importants: unite STL, axe vertical, tirant d'eau, masse et centre de gravite.

from pathlib import Path
import struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------------------------------------------------------------------------
# 1) Parametres utilisateur
# ---------------------------------------------------------------------------

STL_FOLDER = Path("test_coque_stl")
DEFAULT_STL_PATH = Path("test.coque.stl")
STL_PATH = DEFAULT_STL_PATH
STL_UNIT = "mm"                 # "m", "cm" ou "mm". test.coque.stl semble etre en mm.
LONGITUDINAL_AXIS = "X"          # axe de roulis/gite
VERTICAL_AXIS = "Z"              # axe vertical dans le STL
VERTICAL_POSITIVE_UP = True       # True si Z+ monte, False si Z+ descend

# Choix du plan d'eau initial:
# la cellule 9 travaille avec un tirant d'eau fixe, modifiable par widget.
WATERLINE_LEVEL = None
DRAFT_M = None

RHO_WATER = 1025.0                # kg/m3, eau de mer typique
RHO_CARBON = 1500.0               # kg/m3, carbone/stratifie equivalent
RHO_STEEL = 7850.0                # kg/m3, acier
G_ACCEL = 9.81                    # m/s2

# Valeurs par defaut du panneau de reglage.
DRAFT_M_DEFAULT = 0.25
HULL_THICKNESS_MM_DEFAULT = 25.0
SAIL_SKIN_THICKNESS_MM_DEFAULT = 3.0
STEEL_KEEL_MASS_KG_DEFAULT = 5.0

# Dimensions ARGO-VPP utilisees pour les appendices si les widgets des cellules
# precedentes ne sont pas encore disponibles.
ARGO_DERIVE_AREA_M2_DEFAULT = 0.20
ARGO_DERIVE_SPAN_M_DEFAULT = 1.00
ARGO_DERIVE_THICKNESS_RATIO_DEFAULT = 0.12
ARGO_RUDDER_AREA_M2_DEFAULT = 0.052
ARGO_RUDDER_SPAN_M_DEFAULT = 0.65
ARGO_RUDDER_THICKNESS_RATIO_DEFAULT = 0.12
FOIL_VOLUME_SHAPE_FACTOR = 0.68

# Ces valeurs sont calculees automatiquement par la cellule 9.
BOAT_MASS_KG = None
CENTER_OF_GRAVITY = None

HEEL_ANGLES_DEG = np.arange(-60.0, 65.0, 5.0)
RUN_HEEL_SWEEP = True

EPS_REL = 1e-9
MERGE_TOL_REL = 1e-7
MAX_PREVIEW_TRIANGLES = 5000


# ---------------------------------------------------------------------------
# 2) Outils STL, geometrie et maillage
# ---------------------------------------------------------------------------

AXIS_INDEX = {"X": 0, "Y": 1, "Z": 2}
UNIT_SCALE = {"m": 1.0, "cm": 0.01, "mm": 0.001}


def axis_index(axis_name):
    axis = str(axis_name).upper()
    if axis not in AXIS_INDEX:
        raise ValueError(f"Axe inconnu: {axis_name!r}. Utiliser X, Y ou Z.")
    return AXIS_INDEX[axis]


def frame_vectors(longitudinal_axis, vertical_axis, vertical_positive_up=True):
    long_idx = axis_index(longitudinal_axis)
    vertical_idx = axis_index(vertical_axis)
    if long_idx == vertical_idx:
        raise ValueError("L'axe longitudinal et l'axe vertical doivent etre differents.")
    trans_idx = ({0, 1, 2} - {long_idx, vertical_idx}).pop()

    long_vec = np.zeros(3)
    trans_vec = np.zeros(3)
    up_vec = np.zeros(3)
    long_vec[long_idx] = 1.0
    trans_vec[trans_idx] = 1.0
    up_vec[vertical_idx] = 1.0 if vertical_positive_up else -1.0
    return long_idx, trans_idx, vertical_idx, long_vec, trans_vec, up_vec


def load_stl_triangles(path, unit="m"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"STL introuvable: {path}")
    scale = UNIT_SCALE.get(str(unit).lower())
    if scale is None:
        raise ValueError("STL_UNIT doit valoir 'm', 'cm' ou 'mm'.")

    raw = path.read_bytes()
    triangles = []

    # ASCII STL, seulement si l'en-tete ressemble vraiment a un STL texte.
    if raw[:5].lower() == b"solid" and b"facet" in raw[:4096].lower():
        text = raw.decode("utf-8", errors="ignore")
        verts = []
        for line in text.splitlines():
            parts = line.strip().split()
            if len(parts) == 4 and parts[0].lower() == "vertex":
                verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
        if len(verts) >= 3 and len(verts) % 3 == 0:
            return np.asarray(verts, dtype=float).reshape((-1, 3, 3)) * scale

    # Binary STL.
    if len(raw) < 84:
        raise ValueError("Fichier STL invalide ou trop court.")
    n_tri = struct.unpack("<I", raw[80:84])[0]
    offset = 84
    for _ in range(n_tri):
        if offset + 50 > len(raw):
            break
        chunk = raw[offset:offset + 50]
        coords = struct.unpack("<12fH", chunk)
        triangles.append([coords[3:6], coords[6:9], coords[9:12]])
        offset += 50
    if not triangles:
        raise ValueError("Aucun triangle STL lisible.")
    return np.asarray(triangles, dtype=float) * scale


def signed_distance_to_water(points, waterline_level, vertical_idx, up_sign):
    pts = np.asarray(points, dtype=float)
    return (pts[..., vertical_idx] - float(waterline_level)) * float(up_sign)


def triangle_area(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0
    cross = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    return float(0.5 * np.linalg.norm(cross, axis=1).sum())


def clean_polygon_vertices(poly, tol):
    if len(poly) == 0:
        return []
    cleaned = []
    for p in poly:
        if not cleaned or np.linalg.norm(np.asarray(p) - np.asarray(cleaned[-1])) > tol:
            cleaned.append(np.asarray(p, dtype=float))
    if len(cleaned) > 1 and np.linalg.norm(cleaned[0] - cleaned[-1]) <= tol:
        cleaned.pop()
    return cleaned


def line_plane_intersection(p0, p1, d0, d1):
    denom = d0 - d1
    if abs(denom) < 1e-18:
        return 0.5 * (np.asarray(p0, dtype=float) + np.asarray(p1, dtype=float))
    t = d0 / denom
    return np.asarray(p0, dtype=float) + t * (np.asarray(p1, dtype=float) - np.asarray(p0, dtype=float))


def clip_triangle_under_water(tri, waterline_level, vertical_idx, up_sign, eps):
    poly = [np.asarray(p, dtype=float) for p in tri]
    output = []
    for i in range(len(poly)):
        p0 = poly[i]
        p1 = poly[(i + 1) % len(poly)]
        d0 = float(signed_distance_to_water(p0, waterline_level, vertical_idx, up_sign))
        d1 = float(signed_distance_to_water(p1, waterline_level, vertical_idx, up_sign))
        in0 = d0 <= eps
        in1 = d1 <= eps
        if in0 and in1:
            output.append(p1)
        elif in0 and not in1:
            output.append(line_plane_intersection(p0, p1, d0, d1))
        elif not in0 and in1:
            output.append(line_plane_intersection(p0, p1, d0, d1))
            output.append(p1)
    return clean_polygon_vertices(output, eps * 10.0)


def triangulate_fan(poly):
    poly = [np.asarray(p, dtype=float) for p in poly]
    if len(poly) < 3:
        return []
    return [np.asarray([poly[0], poly[i], poly[i + 1]], dtype=float) for i in range(1, len(poly) - 1)]


def dedupe_points(points, tol):
    unique = []
    for p in points:
        p = np.asarray(p, dtype=float)
        if not any(np.linalg.norm(p - q) <= tol for q in unique):
            unique.append(p)
    return unique


def triangle_waterline_segments(tri, waterline_level, vertical_idx, up_sign, eps):
    tri = np.asarray(tri, dtype=float)
    d = signed_distance_to_water(tri, waterline_level, vertical_idx, up_sign)
    segments = []
    points = []

    for i in range(3):
        p0 = tri[i]
        p1 = tri[(i + 1) % 3]
        d0 = float(d[i])
        d1 = float(d[(i + 1) % 3])
        on0 = abs(d0) <= eps
        on1 = abs(d1) <= eps
        if on0 and on1:
            # Cas limite: une arete est exactement dans le plan.
            segments.append((p0.copy(), p1.copy()))
        elif d0 * d1 < -eps * eps:
            points.append(line_plane_intersection(p0, p1, d0, d1))
        elif on0:
            points.append(p0.copy())
        elif on1:
            points.append(p1.copy())

    points = dedupe_points(points, eps * 10.0)
    if len(points) == 2:
        segments.append((points[0], points[1]))
    elif len(points) > 2:
        # Degenerescence rare: garder les deux points les plus eloignes.
        max_pair = None
        max_dist = -1.0
        for i in range(len(points)):
            for j in range(i + 1, len(points)):
                dist = np.linalg.norm(points[i] - points[j])
                if dist > max_dist:
                    max_dist = dist
                    max_pair = (points[i], points[j])
        if max_pair is not None:
            segments.append(max_pair)
    return [(a, b) for a, b in segments if np.linalg.norm(a - b) > eps]


def point_key(point, tol):
    return tuple(np.round(np.asarray(point, dtype=float) / tol).astype(np.int64))


def merge_segments_to_loops(segments, tol):
    points = []
    key_to_id = {}
    edges = set()

    def add_point(p):
        key = point_key(p, tol)
        if key not in key_to_id:
            key_to_id[key] = len(points)
            points.append(np.asarray(p, dtype=float))
        return key_to_id[key]

    for a, b in segments:
        ia = add_point(a)
        ib = add_point(b)
        if ia != ib:
            edges.add(tuple(sorted((ia, ib))))

    adjacency = {i: set() for i in range(len(points))}
    for ia, ib in edges:
        adjacency[ia].add(ib)
        adjacency[ib].add(ia)

    loops = []
    unused = set(edges)
    while unused:
        ia, ib = unused.pop()
        loop = [ia, ib]
        previous = ia
        current = ib

        while True:
            candidates = []
            for nxt in adjacency[current]:
                edge = tuple(sorted((current, nxt)))
                if edge in unused:
                    candidates.append(nxt)
            if not candidates:
                break
            # Preferer l'arete qui ne revient pas immediatement en arriere.
            nxt = candidates[0]
            if len(candidates) > 1:
                for cand in candidates:
                    if cand != previous:
                        nxt = cand
                        break
            unused.remove(tuple(sorted((current, nxt))))
            if nxt == loop[0]:
                break
            loop.append(nxt)
            previous, current = current, nxt

        if len(loop) >= 3 and loop[0] in adjacency[loop[-1]]:
            loops.append(np.asarray([points[i] for i in loop], dtype=float))

    return loops, points, adjacency


def project_to_waterplane(points, vertical_idx):
    axes = [0, 1, 2]
    axes.remove(vertical_idx)
    pts = np.asarray(points, dtype=float)
    return pts[:, axes], axes


def polygon_area_2d(points2d):
    pts = np.asarray(points2d, dtype=float)
    if len(pts) < 3:
        return 0.0
    x = pts[:, 0]
    y = pts[:, 1]
    return float(0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y))


def point_in_ccw_triangle(p, a, b, c, eps):
    def cross2(u, v):
        return u[0] * v[1] - u[1] * v[0]
    p = np.asarray(p)
    a = np.asarray(a)
    b = np.asarray(b)
    c = np.asarray(c)
    return (
        cross2(b - a, p - a) >= -eps and
        cross2(c - b, p - b) >= -eps and
        cross2(a - c, p - c) >= -eps
    )


def ear_clip_indices(points2d, eps=1e-12):
    pts = np.asarray(points2d, dtype=float)
    n = len(pts)
    if n < 3:
        return []
    if n == 3:
        return [[0, 1, 2]]

    vertices = list(range(n))
    if polygon_area_2d(pts) < 0.0:
        vertices.reverse()

    triangles = []
    guard = 0
    while len(vertices) > 3 and guard < n * n:
        guard += 1
        ear_found = False
        for pos in range(len(vertices)):
            i_prev = vertices[pos - 1]
            i_curr = vertices[pos]
            i_next = vertices[(pos + 1) % len(vertices)]
            a = pts[i_prev]
            b = pts[i_curr]
            c = pts[i_next]
            cross = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
            if cross <= eps:
                continue
            contains_other = False
            for j in vertices:
                if j in (i_prev, i_curr, i_next):
                    continue
                if point_in_ccw_triangle(pts[j], a, b, c, eps):
                    contains_other = True
                    break
            if contains_other:
                continue
            triangles.append([i_prev, i_curr, i_next])
            del vertices[pos]
            ear_found = True
            break
        if not ear_found:
            # Secours pour polygone presque convexe ou numeriquement delicat.
            return [[vertices[0], vertices[i], vertices[i + 1]] for i in range(1, len(vertices) - 1)]

    if len(vertices) == 3:
        triangles.append(vertices[:])
    return triangles


def triangulate_cap_loops(loops, vertical_idx, tol):
    cap_triangles = []
    waterplane_area = 0.0
    for loop in loops:
        loop = clean_polygon_vertices(loop, tol)
        if len(loop) < 3:
            continue
        points2d, _ = project_to_waterplane(loop, vertical_idx)
        waterplane_area += abs(polygon_area_2d(points2d))
        for ids in ear_clip_indices(points2d, eps=tol * tol):
            cap_triangles.append(np.asarray([loop[i] for i in ids], dtype=float))
    if cap_triangles:
        return np.asarray(cap_triangles, dtype=float), float(waterplane_area)
    return np.zeros((0, 3, 3), dtype=float), float(waterplane_area)


def orient_triangles_outward(triangles):
    tris = np.asarray(triangles, dtype=float).copy()
    if tris.size == 0:
        return tris
    reference = tris.reshape((-1, 3)).mean(axis=0)
    centers = tris.mean(axis=1)
    normals = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    dots = np.einsum("ij,ij->i", normals, centers - reference)
    flip = dots < 0.0
    tmp = tris[flip, 1].copy()
    tris[flip, 1] = tris[flip, 2]
    tris[flip, 2] = tmp
    if signed_mesh_volume(tris) < 0.0:
        tmp = tris[:, 1].copy()
        tris[:, 1] = tris[:, 2]
        tris[:, 2] = tmp
    return tris


def signed_mesh_volume(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0
    vols = np.einsum("ij,ij->i", tris[:, 0], np.cross(tris[:, 1], tris[:, 2])) / 6.0
    return float(vols.sum())


def mesh_volume_and_centroid(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    vols = np.einsum("ij,ij->i", tris[:, 0], np.cross(tris[:, 1], tris[:, 2])) / 6.0
    volume = float(vols.sum())
    if abs(volume) < 1e-18:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    centroids = tris.sum(axis=1) / 4.0
    centroid = (vols[:, None] * centroids).sum(axis=0) / volume
    if volume < 0.0:
        volume = -volume
    return float(volume), centroid


def analyze_immersion(triangles, waterline_level, vertical_idx, up_sign, eps, merge_tol):
    shell_triangles = []
    segments = []
    for tri in triangles:
        clipped = clip_triangle_under_water(tri, waterline_level, vertical_idx, up_sign, eps)
        shell_triangles.extend(triangulate_fan(clipped))
        segments.extend(triangle_waterline_segments(tri, waterline_level, vertical_idx, up_sign, eps))

    shell_triangles = np.asarray(shell_triangles, dtype=float) if shell_triangles else np.zeros((0, 3, 3), dtype=float)
    loops, merged_points, adjacency = merge_segments_to_loops(segments, merge_tol)
    cap_triangles, waterplane_area = triangulate_cap_loops(loops, vertical_idx, merge_tol)

    if len(cap_triangles):
        closed_triangles = np.vstack([shell_triangles, cap_triangles])
    else:
        closed_triangles = shell_triangles.copy()

    closed_triangles = orient_triangles_outward(closed_triangles)
    volume, center_buoyancy = mesh_volume_and_centroid(closed_triangles)

    return {
        "waterline_level": float(waterline_level),
        "wet_area_m2": triangle_area(shell_triangles),
        "waterplane_area_m2": float(waterplane_area),
        "total_closed_area_m2": triangle_area(shell_triangles) + float(waterplane_area),
        "volume_m3": float(volume),
        "displacement_kg": float(volume) * RHO_WATER,
        "center_buoyancy_m": center_buoyancy,
        "shell_triangles": shell_triangles,
        "cap_triangles": cap_triangles,
        "closed_triangles": closed_triangles,
        "loops": loops,
        "n_waterline_segments": len(segments),
        "n_waterline_loops": len(loops),
        "n_merged_waterline_points": len(merged_points),
        "waterline_graph_degrees": [len(v) for v in adjacency.values()],
    }


def rotate_points_about_axis(points, origin, axis_vector, angle_deg):
    pts = np.asarray(points, dtype=float)
    origin = np.asarray(origin, dtype=float)
    axis = np.asarray(axis_vector, dtype=float)
    axis = axis / max(np.linalg.norm(axis), 1e-18)
    angle = np.radians(float(angle_deg))
    rel = pts - origin
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    cross = np.cross(axis, rel)
    dot = np.einsum("...i,i->...", rel, axis)[..., None]
    return origin + rel * cos_a + cross * sin_a + axis * dot * (1.0 - cos_a)


def rotate_triangles_about_axis(triangles, origin, axis_vector, angle_deg):
    shape = triangles.shape
    rotated = rotate_points_about_axis(triangles.reshape((-1, 3)), origin, axis_vector, angle_deg)
    return rotated.reshape(shape)


def find_waterline_for_volume(triangles, target_volume, vertical_idx, up_sign, eps, merge_tol, max_iter=60):
    verts = triangles.reshape((-1, 3))
    h = verts[:, vertical_idx] * up_sign
    span = max(float(h.max() - h.min()), eps)
    lo_h = float(h.min() - 0.05 * span)
    hi_h = float(h.max() + 0.05 * span)

    res_hi = analyze_immersion(triangles, hi_h / up_sign, vertical_idx, up_sign, eps, merge_tol)
    if target_volume <= 0.0:
        return lo_h / up_sign, analyze_immersion(triangles, lo_h / up_sign, vertical_idx, up_sign, eps, merge_tol)
    if target_volume >= res_hi["volume_m3"]:
        return hi_h / up_sign, res_hi

    last_result = None
    for _ in range(int(max_iter)):
        mid_h = 0.5 * (lo_h + hi_h)
        level = mid_h / up_sign
        result = analyze_immersion(triangles, level, vertical_idx, up_sign, eps, merge_tol)
        last_result = result
        if result["volume_m3"] < target_volume:
            lo_h = mid_h
        else:
            hi_h = mid_h
    final_level = 0.5 * (lo_h + hi_h) / up_sign
    return final_level, analyze_immersion(triangles, final_level, vertical_idx, up_sign, eps, merge_tol)


def set_axes_equal(ax, points):
    pts = np.asarray(points, dtype=float)
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    centers = 0.5 * (mins + maxs)
    radius = 0.5 * max(maxs - mins)
    if radius <= 0.0:
        radius = 1.0
    ax.set_xlim(centers[0] - radius, centers[0] + radius)
    ax.set_ylim(centers[1] - radius, centers[1] + radius)
    ax.set_zlim(centers[2] - radius, centers[2] + radius)


def plot_immersion_preview(all_triangles, result, title="Volume immerge", G_point=None):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    shell = result["shell_triangles"]
    cap = result["cap_triangles"]
    if len(shell) > MAX_PREVIEW_TRIANGLES:
        step = int(np.ceil(len(shell) / MAX_PREVIEW_TRIANGLES))
        shell_plot = shell[::step]
    else:
        shell_plot = shell

    if len(shell_plot):
        shell_collection = Poly3DCollection(shell_plot, alpha=0.55, facecolor="#38bdf8", edgecolor="#0f172a", linewidth=0.25)
        ax.add_collection3d(shell_collection)
    if len(cap):
        cap_collection = Poly3DCollection(cap, alpha=0.65, facecolor="#22c55e", edgecolor="#14532d", linewidth=0.35)
        ax.add_collection3d(cap_collection)

    for loop in result["loops"]:
        closed = np.vstack([loop, loop[0]])
        ax.plot(closed[:, 0], closed[:, 1], closed[:, 2], color="black", linewidth=1.8)

    B = result["center_buoyancy_m"]
    if np.all(np.isfinite(B)):
        ax.scatter([B[0]], [B[1]], [B[2]], color="red", s=60, label="B centre de carene")
    if G_point is not None:
        Gp = np.asarray(G_point, dtype=float)
        if np.all(np.isfinite(Gp)):
            ax.scatter([Gp[0]], [Gp[1]], [Gp[2]], color="green", s=65, marker="^", label="G bateau complet")
    if np.all(np.isfinite(B)) or G_point is not None:
        ax.legend(loc="upper left")

    set_axes_equal(ax, all_triangles.reshape((-1, 3)))
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_zlabel("Z [m]")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------------------
# 3) Widgets, centre de gravite du bateau complet et calcul hydrostatique
# ---------------------------------------------------------------------------

def available_cell9_stl_options():
    candidates = []
    if STL_FOLDER.exists():
        candidates.extend(sorted(
            p for p in STL_FOLDER.iterdir()
            if p.is_file() and p.suffix.lower() == ".stl"
        ))
    if DEFAULT_STL_PATH.exists():
        candidates.append(DEFAULT_STL_PATH)

    unique = []
    seen = set()
    for path in candidates:
        resolved_key = str(path.resolve()) if path.exists() else str(path)
        if resolved_key not in seen:
            seen.add(resolved_key)
            unique.append(path)

    if not unique:
        return [("Aucun STL trouve", "")]
    return [(str(path), str(path)) for path in unique]


def selected_cell9_stl_path():
    value = getattr(cell9_stl_dropdown, "value", "")
    if not value:
        raise FileNotFoundError("Aucun fichier STL selectionne. Ajouter des .stl dans test_coque_stl puis cliquer Rafraichir STL.")
    path = Path(value)
    if not path.exists():
        raise FileNotFoundError(f"STL selectionne introuvable: {path}")
    return path


cell9_stl_dropdown = widgets.Dropdown(
    options=available_cell9_stl_options(),
    description="Coque STL",
    layout=widgets.Layout(width="430px"),
    style={"description_width": "80px"},
)
cell9_refresh_stl_button = widgets.Button(
    description="Rafraichir STL",
    tooltip="Relire le dossier test_coque_stl",
    layout=widgets.Layout(width="135px"),
)


def refresh_cell9_stl_options(_=None):
    current = getattr(cell9_stl_dropdown, "value", "")
    options = available_cell9_stl_options()
    values = [value for _, value in options]
    cell9_stl_dropdown.options = options
    if current in values:
        cell9_stl_dropdown.value = current
    elif values:
        cell9_stl_dropdown.value = values[0]


cell9_refresh_stl_button.on_click(refresh_cell9_stl_options)

cell9_draft_m_widget = widgets.FloatText(
    value=DRAFT_M_DEFAULT,
    description="Tirant eau [m]",
    step=0.01,
    layout=widgets.Layout(width="210px"),
)
cell9_hull_thickness_mm_widget = widgets.FloatSlider(
    value=HULL_THICKNESS_MM_DEFAULT,
    min=1.0,
    max=50.0,
    step=0.5,
    description="Ep coque [mm]",
    continuous_update=False,
    layout=widgets.Layout(width="330px"),
)
cell9_steel_keel_mass_kg_widget = widgets.FloatText(
    value=STEEL_KEEL_MASS_KG_DEFAULT,
    description="M quille [kg]",
    step=0.5,
    layout=widgets.Layout(width="210px"),
)
cell9_sail_skin_thickness_mm_widget = widgets.FloatSlider(
    value=SAIL_SKIN_THICKNESS_MM_DEFAULT,
    min=0.2,
    max=15.0,
    step=0.1,
    description="Ep voile [mm]",
    continuous_update=False,
    layout=widgets.Layout(width="330px"),
)
cell9_run_button = widgets.Button(description="Recalculer cellule 9", button_style="primary")
cell9_output = widgets.Output()


def widget_value(name, default):
    obj = globals().get(name, None)
    try:
        if obj is not None and hasattr(obj, "value"):
            return float(obj.value)
        if obj is not None:
            return float(obj)
    except Exception:
        pass
    return float(default)


def surface_area_centroid(triangles):
    tris = np.asarray(triangles, dtype=float)
    if tris.size == 0:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    cross = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    areas = 0.5 * np.linalg.norm(cross, axis=1)
    total_area = float(areas.sum())
    if total_area <= 1e-18:
        return 0.0, np.array([np.nan, np.nan, np.nan])
    centers = tris.mean(axis=1)
    centroid = (areas[:, None] * centers).sum(axis=0) / total_area
    return total_area, centroid


def parse_contour_text_to_points(text):
    points = []
    for line in str(text).splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.replace(";", ",").replace("\t", ",").replace(" ", ",").split(",")
        parts = [p for p in parts if p]
        if len(parts) < 2:
            continue
        try:
            points.append((float(parts[0]), float(parts[1])))
        except Exception:
            continue
    if len(points) < 3:
        raise ValueError("Contour de voile insuffisant.")
    pts = np.asarray(points, dtype=float)
    if not np.allclose(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])
    return pts


def polygon_area_centroid_2d(points):
    pts = np.asarray(points, dtype=float)
    if len(pts) < 3:
        return 0.0, np.array([np.nan, np.nan])
    if not np.allclose(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])
    x = pts[:, 0]
    y = pts[:, 1]
    cross = x[:-1] * y[1:] - x[1:] * y[:-1]
    area_signed = 0.5 * float(cross.sum())
    if abs(area_signed) <= 1e-18:
        return 0.0, np.array([np.nan, np.nan])
    cx = float(((x[:-1] + x[1:]) * cross).sum() / (6.0 * area_signed))
    cy = float(((y[:-1] + y[1:]) * cross).sum() / (6.0 * area_signed))
    return abs(area_signed), np.array([cx, cy])


def get_sail_contour_for_cell9():
    if "wing_contour" in globals():
        pts = np.asarray(globals()["wing_contour"], dtype=float)
        if len(pts) >= 3:
            if not np.allclose(pts[0], pts[-1]):
                pts = np.vstack([pts, pts[0]])
            return pts, "wing_contour cellule 2"
    if "contour_textarea" in globals() and hasattr(globals()["contour_textarea"], "value"):
        try:
            return parse_contour_text_to_points(globals()["contour_textarea"].value), "textarea cellule 2"
        except Exception:
            pass
    default_text = "0.0,0.0\n0.52,0.0\n0.62,0.70\n0.18,1.20\n0.0,0.0"
    return parse_contour_text_to_points(default_text), "contour voile defaut ARGO"


def foil_volume_from_area_span(area_m2, span_m, thickness_ratio):
    area_m2 = max(float(area_m2), 0.0)
    span_m = max(float(span_m), 1e-9)
    chord_eff = area_m2 / span_m
    return float(FOIL_VOLUME_SHAPE_FACTOR * float(thickness_ratio) * area_m2 * chord_eff)


def make_height_only_point(reference_point, vertical_idx, up_sign, height_value):
    point = np.asarray(reference_point, dtype=float).copy()
    point[vertical_idx] = float(height_value) / float(up_sign)
    return point


def estimate_complete_boat_cg(triangles, vertices, long_idx, trans_idx, vertical_idx, up_sign):
    hull_thickness_m = max(float(cell9_hull_thickness_mm_widget.value), 0.0) * 1e-3
    steel_keel_mass_kg = max(float(cell9_steel_keel_mass_kg_widget.value), 0.0)
    sail_skin_thickness_m = max(float(cell9_sail_skin_thickness_mm_widget.value), 0.0) * 1e-3

    reference = vertices.mean(axis=0)
    h_values = vertices[:, vertical_idx] * up_sign
    h_bottom = float(h_values.min())
    h_top = float(h_values.max())

    rows = []

    def add_component(name, mass_kg, point, method, volume_m3=np.nan, area_m2=np.nan):
        mass_kg = max(float(mass_kg), 0.0)
        point = np.asarray(point, dtype=float)
        rows.append({
            "element": name,
            "method": method,
            "mass_kg": mass_kg,
            "volume_m3": float(volume_m3) if np.isfinite(volume_m3) else np.nan,
            "area_m2": float(area_m2) if np.isfinite(area_m2) else np.nan,
            "G_x_m": point[0],
            "G_y_m": point[1],
            "G_z_m": point[2],
            "height_m": point[vertical_idx] * up_sign,
        })

    hull_area, hull_surface_centroid = surface_area_centroid(triangles)
    hull_height = float(hull_surface_centroid[vertical_idx] * up_sign) if np.all(np.isfinite(hull_surface_centroid)) else 0.5 * (h_bottom + h_top)
    hull_point = make_height_only_point(reference, vertical_idx, up_sign, hull_height)
    hull_mass = RHO_CARBON * hull_area * hull_thickness_m
    add_component(
        "coque carbone",
        hull_mass,
        hull_point,
        "surface STL * epaisseur coque * rho carbone",
        volume_m3=hull_area * hull_thickness_m,
        area_m2=hull_area,
    )

    derive_area = widget_value("S_derive_widget7", ARGO_DERIVE_AREA_M2_DEFAULT)
    derive_span = widget_value("hauteur_quille_widget7", ARGO_DERIVE_SPAN_M_DEFAULT)
    derive_volume = foil_volume_from_area_span(derive_area, derive_span, ARGO_DERIVE_THICKNESS_RATIO_DEFAULT)
    derive_height = h_bottom - 0.5 * derive_span
    derive_point = make_height_only_point(reference, vertical_idx, up_sign, derive_height)
    add_component(
        "derive carbone",
        RHO_CARBON * derive_volume,
        derive_point,
        "volume foil ARGO approx * rho carbone",
        volume_m3=derive_volume,
        area_m2=derive_area,
    )

    steel_height = h_bottom - derive_span
    steel_point = make_height_only_point(reference, vertical_idx, up_sign, steel_height)
    add_component(
        "quille acier",
        steel_keel_mass_kg,
        steel_point,
        "masse imposee au bout de derive",
        volume_m3=steel_keel_mass_kg / RHO_STEEL if RHO_STEEL > 0.0 else np.nan,
        area_m2=np.nan,
    )

    rudder_area = widget_value("S_safran_widget7", ARGO_RUDDER_AREA_M2_DEFAULT)
    rudder_span = widget_value("hauteur_safran_widget7", ARGO_RUDDER_SPAN_M_DEFAULT)
    rudder_volume = foil_volume_from_area_span(rudder_area, rudder_span, ARGO_RUDDER_THICKNESS_RATIO_DEFAULT)
    rudder_height = h_bottom - 0.5 * rudder_span
    rudder_point = make_height_only_point(reference, vertical_idx, up_sign, rudder_height)
    add_component(
        "safran carbone",
        RHO_CARBON * rudder_volume,
        rudder_point,
        "volume foil ARGO approx * rho carbone",
        volume_m3=rudder_volume,
        area_m2=rudder_area,
    )

    sail_pts, sail_source = get_sail_contour_for_cell9()
    sail_area, sail_centroid = polygon_area_centroid_2d(sail_pts)
    sail_y_min = float(np.min(sail_pts[:, 1]))
    sail_height = h_top + max(float(sail_centroid[1] - sail_y_min), 0.0)
    sail_point = make_height_only_point(reference, vertical_idx, up_sign, sail_height)
    sail_mass = RHO_CARBON * sail_area * sail_skin_thickness_m
    add_component(
        "voile carbone",
        sail_mass,
        sail_point,
        f"surface voile {sail_source} * epaisseur voile * rho carbone",
        volume_m3=sail_area * sail_skin_thickness_m,
        area_m2=sail_area,
    )

    df = pd.DataFrame(rows)
    total_mass = float(df["mass_kg"].sum())
    if total_mass <= 1e-12:
        G_total = reference.copy()
    else:
        G_total = np.array([
            float((df["mass_kg"] * df["G_x_m"]).sum() / total_mass),
            float((df["mass_kg"] * df["G_y_m"]).sum() / total_mass),
            float((df["mass_kg"] * df["G_z_m"]).sum() / total_mass),
        ])
    return df, total_mass, G_total


def run_cell9_analysis(_=None):
    global triangles, vertices, summary, upright, stability_df, component_cg_df
    global BOAT_MASS_KG, CENTER_OF_GRAVITY, STL_PATH

    with cell9_output:
        clear_output(wait=True)
        selected_stl_path = selected_cell9_stl_path()
        STL_PATH = selected_stl_path
        triangles = load_stl_triangles(selected_stl_path, STL_UNIT)
        vertices = triangles.reshape((-1, 3))
        long_idx, trans_idx, vertical_idx, long_vec, trans_vec, up_vec = frame_vectors(
            LONGITUDINAL_AXIS,
            VERTICAL_AXIS,
            VERTICAL_POSITIVE_UP,
        )
        up_sign = 1.0 if VERTICAL_POSITIVE_UP else -1.0
        extent = vertices.max(axis=0) - vertices.min(axis=0)
        eps = max(float(extent.max()) * EPS_REL, 1e-12)
        merge_tol = max(float(extent.max()) * MERGE_TOL_REL, 1e-9)

        component_cg_df, BOAT_MASS_KG, CENTER_OF_GRAVITY = estimate_complete_boat_cg(
            triangles,
            vertices,
            long_idx,
            trans_idx,
            vertical_idx,
            up_sign,
        )

        h_values = vertices[:, vertical_idx] * up_sign
        h_min = float(h_values.min())
        h_max = float(h_values.max())
        draft_m = max(float(cell9_draft_m_widget.value), 0.0)
        waterline_level = (h_min + draft_m) / up_sign

        upright = analyze_immersion(triangles, waterline_level, vertical_idx, up_sign, eps, merge_tol)
        B = upright["center_buoyancy_m"]
        target_volume = upright["volume_m3"]
        displacement_kg = float(target_volume * RHO_WATER)

        summary = pd.DataFrame([{
            "STL": selected_stl_path.name,
            "STL_path": str(selected_stl_path),
            "unit": STL_UNIT,
            "triangles_STL": len(triangles),
            "Lx_m": extent[0],
            "Ly_m": extent[1],
            "Lz_m": extent[2],
            "draft_m": draft_m,
            "waterline_level_m": upright["waterline_level"],
            "wet_area_m2": upright["wet_area_m2"],
            "waterplane_area_m2": upright["waterplane_area_m2"],
            "total_closed_area_m2": upright["total_closed_area_m2"],
            "immersed_volume_m3": target_volume,
            "displacement_kg": displacement_kg,
            "boat_mass_from_CG_model_kg": BOAT_MASS_KG,
            "mass_minus_displacement_kg": BOAT_MASS_KG - displacement_kg,
            "B_x_m": B[0],
            "B_y_m": B[1],
            "B_z_m": B[2],
            "G_x_m": CENTER_OF_GRAVITY[0],
            "G_y_m": CENTER_OF_GRAVITY[1],
            "G_z_m": CENTER_OF_GRAVITY[2],
            "waterline_loops": upright["n_waterline_loops"],
            "waterline_segments": upright["n_waterline_segments"],
        }])

        display(HTML("<h3>Cellule 9 - Hydrostatique coque seule + G bateau complet</h3>"))
        display(summary)
        display(HTML("<h4>Centre de gravite estime par composants</h4>"))
        display(component_cg_df)

        if abs(BOAT_MASS_KG - displacement_kg) > max(0.05 * max(displacement_kg, 1e-9), 0.25):
            print("Avertissement: la masse estimee du bateau complet ne correspond pas au deplacement impose par le tirant d'eau.")
            print("Pour un equilibre physique, ajuster le tirant d'eau, les epaisseurs ou la masse de quille.")
        if upright["n_waterline_loops"] == 0 and upright["volume_m3"] > 0.0:
            print("Note: aucun contour de flottaison n'a ete reconstruit. Le volume suppose que la partie conservee est deja fermee.")
        if any(deg not in (0, 2) for deg in upright["waterline_graph_degrees"]):
            print("Avertissement: la ligne de flottaison n'est pas un ensemble de boucles parfaitement regulieres.")
            print("Verifier trous, doublons, triangles degeneres ou plan passant par des sommets/aretes.")

        plot_immersion_preview(triangles, upright, title="Coque immergee - plan initial", G_point=CENTER_OF_GRAVITY)

        if RUN_HEEL_SWEEP:
            rows = []
            G_point = np.asarray(CENTER_OF_GRAVITY, dtype=float)
            heel_origin = G_point.copy()
            for phi in HEEL_ANGLES_DEG:
                rotated = rotate_triangles_about_axis(triangles, heel_origin, long_vec, phi)
                level_phi, result_phi = find_waterline_for_volume(rotated, target_volume, vertical_idx, up_sign, eps, merge_tol)
                B_phi = result_phi["center_buoyancy_m"]

                GZ_signed = np.nan
                MR_signed = np.nan
                if np.all(np.isfinite(B_phi)):
                    arm_vector = B_phi - G_point
                    GZ_signed = float(np.dot(long_vec, np.cross(arm_vector, up_vec)))
                    MR_signed = float(RHO_WATER * result_phi["volume_m3"] * G_ACCEL * GZ_signed)

                rows.append({
                    "phi_deg": float(phi),
                    "waterline_level_m": level_phi,
                    "volume_m3": result_phi["volume_m3"],
                    "volume_error_m3": result_phi["volume_m3"] - target_volume,
                    "wet_area_m2": result_phi["wet_area_m2"],
                    "waterplane_area_m2": result_phi["waterplane_area_m2"],
                    "B_x_m": B_phi[0],
                    "B_y_m": B_phi[1],
                    "B_z_m": B_phi[2],
                    "GZ_signed_m": GZ_signed,
                    "GZ_abs_m": abs(GZ_signed) if np.isfinite(GZ_signed) else np.nan,
                    "MR_signed_Nm": MR_signed,
                    "MR_abs_Nm": abs(MR_signed) if np.isfinite(MR_signed) else np.nan,
                    "waterline_loops": result_phi["n_waterline_loops"],
                })

            stability_df = pd.DataFrame(rows)
            display(HTML("<h3>Balayage en gite a volume constant</h3>"))
            display(stability_df)

            fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
            axes[0].plot(stability_df["phi_deg"], stability_df["GZ_signed_m"], "o-", label="GZ signe")
            axes[0].plot(stability_df["phi_deg"], stability_df["GZ_abs_m"], "s--", label="|GZ|")
            axes[0].set_ylabel("GZ [m]")
            axes[0].legend()
            axes[1].plot(stability_df["phi_deg"], stability_df["MR_signed_Nm"], "o-", label="MR signe")
            axes[1].plot(stability_df["phi_deg"], stability_df["MR_abs_Nm"], "s--", label="|MR|")
            axes[1].set_ylabel("Moment [N.m]")
            axes[1].legend()
            for ax in axes:
                ax.set_xlabel("Gite phi [deg]")
                ax.set_xlim(float(np.min(HEEL_ANGLES_DEG)), float(np.max(HEEL_ANGLES_DEG)))
                ax.grid(True, alpha=0.3)
            fig.suptitle("Moment redressant: B(coque STL) et G(bateau complet estime)")
            plt.tight_layout()
            plt.show()

        print("Variables disponibles: summary, component_cg_df, upright, stability_df, CENTER_OF_GRAVITY, BOAT_MASS_KG")


cell9_run_button.on_click(run_cell9_analysis)
cell9_controls = widgets.VBox([
    widgets.HTML("<b>Reglages cellule 9</b>"),
    widgets.HBox([cell9_stl_dropdown, cell9_refresh_stl_button]),
    widgets.HBox([cell9_draft_m_widget, cell9_steel_keel_mass_kg_widget]),
    widgets.HBox([cell9_hull_thickness_mm_widget, cell9_sail_skin_thickness_mm_widget]),
    cell9_run_button,
])
display(cell9_controls, cell9_output)
run_cell9_analysis()

Output()